# 다중분류(9-Class) 파이프라인 통합본 — 2026-09-11

> **이 노트북은 리뷰용이다. 여기서 직접 실행하지 않는다.** 각 셀은 `/workspace`의 원본 `.py` 파일 내용을 그대로 옮긴 것이고,
> 원본 파일이 정본이다(이 노트북을 고쳐도 실제 파이프라인은 안 바뀐다). 데이터 경로·CLI 인자는 원본 스크립트 참고.
>
> **범위**: 손은총 담당 다중분류(9-Class: 8종 세탁 패턴 + 패턴외)의 **현재 채택 라인**만 코드 전문을 담았다.
> 이진분류(1단계, 김태훈 담당)는 별도이며 여기 없음. 탐색 단계에서 쓰였다가 지금은 안 쓰는 스크립트는
> 맨 아래 §부록에 파일명·역할 한 줄만 남기고 코드는 생략했다(왜 뺐는지도 적음).
>
> ## 파이프라인 순서
> 1. 전처리 — `preprocess_9class.py`
> 2. 언더샘플링 — `prep9/undersample.py`
> 3. 메모리 절약판 유틸 — `prep9/lowmem.py`
> 4. 산출물 좌표계 — `prep9/coords.py`
> 5. 학습셋 로더 — `train9/data.py`
> 6. 모델 팩토리 — `train9/models.py`
> 7. 평가지표 — `train9/metrics.py`
> 8. 실험 러너 — `train9/run.py`
> 9. **현재 채택 학습** — `train9/lgb_ooc.py` (LightGBM out-of-core)
> 10. 학습 실행 스크립트 — `ooc_full_train.py`
> 11. 블록(사건) 묶기 — `build_blocks_9class.py`
> 12. 블록 위상 피처 — `train9/blocks.py`
> 13. 알림 블록 후처리 — `prep9/postprocess_blocks.py`
> 14. Cascade 재순위 — `rerank_ego_9class.py`


## 1. 전처리 — `preprocess_9class.py`

HI-Small/HI-Large 원본 CSV → 2단계 계층 파이프라인(1차 9-class 다중분류용 피처·라벨, 2차 패턴외 이진 대상 분리). 81열 피처 생성, 꼬리 절단, split 배정.

원본 경로: `preprocess_9class.py`

In [ ]:
#!/usr/bin/env python3
"""HI-Small 전처리 — 2단계 계층 파이프라인(1차 9-class · 2차 패턴외 이진)

근거: 2026-08-26 팀 회의. 이 스크립트는 회의에서 결정된 것만 구현하고,
회의가 정하지 않은 값은 임의로 고르지 않고 `assumptions` 에 이름을 붙여 남긴다.

  회의 결정 1  1차 = 8종 패턴 다중분류, 2차 = '패턴 외' 대상 이진분류  -> §A §C
  회의 결정 1  1차 다중분류는 9-Class                                  -> §A
  회의 결정 2  패턴 적발 = 패턴 블록 알림 / 패턴 외 적발 = 단건 알림    -> §F
  회의 결정 2  거래별 점수만 나오므로 묶어주는 후처리가 필수            -> §F
  회의 결정 3  Bipartite vs Stack 혼동 원시 데이터 심층 분석            -> §G
  회의 결정 3  1~10일 구간과 18일 전체 구간을 각각 처리                 -> --basis
  회의 결정 3  클러스터 기반 언더샘플링으로 정상 비율 조정              -> §E

피처·정제·분할은 새로 만들지 않는다. 2026-08-25 산출물
`preprocess_multiclass.ipynb`(팀 EDA 수치 5개 재현 검증 통과)의 정의 셀을 그대로 싣고,
회의가 바꾼 **라벨 구성과 하위 산출물**만 다시 짠다(§A 이후).
"""
from __future__ import annotations

import argparse
import gc
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, '/workspace')
from prep9 import nbcells, postprocess_blocks as pb, undersample as us, lowmem as lm   # noqa: E402

SEED = 42
# 2026-09-02: 컨테이너 RAM 상한이 cgroup 27.3 GiB 라(호스트 503 GB 아님) HI-Large 는 원래 함수로 OOM 이다.
# --lowmem 이면 노트북 원본을 건드리지 않고 prep9/lowmem.py 의 메모리 절약판 함수로 갈아끼운다(값 동일).
LOWMEM = False
LOWMEM_SCRATCH = Path('/workspace/processed_9class/_lowmem_scratch')
SPLIT_TAGS = ('tr', 'va', 'te')
CLASS8 = 8                                  # 9번째 클래스 = 패턴 외(정상 + 패턴외 세탁)
# 2026-09-02: 언더샘플 방식. 'cluster' = 원래 동작(MiniBatchKMeans 층화 + 무작위 4벌씩).
# 'random' = 무작위 4벌만(피처 행렬을 RAM 에 안 올림 — HI-Large 용). 'none' = 생략.
# 근거: 08-27 선택 결과가 full(언더샘플 없음)이었고, HI-Large 는 train 정상 1억 행이라
# 클러스터링이 몇 시간을 먹는데 쓰이는 곳이 없다. main() 이 --undersample 로 덮어쓴다.
UNDERSAMPLE_MODE = 'cluster'
BASES = {
    # 회의: "1~10일 구간과 18일 전체 구간 데이터의 패턴 분포 차이를 고려한 모델 실험"
    # 아래 두 키는 HI-Small 의 기간(10일/18일)을 이름에 박은 것이라 다른 세트에 쓰면 거짓말이 된다.
    # 기존 산출물 경로(processed_9class/HI-Small_10day)를 유지하려고 그대로 둔다.
    '10day': dict(tail_min_frac=0.05, note='공식 주 기간만(화두 17). 꼬리 8일 제외', only='HI-Small'),
    '18day': dict(tail_min_frac=0.00, note='꼬리 포함 전체 기간', only='HI-Small'),
    # 세트 무관 이름. 절단 기준은 '일별 거래량이 중앙값의 5% 미만인 꼬리'라는 상대 규칙이라
    # 기간 길이와 무관하게 그대로 성립한다(HI-Medium 16일, HI-Large 97일 + 꼬리).
    'main': dict(tail_min_frac=0.05, note='공식 주 기간만(화두 17). 저조 꼬리 제외'),
    'full': dict(tail_min_frac=0.00, note='꼬리 포함 전체 기간'),
}
# 데이터셋별 기본 basis — HI-Small 은 기존 이름을 유지해 산출물 경로가 바뀌지 않게 한다.
DEFAULT_BASIS = {'HI-Small': ['10day', '18day']}
DEFAULT_BASIS_OTHER = ['main', 'full']
# 팀 EDA 대조 상수는 HI-Small 실측값이다. 다른 세트에서는 대조 자체를 건너뛴다.
EDA_EXPECTED = {'HI-Small': {'label1_total': 5177, 'in_pattern': 3209, 'out_of_pattern': 1968}}
UNDERSAMPLE_RATIOS = (10, 30, 100, 300)     # 정상:패턴 목표 비율
KMEANS_KS = (64, 256, 1024)
BLOCK_WINDOWS = (60, 360, 720, 1440, 2160, 4320, 10080, 25920)   # 분


# ─────────────────────────────────────────────────────────────────────────────
# §A. 9-class 라벨
# ─────────────────────────────────────────────────────────────────────────────
def make_y9(y: np.ndarray) -> np.ndarray:
    """8종 패턴은 0~7 그대로, 그 밖(패턴 외 세탁 + 정상)은 전부 8.

    회의 결정: 1차 다중분류가 **전 거래**를 받아 9개 중 하나로 보내고, 8로 간 거래만
    2차 이진으로 넘긴다. 따라서 9번째 클래스는 '패턴 외 세탁'만이 아니라
    '8종에 속하지 않는 모든 거래'다 — 정상 거래가 여기 포함된다.

    팀 문서(`EDA 결과와 모델 설계 결정`)는 "패턴 외를 하나의 일관된 9번째 클래스로
    가정하지 않는다"고 경고한 바 있다. 회의 결정이 그 경고를 덮어쓴 것이므로
    경고 자체는 meta 의 `open_risks` 에 그대로 남긴다.
    """
    return np.where((y >= 0) & (y <= 7), y, CLASS8).astype(np.int8)


# ─────────────────────────────────────────────────────────────────────────────
# 실행
# ─────────────────────────────────────────────────────────────────────────────
def run_basis(ns: dict, raw_all: dict, vocab_keys: dict, n_nodes_raw: int,
              basis: str, out_root: Path, clean_info: dict, copy_raw: bool = True) -> dict:
    cfg = BASES[basis]
    out = out_root / f'{ns["DATASET"]}_{basis}'
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    print(f'\n{"=" * 78}\n[{basis}] {cfg["note"]}\n{"=" * 78}')

    # ── 1. 꼬리 절단 · 시간순 분할 (재사용: trim_tails / sort_split_reindex) ──
    ns['TAIL_MIN_FRAC'] = cfg['tail_min_frac']
    # basis 가 하나뿐이면 원본을 그대로 소비한다(1.8억 행 사본 = 9 GB). 둘 이상이면 원래대로 복사.
    if copy_raw:
        raw = {k: v.copy() for k, v in raw_all.items()}
    else:
        raw = dict(raw_all)                             # 배열은 공유, 원래 dict 만 비운다(참조 해제용)
        raw_all.clear()
    raw, trim_info = ns['trim_tails'](raw)
    raw, old2new, split_info = ns['sort_split_reindex'](raw, n_nodes_raw)
    split = raw['split']
    n_rows, n_nodes = len(split), split_info['n_accounts']

    # ── 2. 고립 계좌 마스크: 계산해서 저장하되 **적용하지 않는다** ────────────
    # 2026-08-25 회의의 실험 옵션이다. 이번 회의는 불균형 대응으로 클러스터 기반
    # 언더샘플링(§E)을 택했으므로, 라벨에서 파생된 필터를 겹쳐 적용하면 두 효과가
    # 뒤섞인다. 마스크는 남겨 모델 담당자가 켤 수 있게 한다.
    iso_edge, iso_stats = ns['flag_isolated'](raw, n_nodes)
    iso_edge = iso_edge & (raw['is_pos'] != 1)          # 양성은 어떤 필터로도 빼지 않는다
    sample_mask_isolated = ~(iso_edge & (split == 0))   # train 에만 해당하는 마스크
    del iso_edge

    # ── 3. 패턴 정답지 · 파생 배열 · 라벨 (재사용) ───────────────────────────
    pat = ns['load_patterns'](ns['PATTERNS_PATH'], vocab_keys, old2new,
                              split_info['ts_offset_epoch_min'])
    base_keys = ['ts_min', 'src_id', 'dst_id', 'pair_id', 'paid', 'recv', 'cents',
                 'fmt_code', 'pay_code', 'recv_code', 'split']
    if not LOWMEM:
        base_keys.insert(7, 'recv_cents')
    a = {k: raw[k] for k in base_keys}
    a['is_pos'] = (raw['is_pos'] == 1)
    _day = raw['ts_epoch_min'] // 1440
    a['hour'] = ((raw['ts_epoch_min'] % 1440) // 60).astype(np.int8)
    a['dow'] = ((_day + 3) % 7).astype(np.int8)
    a['ccy_mismatch'] = a['pay_code'] != a['recv_code']
    def _rate(idx):                                     # 노트북 셀 19(실행 셀)의 재현
        names = list(idx)
        miss = [nm for nm in names if nm not in ns['USD_RATE']]
        return np.array([ns['USD_RATE'].get(nm, ns['USD_RATE_FALLBACK'])
                         for nm in names], dtype=np.float64), miss
    pay_rate, pay_miss = _rate(vocab_keys['pay'].index)
    recv_rate, recv_miss = _rate(vocab_keys['recv'].index)
    if LOWMEM:
        # 전체 길이 파생 배열(paid_log·recv_log·usd_paid·usd_recv = 각 1.4 GB)을 만들지 않는다.
        # FeatureBuilderLM 이 블록마다 같은 식으로 즉석 계산한다. amt_mismatch 는 recv_cents 를
        # 버렸으므로 rint(recv*100) 을 청크로 다시 만들어 비교한다(값은 _prep_chunk 와 동일).
        a['_pay_rate'], a['_recv_rate'] = pay_rate, recv_rate
        amt_mm = np.empty(n_rows, dtype=bool)
        for st in range(0, n_rows, 20_000_000):
            en = min(st + 20_000_000, n_rows)
            rc = np.rint(a['recv'][st:en] * 100).astype(np.int64)
            amt_mm[st:en] = (~a['ccy_mismatch'][st:en]) & (a['cents'][st:en] != rc)
        a['amt_mismatch'] = amt_mm
        del amt_mm
    else:
        a['paid_log'] = np.log1p(a['paid'])
        a['recv_log'] = np.log1p(a['recv'])
        a['amt_mismatch'] = (~a['ccy_mismatch']) & (a['cents'] != a['recv_cents'])
        a['usd_paid'] = a['paid'] * pay_rate[a['pay_code']]
        a['usd_recv'] = a['recv'] * recv_rate[a['recv_code']]
    day_idx = (_day - _day.min()).astype(np.int16)
    del _day

    y, attempt, label_stats = ns['attach_labels'](a, pat)
    y9 = make_y9(y)
    is_oop = (y == CLASS8)                              # 패턴 외 '세탁' (정상 아님)
    del raw
    if LOWMEM:
        # cents(int64 1.4 GB)는 라벨 매칭(JOIN_KEYS)까지만 필요하다. 피처는 '만 단위 반올림 금액' bool 만 쓴다.
        ra = np.empty(n_rows, dtype=bool)
        for st in range(0, n_rows, 20_000_000):
            en = min(st + 20_000_000, n_rows)
            ra[st:en] = (a['cents'][st:en] % 10000 == 0)
        a['round_amt'] = ra
        del a['cents'], ra
        # 행 단위 큰 배열(9 GB)을 디스크 memmap 으로 내린다. 피처 계산은 1M 행 연속 블록이라 순차 읽기다.
        # RAM 은 무작위 접근하는 prefix 합·정렬 키가 쓴다(FeatureBuilderLM). 끝나면 파일을 지운다.
        spill_dir = LOWMEM_SCRATCH / f'a_{basis}'
        spill_dir.mkdir(parents=True, exist_ok=True)
        spilled = [k for k, v in a.items() if isinstance(v, np.ndarray) and v.ndim == 1
                   and len(v) == n_rows and k not in ('split', 'is_pos')]
        for k in spilled:
            p = spill_dir / f'{k}.npy'
            np.save(p, a[k])
            a[k] = np.load(p, mmap_mode='r')
        print(f'[lowmem] a 배열 {len(spilled)}개를 디스크 memmap 으로 내림 -> {spill_dir}')
    gc.collect()

    hist9 = {int(c): int((y9 == c).sum()) for c in range(9)}
    print(f'[9class] {hist9} · 8번 클래스 중 세탁 {int(is_oop.sum()):,} / '
          f'정상 {int((y9 == CLASS8).sum() - is_oop.sum()):,}')

    # ── 4. 1차 9-class 학습셋 (전 거래 · 엣지 단위) ───────────────────────────
    ns['a'], ns['vocab_keys'] = a, vocab_keys
    fb = ns['FeatureBuilder'](a, vocab_keys)
    feat_names, feat_blocks = fb.names, fb.blocks
    n_feat = len(feat_names)
    b1, b2 = int(np.searchsorted(split, 1)), int(np.searchsorted(split, 2))
    bounds = [(0, b1), (b1, b2), (b2, n_rows)]

    s1 = out / 'stage1_9class'
    s1.mkdir(exist_ok=True)
    BLK = 1_000_000
    for s, (lo, hi) in enumerate(bounds):
        tag = SPLIT_TAGS[s]
        mm = np.lib.format.open_memmap(s1 / f'X_{tag}.npy', mode='w+',
                                       dtype=np.float32, shape=(hi - lo, n_feat))
        for st in range(lo, hi, BLK):
            en = min(st + BLK, hi)
            mm[st - lo:en - lo] = fb.transform(np.arange(st, en))
        mm.flush(); del mm
        np.save(s1 / f'y9_{tag}.npy', y9[lo:hi])
        np.save(s1 / f'is_pos_{tag}.npy', a['is_pos'][lo:hi].astype(np.int8))
        np.save(s1 / f'is_oop_{tag}.npy', is_oop[lo:hi])
        if tag == 'tr':     # 정의상 train 에서만 False 가 될 수 있다. va/te 는 전부 True 라 안 만든다
            np.save(s1 / 'sample_mask_isolated_tr.npy', sample_mask_isolated[lo:hi])
        h = np.bincount(y9[lo:hi], minlength=9)
        print(f'[stage1] {tag}: {hi - lo:,}x{n_feat} · 패턴 {int(h[:8].sum()):,} '
              f'· 클래스8 {int(h[8]):,} (그중 세탁 {int(is_oop[lo:hi].sum()):,})')

    # 절대 시각 피처 열 위치 — 꼬리 지름길 검증·제거 실험용(§보고서)
    abs_time_cols = [i for i, nm in enumerate(feat_names)
                     if nm in ('hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend')]
    if hasattr(fb, 'close'):                            # lowmem: prefix 합 memmap 파일 정리
        fb.close()
    del fb
    gc.collect()

    # ── 5. 2차 이진 학습셋 (1차가 8로 보낸 거래) ──────────────────────────────
    s2 = out / 'stage2_binary'
    s2.mkdir(exist_ok=True)
    for s, (lo, hi) in enumerate(bounds):
        tag = SPLIT_TAGS[s]
        loc = np.flatnonzero(y9[lo:hi] == CLASS8).astype(np.int64)   # split 내부 인덱스
        np.save(s2 / f'idx_{tag}.npy', loc)
        np.save(s2 / f'y2_{tag}.npy', a['is_pos'][lo:hi][loc].astype(np.int8))
        print(f'[stage2] {tag}: 대상 {len(loc):,} · 양성(패턴외 세탁) '
              f'{int(a["is_pos"][lo:hi][loc].sum()):,}')

    # ── 6. 클러스터 기반 언더샘플링 (train 전용) ──────────────────────────────
    und = out / 'undersample'
    und.mkdir(exist_ok=True)
    y9tr, postr = y9[:b1], a['is_pos'][:b1]
    normal_tr = np.flatnonzero((y9tr == CLASS8) & (~postr))     # 정상만 줄인다
    keep_always = np.flatnonzero((y9tr != CLASS8) | postr)      # 패턴 + 패턴외 세탁
    n_pattern_tr = int((y9tr <= 7).sum())
    oop_tr = np.flatnonzero((y9tr == CLASS8) & postr)
    print(f'[undersample] mode={UNDERSAMPLE_MODE} · train 정상 {len(normal_tr):,}행 · '
          f'패턴 {n_pattern_tr:,}행 · 항상 유지 {len(keep_always):,}행')
    k_sweep, k_best, km_stat, lab, dist = [], None, None, None, None
    urep, s2rep = [], []
    if UNDERSAMPLE_MODE != 'none':
        if UNDERSAMPLE_MODE == 'cluster':
            Xtr = np.load(s1 / 'X_tr.npy', mmap_mode='r')
            Xn = np.asarray(Xtr[normal_tr])
            k_sweep = us.sweep_k(Xn, KMEANS_KS, seed=SEED)
            # K 는 회의 미확정. '양자화 오차가 최적 대비 10% 이내인 가장 작은 K' 규칙으로 고른다
            # — 무조건 큰 K 를 고르면(오차는 K 와 함께 단조 감소) 군집당 표본이 말라 층화가 깨진다.
            _best_err = min(r['inertia_per_row'] for r in k_sweep)
            k_best = int(min(r['k'] for r in k_sweep if r['inertia_per_row'] <= _best_err * 1.10))
            lab, dist, km_stat = us.fit_clusters(Xn, k_best, seed=SEED)
            del Xn, Xtr
            gc.collect()
        rng = np.random.default_rng(SEED)

        def _picks(n_t: int) -> list[tuple[str, np.ndarray]]:
            # 원래 순서(cluster 먼저, random 은 rng 한 번)를 유지해 cluster 모드 산출물이 그대로 재현된다
            out_ = []
            if lab is not None:
                out_.append(('cluster', normal_tr[us.sample_indices(lab, dist, n_t, seed=SEED)]))
            out_.append(('random', normal_tr[np.sort(rng.choice(len(normal_tr), n_t, replace=False))]))
            return out_

        for r in UNDERSAMPLE_RATIOS:
            n_t = min(r * n_pattern_tr, len(normal_tr))
            for nm, pick in _picks(n_t):
                idx = np.sort(np.concatenate([keep_always, pick]))
                np.save(und / f'{nm}_r{r}_tr.npy', idx.astype(np.int64))
                urep.append({'method': nm, 'ratio': r, 'n_normal': len(pick),
                             'n_total': len(idx), 'n_pattern': n_pattern_tr,
                             'n_oop_pos': int(len(oop_tr)),
                             'clusters_covered': int(len(np.unique(lab[np.searchsorted(
                                 normal_tr, pick)]))) if nm == 'cluster' else None})
        pd.DataFrame(urep).to_csv(und / 'variants.csv', index=False)
        if k_sweep:
            pd.DataFrame(k_sweep).to_csv(und / 'kmeans_sweep.csv', index=False)

        # 2차 이진도 불균형이 1차 못지않다(정상 : 패턴 외 세탁). 같은 규칙으로 변형을 만든다.
        for r in UNDERSAMPLE_RATIOS:
            n_t = min(r * len(oop_tr), len(normal_tr))
            for nm, pick in _picks(n_t):
                idx = np.sort(np.concatenate([oop_tr, pick]))
                np.save(und / f'stage2_{nm}_r{r}_tr.npy', idx.astype(np.int64))
                s2rep.append({'stage': 'stage2', 'method': nm, 'ratio': r,
                              'n_normal': len(pick), 'n_pos': len(oop_tr), 'n_total': len(idx)})
        pd.DataFrame(s2rep).to_csv(und / 'variants_stage2.csv', index=False)
        print(f'[undersample] 1차용 {len(urep)}벌 · 2차용 {len(s2rep)}벌 (양성 {len(oop_tr):,}건 전량 유지)')
    else:
        print('[undersample] 생략 (--undersample none)')

    # ── 7. 후처리 블록 — **정답으로 만든 오라클 상한** (§F) ────────────────────
    # 여기 저장되는 블록은 운영 산출물이 아니다. 1차 모델이 완벽하다고 가정했을 때
    # '계좌 공유 + 시간창' 묶기 규칙이 정답 시도를 얼마나 되살리는지 재는 상한선이다.
    # 그래서 파일 이름을 oracle_ 로 붙인다.
    blk = out / 'blocks'
    blk.mkdir(exist_ok=True)
    pos_rows = np.flatnonzero(y9 <= 7)                  # 8종 패턴 거래 = 패턴 블록 후보

    # 채점 자격: (a) 기간 절단으로 잘리지 않은 완결 시도, (b) 거래 2건 이상,
    #            (c) 한 split 안에 온전히 들어온 시도.
    # (b) 를 빼면 거래가 1건만 남은 시도가 어떤 창에서도 자동 성공이 돼 지표가 부풀려진다.
    declared = ns['PATTERN_ATTEMPTS']['n_edges_declared']
    ga = pd.DataFrame({'att': attempt[pos_rows], 'split': split[pos_rows],
                       'cls': y9[pos_rows]}).groupby('att').agg(
        n=('split', 'size'), s_min=('split', 'min'), s_max=('split', 'max'),
        cls=('cls', lambda v: int(v.mode().iloc[0])))
    ga['declared'] = declared.reindex(ga.index).to_numpy()
    ga['complete'] = ga['n'] == ga['declared']
    ga['contained'] = ga['s_min'] == ga['s_max']
    ga['eligible'] = ga['complete'] & ga['contained'] & (ga['n'] >= 2)
    ga.to_csv(blk / 'attempt_eligibility.csv')
    elig_by_split = {s_: ga.index[ga['eligible'] & (ga['s_min'] == s_)].to_numpy()
                     for s_ in range(3)}
    print(f'[block] 시도 {len(ga)} · 완결 {int(ga["complete"].sum())} · '
          f'split 내 완결+2건이상 {int(ga["eligible"].sum())} '
          f'(train {len(elig_by_split[0])} / val {len(elig_by_split[1])} / test {len(elig_by_split[2])})')

    def sweep_one(w: int, s_: int) -> tuple[dict, dict]:
        sel = pos_rows[split[pos_rows] == s_]           # 블록은 split 안에서만 만든다
        comp_ = pb.link_components(a['src_id'][sel], a['dst_id'][sel], a['ts_min'][sel], w)
        ev_ = pb.evaluate_blocks(comp_, attempt[sel], eligible=elig_by_split[s_],
                                 per_attempt=True)
        ev_ = {'n_blocks': 0, 'purity': float('nan'), 'frag_mean': float('nan'),
               'mixed_blocks': 0, 'strict_recovery': float('nan'),
               'attempt_ids': np.zeros(0), 'strict_flags': np.zeros(0, bool), **ev_}
        per_c = {}
        for c in range(8):
            ids = ga.index[(ga['cls'] == c) & ga['eligible'] & (ga['s_min'] == s_)].to_numpy()
            m_ = np.isin(ev_.get('attempt_ids', np.zeros(0)), ids)
            per_c[ns['CLASS_NAMES'][c]] = (float(ev_['strict_flags'][m_].mean())
                                           if m_.any() else float('nan'))
        return ev_, per_c

    sweep = []
    for w in BLOCK_WINDOWS:
        for s_, tag in enumerate(SPLIT_TAGS):
            ev_, per_c = sweep_one(w, s_)
            macro = float(np.nanmean(list(per_c.values()))) if per_c else float('nan')
            sweep.append({'window_min': w, 'split': tag,
                          'n_scored': ev_['n_scored'], 'n_blocks': ev_['n_blocks'],
                          'strict_recovery': round(ev_['strict_recovery'], 4),
                          'macro_strict_8class': round(macro, 4),
                          'purity': round(ev_['purity'], 4),
                          'frag_mean': round(ev_['frag_mean'], 4),
                          'mixed_blocks': ev_['mixed_blocks'],
                          **{k_: round(v_, 4) for k_, v_ in per_c.items()}})
    sw = pd.DataFrame(sweep)
    sw.to_csv(blk / 'window_sweep.csv', index=False)

    # 창은 **train 구간에서만** 고른다. 8종 macro 평균을 최대화하고, 동률이면 좁은 창.
    # (시도 단순평균으로 고르면 다수 클래스가 끌고 가 BIPARTITE 가 0 으로 눌린다 — §보고서)
    tr_sw = sw[sw['split'] == 'tr']
    _top = tr_sw['macro_strict_8class'].max()
    w_best = int(tr_sw.loc[tr_sw['macro_strict_8class'] >= _top - 1e-12, 'window_min'].min())
    print(sw[sw['split'] == 'tr'][['window_min', 'n_scored', 'n_blocks', 'strict_recovery',
                                   'macro_strict_8class', 'purity', 'frag_mean']]
          .to_string(index=False))
    print(f'[block] 채택 창 W={w_best}분 (train 8종 macro 기준)')

    rows_all, comp_all, off = [], [], 0
    for s_ in range(3):
        sel = pos_rows[split[pos_rows] == s_]
        c_ = pb.link_components(a['src_id'][sel], a['dst_id'][sel], a['ts_min'][sel], w_best)
        rows_all.append(sel)
        comp_all.append(c_ + off)
        off += (int(c_.max()) + 1) if len(c_) else 0
    rows_all, comp_all = np.concatenate(rows_all), np.concatenate(comp_all)
    np.savez_compressed(blk / f'oracle_blocks_W{w_best}.npz', rows=rows_all, comp=comp_all,
                        y9=y9[rows_all], attempt=attempt[rows_all], split=split[rows_all],
                        ts=a['ts_min'][rows_all],
                        note=np.array(['정답 라벨로 만든 오라클 블록. 블록은 split 안에서만 '
                                       '구성된다. 운영 산출물이 아니다.']))
    single = np.flatnonzero(is_oop)
    np.save(blk / 'oracle_single_alert_rows.npy', single.astype(np.int64))
    n_blk_total = int(len(np.unique(comp_all)))
    print(f'[block] 오라클 패턴 블록 {n_blk_total:,} · 단건 알림 후보 {len(single):,}')

    # ── 8. BIPARTITE vs STACK 심층 분석 재료 (§G) ─────────────────────────────
    eda = out / 'eda'
    eda.mkdir(exist_ok=True)
    struct = []
    for c in range(8):
        m = np.flatnonzero(y9 == c)
        at = attempt[m]
        ua = np.unique(at[at >= 0])
        comp_inf = pb.link_components(a['src_id'][m], a['dst_id'][m], a['ts_min'][m],
                                      10 ** 9)          # 창 무제한
        ev_inf = pb.evaluate_blocks(comp_inf, at)
        sizes = np.bincount(at[at >= 0] - at[at >= 0].min()) if len(ua) else np.zeros(1)
        sizes = sizes[sizes > 0]
        acc = [len(np.unique(np.concatenate([a['src_id'][m][at == q], a['dst_id'][m][at == q]])))
               for q in ua]
        dur = [int(a['ts_min'][m][at == q].max() - a['ts_min'][m][at == q].min()) for q in ua]
        struct.append({
            'class': c, 'name': ns['CLASS_NAMES'][c], 'n_edges': len(m), 'n_attempts': len(ua),
            'edges_per_attempt_median': float(np.median(sizes)),
            'accounts_per_attempt_median': float(np.median(acc)) if acc else 0.0,
            'duration_min_median': float(np.median(dur)) if dur else 0.0,
            '창무제한_블록수': ev_inf.get('n_blocks', 0),
            '창무제한_시도당조각': round(ev_inf.get('frag_mean', 0.0), 3),
            '창무제한_strict복원율': round(ev_inf.get('strict_recovery', 0.0), 4),
        })
    st_df = pd.DataFrame(struct)
    st_df.to_csv(eda / 'per_class_structure.csv', index=False)
    print('\n[eda] 클래스별 구조 (창 무제한에서도 못 묶이면 후처리로 복원 불가)')
    print(st_df.to_string(index=False))

    # 원시 거래 덤프 — 담당자가 원본 CSV 와 바로 대조할 수 있어야 한다.
    # 재부여된 정수 id·오프셋 시각만 주면 원본을 못 찾으므로 원본 (Bank, Account) 와
    # 원본 timestamp 를 복원해 함께 싣는다. 시도는 앞에서 6개가 아니라 고르게 뽑는다.
    new2old = np.full(n_nodes, -1, dtype=np.int64)
    _old_used = np.flatnonzero(old2new >= 0)
    new2old[old2new[_old_used]] = _old_used
    node_key = vocab_keys['nodes'].index.to_numpy()
    ts0 = split_info['ts_offset_epoch_min']

    def orig_acct(nid: int) -> str:
        o = int(new2old[nid])
        return str(node_key[o]) if o >= 0 else f'<미등재 new_id={nid}>'

    dump = []
    for c in (5, 6):                                    # BIPARTITE, STACK
        m = np.flatnonzero(y9 == c)
        ua_all = np.unique(attempt[m][attempt[m] >= 0])
        pick = ua_all[np.linspace(0, len(ua_all) - 1, min(8, len(ua_all))).astype(int)] \
            if len(ua_all) else ua_all
        for q in pick:
            r = m[attempt[m] == q]
            for i in r[np.argsort(a['ts_min'][r])]:
                dump.append({
                    'class': ns['CLASS_NAMES'][c], 'attempt_id': int(q),
                    'timestamp': str(np.datetime64(0, 'm')
                                     + np.timedelta64(int(a['ts_min'][i]) + ts0, 'm')),
                    'day': int(day_idx[i]),
                    'from_bank_account': orig_acct(int(a['src_id'][i])),
                    'to_bank_account': orig_acct(int(a['dst_id'][i])),
                    'amount_paid': float(a['paid'][i]),
                    'payment_currency': vocab_keys['pay'].index[a['pay_code'][i]],
                    'payment_format': vocab_keys['fmt'].index[a['fmt_code'][i]],
                    'self_loop': bool(a['src_id'][i] == a['dst_id'][i]),
                    'split': SPLIT_TAGS[int(split[i])],
                    'global_row': int(i)})
    dmp = pd.DataFrame(dump).sort_values(['class', 'attempt_id', 'timestamp'])
    dmp.to_csv(eda / 'bipartite_stack_raw.csv', index=False)
    print(f'[eda] 원시 덤프 {len(dmp)}행 / 시도 {dmp["attempt_id"].nunique()}개 '
          f'-> {eda / "bipartite_stack_raw.csv"}')

    # ── 9. 색인 · meta · 자체 검증 ────────────────────────────────────────────
    # lowmem 에서는 ts_min·pair_id 가 int32 다. 파일 스키마는 HI-Small 산출물과 같게 int64 로 저장한다.
    np.savez_compressed(out / 'index_full.npz', ts_min=a['ts_min'].astype(np.int64),
                        src_id=a['src_id'], dst_id=a['dst_id'],
                        pair_id=a['pair_id'].astype(np.int64), split=split,
                        y9=y9, y_raw=y, is_pos=a['is_pos'], attempt=attempt, day=day_idx)

    checks = run_checks(a, y, y9, split, feat_names, s1, s2, und, bounds, basis, label_stats,
                        trim_info, ns['DATASET'])
    meta = {
        'created': time.strftime('%Y-%m-%d %H:%M:%S'), 'dataset': ns['DATASET'],
        'basis': basis, 'basis_note': cfg['note'], 'seed': SEED,
        'meeting': '2026-08-26 — 2단계 계층(1차 9-class 다중분류 / 2차 패턴외 이진)',
        'reused_from': {'notebook': str(nbcells.NB_PATH),
                        'cells': [c for c, _ in nbcells.DEF_CELLS],
                        'cell_sha1': nbcells.fingerprint()},
        'lowmem': ({'enabled': True, 'overrides': list(lm.OVERRIDE_NAMES),
                    'module': 'prep9/lowmem.py',
                    'why': 'cgroup RAM 27.3 GiB 상한. 노트북 원본은 그대로 두고 메모리 배치만 바꾼 함수로 갈아끼움. '
                           'HI-Small 로 기존 산출물과 동치 검증(compare_lowmem_small.py).',
                    'dtype_changes': 'ts_min·pair_id int32 (index_full.npz 저장 시 int64 복원), recv_cents 는 중복 판정 후 폐기'}
                   if LOWMEM else {'enabled': False}),
        'clean': clean_info, 'trim': trim_info, 'split': split_info, 'labels': label_stats,
        'class_hist_9': hist9,
        'class8_composition': {'normal': int(hist9[8] - int(is_oop.sum())),
                               'out_of_pattern_laundering': int(is_oop.sum())},
        'features': {'n': n_feat, 'names': feat_names, 'blocks': feat_blocks,
                     'absolute_time_cols': abs_time_cols},
        'undersample': {'mode': UNDERSAMPLE_MODE,
                        'k_sweep': k_sweep, 'k_chosen': k_best, 'kmeans': km_stat,
                        'ratios': list(UNDERSAMPLE_RATIOS), 'variants': urep,
                        'variants_stage2': s2rep, 'center_frac': 0.5,
                        'scope': ('train split 의 정상 거래만. val/test 는 손대지 않음. '
                                  '1차용(stage1 비율 = 정상:8종패턴)과 2차용(stage2 비율 = '
                                  '정상:패턴외세탁)을 따로 만든다')},
        'blocks': {'window_sweep': sweep, 'window_chosen': w_best,
                   'window_selected_on': 'train split · 8종 macro strict · 동률이면 좁은 창',
                   'n_pattern_blocks': n_blk_total,
                   'n_single_alert_candidates': int(len(single)),
                   'scoring_eligibility': '완결 시도 + 거래 2건 이상 + 한 split 안에 온전히 포함',
                   'attempts_total': int(len(ga)),
                   'attempts_complete': int(ga['complete'].sum()),
                   'attempts_eligible': int(ga['eligible'].sum()),
                   'kind': 'oracle — 정답 라벨로 구성. 운영 산출물이 아니다'},
        'isolated_filter': {
            **iso_stats, 'applied': False,
            'saved_splits': ['tr'],
            'reason': ('마스크만 저장하고 적용하지 않았다. (1) 이번 회의는 불균형 대응으로 '
                       '클러스터 언더샘플링을 택했고 라벨 파생 필터를 겹치면 두 효과가 섞인다. '
                       '(2) 이 마스크는 train/val/test 를 구분하지 않은 전체 그래프에서 '
                       '세탁 라벨로 계산되므로 **미래(val·test) 라벨의 함수**다. 절대 피처로 '
                       '쓰지 말고, 켜면 학습 분포가 서빙 분포와 달라진다는 점을 함께 보고한다.'),
            'derived_from': '전 구간 라벨(미래 정보 포함)'},
        'assumptions': [
            'USD_RATE 는 2026-08-25 노트북의 팀 미확정 가정값을 그대로 물려받았다.',
            (f'클러스터 K={k_best} 는 회의에서 정한 값이 아니라 양자화 오차 스윕으로 골랐다.'
             if k_best is not None else
             f'언더샘플 mode={UNDERSAMPLE_MODE}: 클러스터 층화를 만들지 않았다(08-27 선택이 full 이라 '
             '쓰이는 곳이 없고, 이 세트 크기에서는 클러스터링이 병목이다).'),
            f'블록 시간창 W={w_best}분은 회의 미확정. 후속 검증 9번 대상.',
            '언더샘플 비율 10/30/100/300 은 회의 미확정 — 모델 담당자가 고르라고 4벌 만든다.',
            'center_frac=0.5 (군집 안에서 중심 근접 절반 + 무작위 절반) 는 회의 미확정 값이다. '
            '1.0 으로 올리면 경계 표본이 사라지고 0.0 이면 군집 층화 무작위와 같아진다.',
            'USE_ABSOLUTE_TIME_FEATS=True 를 08-25 노트북에서 그대로 물려받아 hour/dow 5개가 '
            'X 에 들어 있다. 화두 10·17 은 절대 시각을 쓰지 않기로 했으므로 학습 직전에 '
            'features.absolute_time_cols 열을 빼는 것이 팀 결정에 맞는다.',
            f'블록 창 W 는 train 구간의 8종 macro strict 로 골랐다(자격: 완결+2건이상+split 내 포함). '
            f'클래스별 최적 창이 서로 달라 단일 창으로 전부를 만족시킬 수 없다.',
        ],
        'open_risks': [
            '팀 문서 16:05 결정은 "패턴 외를 하나의 일관된 9번째 클래스로 가정하지 않는다"였다. '
            '이번 회의의 9-Class 는 그 경고를 덮어쓴 구조이므로 클래스 8 내부 이질성을 반드시 측정한다.',
            '언더샘플링은 사전확률을 바꾼다. 정밀도는 손대지 않은 val/test 전량에서만 잰다.',
            '2차 이진 학습셋(stage2_binary)은 1차의 예측이 아니라 정답 y9==8 로 모집단을 정한 '
            '**오라클 라우팅**이다. 운영에서는 1차가 8로 오분류한 패턴 거래가 섞여 들어오므로 '
            '분포가 다르다. 종단 성능은 1차 출력을 실제로 연결해 따로 재야 한다.',
            'blocks/ 의 블록과 단건 후보도 정답으로 만든 오라클 상한이다. 파일명 oracle_ 접두사 '
            '그대로 읽고, 운영 알림 성능으로 인용하지 않는다.',
            'evaluate_blocks 의 순도는 라벨된 시도끼리의 섞임만 잰다. 1차 오탐(정상 거래)이 '
            '블록에 섞이는 몫은 여기 안 잡히므로 실제 알림 순도는 이보다 낮다.',
        ],
        'checks': checks, 'elapsed_s': round(time.time() - t0, 1),
    }
    (out / 'features_meta.json').write_text(
        json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
    if LOWMEM:
        for p in (LOWMEM_SCRATCH / f'a_{basis}').glob('*.npy'):
            try:
                p.unlink()
            except OSError:
                pass
    print(f'[done] {basis} — {time.time() - t0:,.1f}s -> {out}')
    return meta


def run_checks(a, y, y9, split, feat_names, s1, s2, und, bounds, basis, label_stats,
               trim_info, dataset: str = 'HI-Small') -> list[dict]:
    """전처리가 팀이 이미 합의한 사실을 재현하는지, 누수가 없는지 자체 점검.

    18day 전용이던 팀 EDA 대조를 두 basis 모두에서 돌게 고쳤다(2026-08-27).
    10day는 꼬리 8일을 잘라내므로 팀 EDA 원값(5,177/3,209/1,968)과 원시 카운트가
    그대로 맞지 않는다 — 다만 '패턴 외' 1,968만은 전량이 1~10일 안에 있어 트림과
    무관하게 두 basis 모두 그대로 성립한다(재사용 노트북 셀 35의 보정식과 동일한 사실).
    거기에 unmatched_keys == positives_cut 항등식을 추가한다 — 이건 트림으로 잘려나간
    양성 행 수와, 그래서 패턴 매칭이 안 된 정답지 키 수가 같아야 한다는 뜻이라 계좌
    재부여(old2new)나 시간 오프셋이 어긋나면 두 basis 어느 쪽에서도 깨진다.
    """
    out = []

    def chk(name, ok, got, want=''):
        out.append({'검사': name, '통과': bool(ok), '실측': got, '기대': want})

    # 트림 정합은 세트와 무관한 항등식이라 항상 검사한다.
    chk('트림 정합 — unmatched_keys == positives_cut',
        label_stats['unmatched_keys'] == trim_info['positives_cut'],
        f"{label_stats['unmatched_keys']} vs {trim_info['positives_cut']}",
        '같아야 함 (꼬리로 잘린 양성 수 = 패턴 매칭 실패 키 수)')

    # 팀 EDA 대조는 HI-Small 실측 상수라 다른 세트에는 적용할 수 없다.
    # 상수를 그대로 들고 가면 정상인 실행이 FAIL 로 뜬다 — 건너뛰되 건너뛴 사실을 남긴다.
    exp = EDA_EXPECTED.get(dataset)
    if exp is None:
        chk(f'팀 EDA 대조 — {dataset} 기준값 없음(건너뜀)', True, '미대조',
            'EDA_EXPECTED 에 그 세트의 실측값을 넣으면 대조한다')
    else:
        is_full = BASES[basis]['tail_min_frac'] == 0.0        # 꼬리를 자르지 않은 기준
        chk('팀 EDA — 패턴 외 (트림 무관, 전량 주 기간 내)',
            label_stats['positives_out_of_pattern'] == exp['out_of_pattern'],
            label_stats['positives_out_of_pattern'], exp['out_of_pattern'])
        if is_full:
            chk('팀 EDA — 라벨1 총계', int(a['is_pos'].sum()) == exp['label1_total'],
                int(a['is_pos'].sum()), exp['label1_total'])
            chk('팀 EDA — 8종 소속', label_stats['positives_in_pattern'] == exp['in_pattern'],
                label_stats['positives_in_pattern'], exp['in_pattern'])
        else:
            chk('팀 EDA — 8종 소속 (트림 보정: in_pattern + unmatched_keys)',
                label_stats['positives_in_pattern'] + label_stats['unmatched_keys'] == exp['in_pattern'],
                label_stats['positives_in_pattern'] + label_stats['unmatched_keys'], exp['in_pattern'])
            chk('팀 EDA — 라벨1 총계 (트림 보정: positives + positives_cut)',
                int(a['is_pos'].sum()) + trim_info['positives_cut'] == exp['label1_total'],
                int(a['is_pos'].sum()) + trim_info['positives_cut'], exp['label1_total'])
    chk('9-class 정의: 8종은 0~7 그대로',
        bool(np.array_equal(y9[y >= 0], np.minimum(y[y >= 0], 8))), '일치', 'y in 0..7 -> y9 == y')
    chk('9-class 정의: 그 밖은 전부 8',
        bool((y9[(y < 0) | (y == 8)] == 8).all()), '일치', '정상·패턴외 -> 8')
    chk('2차 대상 = 클래스8 전량', int((y9 == 8).sum()) == int(((y < 0) | (y == 8)).sum()),
        int((y9 == 8).sum()), int(((y < 0) | (y == 8)).sum()))
    chk('라벨 파생 컬럼이 피처에 없음',
        not any(k in ' '.join(feat_names) for k in
                ('label', 'laundering', 'pattern', 'attempt', 'y9', 'is_pos')),
        '없음', '피처명에 라벨·패턴·시도 관련 문자열 없음')

    # 분할 경계는 '엄격히' 벌어져야 한다. 같은 분이 두 split 에 걸치면 동시 거래가 갈린다.
    t0m, t1m = int(a['ts_min'][split == 0].max()), int(a['ts_min'][split == 1].min())
    t1M, t2m = int(a['ts_min'][split == 1].max()), int(a['ts_min'][split == 2].min())
    chk('시간순 분할 — 경계가 엄격히 벌어짐(같은 분이 두 split 에 없음)',
        (t0m < t1m) and (t1M < t2m), f'tr_max {t0m} < va_min {t1m} · va_max {t1M} < te_min {t2m}',
        'train.max < val.min 이고 val.max < test.min')

    # as-of 인과성: 세 split 모두에서 확인한다(train 앞부분만 보면 뒷구간을 못 잡는다)
    ci = {nm: i for i, nm in enumerate(feat_names)}
    for k_, tag in enumerate(SPLIT_TAGS):
        X = np.load(s1 / f'X_{tag}.npy', mmap_mode='r')
        take = np.linspace(0, X.shape[0] - 1, min(200_000, X.shape[0])).astype(np.int64)
        Xs = np.asarray(X[take])
        first = Xs[:, ci['dt_src_first']] == 1
        mx = float(Xs[first, ci['src_out_cnt_24h']].max()) if first.any() else 0.0
        chk(f'엄격 과거(as-of) — 첫 송금에 과거 건수 0 [{tag}]', mx == 0.0, mx, 0.0)
        chk(f'피처 NaN/inf 없음 [{tag}]', bool(np.isfinite(Xs).all()), '유한', '유한')
        del X, Xs

    # 인덱스 좌표계: 저장한 인덱스가 자기 split 범위 안에 있고 올바른 행을 가리키는가
    for k_, tag in enumerate(SPLIT_TAGS):
        lo, hi = bounds[k_]
        loc = np.load(s2 / f'idx_{tag}.npy')
        y2 = np.load(s2 / f'y2_{tag}.npy')
        okr = (len(loc) == 0) or (loc.min() >= 0 and loc.max() < hi - lo)
        chk(f'2차 인덱스가 split 내부 좌표 [{tag}]', okr and (y9[lo:hi][loc] == 8).all(),
            f'0~{int(loc.max()) if len(loc) else -1} / 상한 {hi - lo - 1}', 'split 범위 내 · y9==8')
        chk(f'2차 라벨 = 원본 Is Laundering [{tag}]',
            bool(np.array_equal(y2, a['is_pos'][lo:hi][loc].astype(np.int8))), '일치', '일치')
    n_tr = bounds[0][1] - bounds[0][0]
    ytr, ptr = y9[:n_tr], a['is_pos'][:n_tr]
    must = np.flatnonzero((ytr <= 7) | ptr)
    for f_ in sorted(und.glob('*_tr.npy')):
        i_ = np.load(f_)
        stage2 = f_.name.startswith('stage2_')
        need = np.flatnonzero((ytr == 8) & ptr) if stage2 else must
        chk(f'언더샘플 인덱스 [{f_.name}]',
            bool(i_.min() >= 0 and i_.max() < n_tr and np.isin(need, i_).all()),
            f'0~{int(i_.max())} / 상한 {n_tr - 1} · 필수 {len(need):,}건 포함',
            'train 내부 좌표 · 양성 전량 유지')
    for r in out:
        print(f'  [{"OK " if r["통과"] else "FAIL"}] {r["검사"]}: {r["실측"]} (기대 {r["기대"]})')
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--dataset', default='HI-Small',
                    help='HI-Small / HI-Medium / HI-Large 등. 세트는 합치지 않는다(화두 7)')
    ap.add_argument('--basis', nargs='+', default=None, choices=list(BASES),
                    help='기본: HI-Small 은 10day·18day, 그 밖은 main·full')
    ap.add_argument('--scale', default='auto', choices=('auto', 'in_memory', 'chunked'),
                    help="auto 면 파일 크기로 고른다(2GiB 초과 -> chunked)")
    ap.add_argument('--out', default='/workspace/processed_9class')
    ap.add_argument('--undersample', default='cluster', choices=('cluster', 'random', 'none'),
                    help="cluster=원래 동작 / random=무작위 4벌만(피처 행렬 미적재) / none=생략")
    ap.add_argument('--max-chunks', type=int, default=0,
                    help='스모크용: chunked 적재를 앞 N 청크(N×2,000,000행)에서 멈춘다. '
                         '0 이면 전량. 켜면 --out 을 기본 경로와 다르게 줘야 한다.')
    ap.add_argument('--lowmem', action='store_true',
                    help='메모리 절약판(prep9/lowmem.py)으로 적재·정제·정렬·FeatureBuilder 를 갈아끼운다. '
                         'cgroup 27 GiB 상한용. 값은 동일(HI-Small 동치 검증).')
    args = ap.parse_args()
    global UNDERSAMPLE_MODE, LOWMEM, LOWMEM_SCRATCH
    UNDERSAMPLE_MODE = args.undersample
    LOWMEM = args.lowmem
    LOWMEM_SCRATCH = Path(args.out) / '_lowmem_scratch'
    if args.max_chunks and Path(args.out).resolve() == Path('/workspace/processed_9class').resolve():
        ap.error('--max-chunks 는 스모크 전용이다. 정식 산출물 경로를 덮어쓰지 않도록 --out 을 따로 준다')

    basis_list = args.basis or DEFAULT_BASIS.get(args.dataset, DEFAULT_BASIS_OTHER)
    for b in basis_list:                     # 기간을 이름에 박은 별칭은 그 세트에서만 허용
        only = BASES[b].get('only')
        if only and only != args.dataset:
            ap.error(f"basis '{b}' 는 {only} 전용 이름이다. {args.dataset} 에는 "
                     f"'main'/'full' 을 쓴다 — 기간이 다른데 이름만 같으면 산출물이 거짓말을 한다")

    print(f'[reuse] 2026-08-25 검증 통과 전처리 코드 적재 · dataset={args.dataset}')
    # 노트북을 수정하지 않고 네임스페이스에서 갈아끼운다(sha1 게이트 유지). 셀 4 가
    # 이 값으로 TRANS_PATH·PATTERNS_PATH 와 SCALE_MODE 를 확정한다.
    ns = nbcells.load(overrides={'DATASET': args.dataset})
    # 안전장치: 재사용 셀의 OUT_DIR 은 공용 산출물 /workspace/processed_multiclass 를 가리킨다.
    # in_memory 경로에서는 아무것도 쓰지 않지만, 실수로도 덮어쓰지 못하게 즉시 우회한다.
    ns['OUT_DIR'] = Path(args.out) / '_nb_scratch'
    ns['OUT_DIR'].mkdir(parents=True, exist_ok=True)
    if LOWMEM:
        scratch = Path(args.out) / '_lowmem_scratch'
        ns.update(lm.make_overrides(ns, scratch))
        ns['_LOWMEM_MAX_CHUNKS'] = args.max_chunks
        ns['SCALE_MODE'] = 'chunked'                    # lowmem 적재는 parquet 파트 경로만 있다
        print(f'[lowmem] override: {lm.OVERRIDE_NAMES} · scratch={scratch}')
    if args.max_chunks and not LOWMEM:
        # 노트북 셀은 못 고치므로(sha1 게이트) 그 셀이 보는 `pd` 만 감싼다: chunksize 로 읽을 때
        # 앞 N 청크에서 멈춘다. 나머지 pandas 기능은 그대로 통과한다.
        import itertools

        class _PdShim:
            def __init__(self, real, n):
                self._pd, self._n = real, n

            def __getattr__(self, k):
                return getattr(self._pd, k)

            def read_csv(self, *a, **k):
                it = self._pd.read_csv(*a, **k)
                return itertools.islice(it, self._n) if k.get('chunksize') else it

        ns['pd'] = _PdShim(pd, args.max_chunks)
        print(f'[smoke] chunked 적재를 앞 {args.max_chunks} 청크에서 멈춘다 -> {args.out}')
    t0 = time.time()
    # 적재 모드를 'in_memory' 로 못박으면 HI-Large(1.8억 행)에서 죽는다. 셀 4 가 파일 크기로
    # 정해 둔 SCALE_MODE 를 그대로 따르고, --scale 로만 덮어쓴다.
    mode = ns['SCALE_MODE'] if args.scale == 'auto' else args.scale
    print(f'[load] mode={mode} · {ns["TRANS_PATH"]}')
    raw_all, vocab_keys = ns['load_compact'](ns['TRANS_PATH'], mode)
    raw_all, clean_info = ns['clean_rows'](raw_all)
    n_nodes_raw = len(vocab_keys['nodes'])
    print(f'[load] {time.time() - t0:,.1f}s')

    out_root = Path(args.out)
    out_root.mkdir(parents=True, exist_ok=True)
    metas = {}
    for b in basis_list:
        metas[b] = run_basis(ns, raw_all, vocab_keys, n_nodes_raw, b, out_root, clean_info,
                             copy_raw=(len(basis_list) > 1))
    (out_root / 'run_summary.json').write_text(
        json.dumps({b: {k: m[k] for k in ('trim', 'split', 'class_hist_9',
                                          'class8_composition', 'blocks', 'checks')}
                    for b, m in metas.items()}, ensure_ascii=False, indent=2, default=str),
        encoding='utf-8')
    print(f'\n[all done] {time.time() - t0:,.1f}s -> {out_root}')


if __name__ == '__main__':
    main()


## 2. 언더샘플링 — `prep9/undersample.py`

클러스터(k-means) 기반 언더샘플링(2026-08-26 회의 결정 3) + 같은 크기 무작위 대조군. train split에만 적용, 표준화 통계도 train으로만 적합.

원본 경로: `prep9/undersample.py`

In [ ]:
"""클러스터 기반 언더샘플링 — 2026-08-26 회의 결정 3.

회의 문구: "정상 표본을 무작정 삭제하지 않고 대표값 위주로 정제하여 학습 안정성을 확보".

구현: 표준화한 피처 공간에서 MiniBatchKMeans 로 정상 거래를 K개 군집으로 나눈 뒤,
군집 크기에 비례해 각 군집에서 뽑는다(층화 추출). 군집 안에서는 중심에 가까운 순으로
`center_frac` 만큼을 대표값으로 먼저 채우고 나머지는 무작위로 채운다.

지켜야 할 것 세 가지 — 어기면 수치가 무의미해진다.
  1. **train split 에만 적용한다.** val/test 의 행을 지우면 평가 분포가 바뀌어
     다른 실험과 비교가 성립하지 않는다.
  2. **표준화 통계와 군집 중심은 train 으로만 적합한다.**
  3. **같은 크기의 무작위 대조군을 함께 만든다.** 대조군이 없으면
     "클러스터가 낫다"는 주장을 검증할 수 없다.

언더샘플링은 사전확률(prior)을 바꾼다. 언더샘플한 셋으로 잰 정밀도는 운영 정밀도가
아니므로, 평가는 반드시 손대지 않은 val/test 전량에서 한다.
"""
from __future__ import annotations

import numpy as np

__all__ = ['fit_clusters', 'sample_indices', 'sweep_k']


def _standardize(X: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ((X - mean) / (std + 1e-6)).astype(np.float32, copy=False)


def fit_clusters(X: np.ndarray, k: int, seed: int = 42, batch: int = 20_000,
                 fit_subsample: int | None = 1_000_000):
    """train 정상 행 -> (군집 배정, 중심까지 거리, 적합 통계).

    중심은 `fit_subsample` 행으로 적합하고 배정은 전 행에 한다. 중심 위치는 표본
    100만 행이면 충분히 안정적이고, 전 행 적합은 시간만 몇 배로 든다.
    """
    from sklearn.cluster import MiniBatchKMeans
    mean, std = X.mean(0), X.std(0)
    Z = _standardize(X, mean, std)
    km = MiniBatchKMeans(n_clusters=k, random_state=seed, batch_size=batch,
                         n_init=3, max_iter=100, reassignment_ratio=0.01)
    if fit_subsample is not None and len(Z) > fit_subsample:
        sub = np.random.default_rng(seed).choice(len(Z), fit_subsample, replace=False)
        km.fit(Z[sub])
        lab = km.predict(Z)
    else:
        lab = km.fit_predict(Z)
    d = np.linalg.norm(Z - km.cluster_centers_[lab], axis=1)
    n_fit = int(min(len(X), fit_subsample or len(X)))
    return lab.astype(np.int32), d.astype(np.float32), {
        'k': int(k), 'seed': int(seed), 'inertia': float(km.inertia_),
        'inertia_per_row': float(km.inertia_ / max(n_fit, 1)),   # 적합에 쓴 행 수로 나눈다
        'n_rows_assigned': int(len(X)),
        'n_rows_fit': int(min(len(X), fit_subsample or len(X))),
        'empty_clusters': int(k - len(np.unique(lab))),
        'cluster_size_min': int(np.bincount(lab, minlength=k).min()),
        'cluster_size_max': int(np.bincount(lab, minlength=k).max()),
    }


def sample_indices(lab: np.ndarray, dist: np.ndarray, n_target: int,
                   center_frac: float = 0.5, seed: int = 42) -> np.ndarray:
    """군집 크기 비례 층화 추출. 군집 내부는 (중심 근접 대표값 + 무작위) 혼합."""
    rng = np.random.default_rng(seed)
    n = len(lab)
    if n_target >= n:
        return np.arange(n)
    k = int(lab.max()) + 1
    sizes = np.bincount(lab, minlength=k)
    quota = np.floor(sizes * (n_target / n)).astype(np.int64)
    short = n_target - int(quota.sum())
    if short > 0:                                   # 남은 몫은 큰 군집부터 1개씩
        for c in np.argsort(sizes)[::-1][:short]:
            quota[c] += 1
    order = np.argsort(lab, kind='stable')
    offs = np.searchsorted(lab[order], np.arange(k + 1))
    out = []
    for c in range(k):
        mem = order[offs[c]:offs[c + 1]]
        q = int(min(quota[c], len(mem)))
        if q == 0:
            continue
        n_center = int(round(q * center_frac))
        near = mem[np.argsort(dist[mem], kind='stable')[:n_center]]
        rest = np.setdiff1d(mem, near, assume_unique=False)
        extra = rng.choice(rest, size=min(q - n_center, len(rest)), replace=False) \
            if q > n_center and len(rest) else np.zeros(0, dtype=mem.dtype)
        out.append(np.concatenate([near, extra]))
    return np.sort(np.concatenate(out)) if out else np.zeros(0, dtype=np.int64)


def sweep_k(X: np.ndarray, ks, seed: int = 42, subsample: int | None = 400_000) -> list[dict]:
    """K 후보별 양자화 오차. K 를 감으로 고르지 않기 위한 표."""
    rng = np.random.default_rng(seed)
    Xs = X if subsample is None or len(X) <= subsample else \
        X[rng.choice(len(X), subsample, replace=False)]
    rows = []
    for k in ks:
        _, _, st = fit_clusters(Xs, k, seed=seed)
        rows.append(st)
        print(f'  [kmeans] K={k:>5} inertia/row={st["inertia_per_row"]:.4f} '
              f'빈군집={st["empty_clusters"]} 크기 {st["cluster_size_min"]}~{st["cluster_size_max"]}')
    return rows


## 3. 메모리 절약판 유틸 — `prep9/lowmem.py`

cgroup RAM 27.3GiB 상한 안에서 HI-Large(1.8억 행)를 처리하기 위한 저메모리 전처리 함수 모음(2026-09-02).

원본 경로: `prep9/lowmem.py`

In [ ]:
"""메모리 절약판 전처리 함수 — 2026-09-02.

왜: 작업 컨테이너의 RAM 상한이 cgroup 27.3 GiB 다(`free` 의 503 GB 는 호스트 값). HI-Large(1.8억 행)는
`preprocess_multiclass.ipynb` 의 원래 함수로는 적재 직후 OOM 으로 죽는다. 노트북은 sha1 게이트로
보호되므로 **원본을 고치지 않고** `nbcells.load(overrides=)` 로 아래 함수들만 갈아끼운다.

바뀌는 것은 메모리 배치뿐이고 값은 같아야 한다:
  · load_compact       parquet 파트를 열 단위로 미리 잡은 배열에 채운다(concat 사본 없음). 계좌 사전을 저장해 재사용.
  · clean_rows         DataFrame.duplicated() 대신 키 3개를 int64 로 묶어 안정 lexsort — 결과(첫 등장 유지) 동일.
  · trim_tails / sort_split_reindex   열별 제자리 필터·정렬. ts_min·pair_id 는 int32 로 둔다(값은 동일).
  · FeatureBuilder     같은 피처식. paid_log·recv_log·usd_* 전체 배열을 만들지 않고 블록마다 즉석 계산.
                       PastIndex 의 prefix 합 7종은 디스크 memmap(페이지 캐시), order 는 만든 뒤 버린다.
                       prefix 합은 순차 누적이라 비트 동일. 그룹 평균·표준편차는 청크 합산이라 1e-15 수준 차이.

검증: 같은 코드로 HI-Small 을 돌려 기존 산출물과 대조한다(compare_lowmem_small.py).
"""
from __future__ import annotations

import gc
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

CHUNK = 5_000_000            # prefix 합·통계 계산 청크(행). 5M×8B = 40MB 임시.


def _log(msg):
    print(msg, flush=True)


def _bits(v: int) -> int:
    return int(v).bit_length()


# ═════════════════════════════════════════════════════════════════════════════
# 1. 적재 — parquet 파트 → 열 단위 채우기
# ═════════════════════════════════════════════════════════════════════════════
def make_load_compact(ns: dict):
    """ns 의 KeyDict/_prep_chunk/RAW_DTYPE/CSV_CHUNK_ROWS/OUT_DIR/DATASET 를 그대로 쓴다."""

    def load_compact(path: Path, mode: str):
        KeyDict, _prep_chunk = ns['KeyDict'], ns['_prep_chunk']
        part_dir = Path(ns['OUT_DIR']) / 'interim' / ns['DATASET']
        part_dir.mkdir(parents=True, exist_ok=True)
        max_chunks = int(ns.get('_LOWMEM_MAX_CHUNKS', 0) or 0)
        st = Path(path).stat()
        manifest = part_dir / 'manifest.json'
        reuse = False
        if manifest.exists() and not max_chunks:
            m = json.loads(manifest.read_text())
            reuse = (m.get('csv_size') == st.st_size and m.get('csv_mtime') == int(st.st_mtime)
                     and all((part_dir / f'{k}.parquet').exists() for k in ('nodes', 'fmt', 'pay', 'recv'))
                     and len(sorted(part_dir.glob('part-*.parquet'))) == m.get('n_parts'))
        if reuse:
            _log(f'[load] 파트 {m["n_parts"]}개·계좌 사전 재사용 ({part_dir})')
            dicts = {}
            for k in ('nodes', 'fmt', 'pay', 'recv'):
                vals = pd.read_parquet(part_dir / f'{k}.parquet')['key'].to_numpy(dtype=object)
                d = KeyDict(); d.index = pd.Index(vals, dtype=object); dicts[k] = d
            nodes, fmt, pay, recv = (dicts[k] for k in ('nodes', 'fmt', 'pay', 'recv'))
            n_total = int(m['n_rows'])
        else:
            for f in part_dir.glob('part-*.parquet'):
                f.unlink()
            nodes = KeyDict()
            fmt, pay, recv = KeyDict(ns['FMT_CANON']), KeyDict(ns['CCY_CANON']), KeyDict(ns['CCY_CANON'])
            n_total = 0
            it = pd.read_csv(path, dtype=ns['RAW_DTYPE'], chunksize=ns['CSV_CHUNK_ROWS'])
            for i, ch in enumerate(it):
                if max_chunks and i >= max_chunks:
                    break
                part = _prep_chunk(ch, nodes, fmt, pay, recv)
                part.to_parquet(part_dir / f'part-{i:05d}.parquet', index=False, compression='zstd')
                n_total += len(part)
                _log(f'  [chunk {i:>3}] {len(part):,}행 누적 {n_total:,} / 계좌 {len(nodes):,}')
                del ch, part
                gc.collect()
            for k, d in (('nodes', nodes), ('fmt', fmt), ('pay', pay), ('recv', recv)):
                pd.DataFrame({'key': d.index.to_numpy(dtype=object)}).to_parquet(part_dir / f'{k}.parquet', index=False)
            if not max_chunks:
                manifest.write_text(json.dumps({'csv_size': st.st_size, 'csv_mtime': int(st.st_mtime),
                                                'n_rows': n_total,
                                                'n_parts': len(sorted(part_dir.glob('part-*.parquet')))}))
        parts = sorted(part_dir.glob('part-*.parquet'))
        # 열별로 채운다 — 한 번에 한 열의 파트 하나만 메모리에 뜬다
        dtypes = {'ts_epoch_min': np.int64, 'src_id': np.int32, 'dst_id': np.int32,
                  'cents': np.int64, 'recv_cents': np.int64, 'paid': np.float64, 'recv': np.float64,
                  'fmt_code': np.int16, 'pay_code': np.int16, 'recv_code': np.int16, 'is_pos': np.int8}
        raw = {}
        try:                                               # arrow 메모리 풀이 해제한 메모리를 붙들지 않게
            import pyarrow as pa
            import pyarrow.parquet as pq
            pool = pa.default_memory_pool()

            def read_col(f, c):
                v = pq.read_table(f, columns=[c], memory_map=True).column(0).to_numpy()
                return v
        except Exception:                                  # pyarrow 가 없으면 pandas 로
            pool = None

            def read_col(f, c):
                return pd.read_parquet(f, columns=[c])[c].to_numpy()
        for c, dt in dtypes.items():
            arr = np.empty(n_total, dtype=dt)
            pos = 0
            for f in parts:
                v = read_col(f, c)
                arr[pos:pos + len(v)] = v
                pos += len(v)
                del v
            assert pos == n_total, (c, pos, n_total)
            raw[c] = arr
            if pool is not None:
                pool.release_unused()
            gc.collect()
        try:
            anon = int(next(l for l in open('/sys/fs/cgroup/memory.stat') if l.startswith('anon ')).split()[1])
            _log(f'[load] cgroup anon {anon / 2**30:.1f} GiB (열 채우기 완료)')
        except Exception:
            pass
        _log(f'[load] chunked(lowmem): {n_total:,}행 / 계좌 {len(nodes):,} / 수단 {len(fmt)} / '
             f'통화(지급) {len(pay)} / 통화(수취) {len(recv)}')
        return raw, {'nodes': nodes, 'fmt': fmt, 'pay': pay, 'recv': recv}

    return load_compact


# ═════════════════════════════════════════════════════════════════════════════
# 2. 정제 — 완전 중복(9키) 첫 등장 유지 + 비양수 금액 제거
# ═════════════════════════════════════════════════════════════════════════════
def _filter_inplace(raw: dict, keep: np.ndarray):
    for k in list(raw):
        raw[k] = raw[k][keep]
    gc.collect()


def clean_rows(raw: dict):
    n0 = len(raw['src_id'])
    ts = raw['ts_epoch_min']
    t0 = int(ts.min())
    tso = (ts - t0).astype(np.int64)
    b_t, b_s, b_d = _bits(int(tso.max())), _bits(int(raw['src_id'].max())), _bits(int(raw['dst_id'].max()))
    assert b_t + b_s + b_d <= 63, '키 packing 이 int64 를 넘는다'
    k1 = (tso << (b_s + b_d)) | (raw['src_id'].astype(np.int64) << b_d) | raw['dst_id'].astype(np.int64)
    del tso
    # 금액(cents)은 조 단위까지 있어(HI-Large) 다른 키와 한 int64 에 못 묶는다 — 키 4개로 정렬한다.
    # 코드 4개(fmt·pay·recv ≤ 31, is_pos 0/1)만 int32 하나로 묶는다.
    k4 = ((raw['fmt_code'].astype(np.int32) << 11) | (raw['pay_code'].astype(np.int32) << 6)
          | (raw['recv_code'].astype(np.int32) << 1) | raw['is_pos'].astype(np.int32))
    k2, k3 = raw['cents'], raw['recv_cents']
    order = np.lexsort((k4, k3, k2, k1))                   # 안정 정렬: 같은 키는 원래 순서
    dup = np.zeros(n0, dtype=bool)
    # 인접 키 비교는 청크로(전체 길이 임시 배열을 피한다). 같은 키 묶음의 첫 행만 남긴다(= duplicated()).
    for a in range(1, n0, CHUNK):
        b = min(a + CHUNK, n0)
        cur, prv = order[a:b], order[a - 1:b - 1]
        same = ((k1[cur] == k1[prv]) & (k2[cur] == k2[prv]) & (k3[cur] == k3[prv])
                & (k4[cur] == k4[prv]))
        dup[cur[same]] = True
    del k1, k4, order, same, cur, prv
    gc.collect()
    bad = (raw['paid'] <= 0) | (raw['recv'] <= 0)
    keep = ~dup & ~bad
    n_dup, n_bad = int(dup.sum()), int((bad & ~dup).sum())
    n_dup_pos = int((raw['is_pos'][dup] == 1).sum())
    n_bad_pos = int((raw['is_pos'][bad & ~dup] == 1).sum())
    del dup, bad
    _filter_inplace(raw, keep)
    del raw['recv_cents']                                   # 중복 판정에만 쓰인다
    _log(f'[clean] {n0:,}행 -> 완전중복 {n_dup:,}(양성 {n_dup_pos:,}) / '
         f'비양수금액 {n_bad:,}(양성 {n_bad_pos:,}) 제거 -> {len(raw["src_id"]):,}행')
    return raw, {'rows_in': n0, 'exact_duplicates': n_dup, 'exact_duplicates_positive': n_dup_pos,
                 'nonpositive_amount': n_bad, 'nonpositive_amount_positive': n_bad_pos,
                 'rows_out': int(len(raw['src_id']))}


# ═════════════════════════════════════════════════════════════════════════════
# 3. 꼬리 절단 · 정렬 · 분할 · 재부여 (열별 제자리)
# ═════════════════════════════════════════════════════════════════════════════
def make_trim_tails(ns: dict):
    def trim_tails(raw: dict):
        day = raw['ts_epoch_min'] // 1440
        uniq, cnt = np.unique(day, return_counts=True)
        thr = ns['TAIL_MIN_FRAC'] * float(cnt.max())
        ok = uniq[cnt >= thr]
        lo, hi = int(ok.min()), int(ok.max())
        keep = (day >= lo) & (day <= hi)
        del day
        n_cut, n_cut_pos = int((~keep).sum()), int((raw['is_pos'][~keep] == 1).sum())
        d0 = np.datetime64(0, 'D') + np.timedelta64(lo, 'D')
        d1 = np.datetime64(0, 'D') + np.timedelta64(hi, 'D')
        _log(f'[trim] 임계 {thr:,.0f}건/일 -> 유지 {d0}~{d1} ({hi - lo + 1}일), '
             f'절단 {n_cut:,}행(양성 {n_cut_pos:,})')
        _filter_inplace(raw, keep)
        return raw, {'threshold_per_day': thr, 'kept_start': str(d0), 'kept_end': str(d1),
                     'kept_days': hi - lo + 1, 'rows_cut': n_cut, 'positives_cut': n_cut_pos}
    return trim_tails


def make_sort_split_reindex(ns: dict):
    def sort_split_reindex(raw: dict, n_nodes_raw: int):
        order = np.argsort(raw['ts_epoch_min'], kind='stable')
        for k in list(raw):
            raw[k] = raw[k][order]
        del order
        gc.collect()
        ts = raw['ts_epoch_min']
        n = len(ts)
        TRAIN_FRAC, VAL_FRAC = ns['TRAIN_FRAC'], ns['VAL_FRAC']
        t_val, t_te = ts[int(n * TRAIN_FRAC)], ts[int(n * (TRAIN_FRAC + VAL_FRAC))]
        assert t_val < t_te, '분할 경계 시각이 겹친다'
        split = np.full(n, 2, dtype=np.int8)
        split[ts < t_te] = 1
        split[ts < t_val] = 0
        raw['split'] = split
        cnt = np.bincount(split, minlength=3)

        # np.unique(concat) 과 같은 결과를 존재 비트맵으로 (정렬 없이)
        present = np.zeros(n_nodes_raw, dtype=bool)
        present[raw['src_id']] = True
        present[raw['dst_id']] = True
        used = np.flatnonzero(present)
        old2new = np.full(n_nodes_raw, -1, dtype=np.int32)
        old2new[used] = np.arange(len(used), dtype=np.int32)
        raw['src_id'] = old2new[raw['src_id']].astype(np.int32)
        raw['dst_id'] = old2new[raw['dst_id']].astype(np.int32)
        del present

        pair_key = raw['src_id'].astype(np.int64) * len(used) + raw['dst_id']
        _, pair_codes = np.unique(pair_key, return_inverse=True)
        del pair_key
        n_pairs = int(pair_codes.max()) + 1
        raw['pair_id'] = pair_codes.astype(np.int32 if n_pairs < 2**31 else np.int64)
        del pair_codes
        raw['ts_min'] = (ts - ts.min()).astype(np.int32)
        gc.collect()

        dv = np.datetime64(0, 'm') + np.timedelta64(int(t_val), 'm')
        dt_ = np.datetime64(0, 'm') + np.timedelta64(int(t_te), 'm')
        _log(f'[split] train {cnt[0]:,} / val {cnt[1]:,} / test {cnt[2]:,} (경계 {dv} / {dt_})')
        _log(f'[ids] 계좌 {len(used):,} / 계좌쌍 {n_pairs:,}')
        return raw, old2new, {
            'val_start': str(dv), 'test_start': str(dt_), 'rows': cnt.tolist(),
            'fracs': [TRAIN_FRAC, VAL_FRAC, round(1 - TRAIN_FRAC - VAL_FRAC, 4)],
            'boundary_rule': '경계 분(minute)은 뒤 구간에 귀속',
            'n_accounts': int(len(used)), 'n_pairs': n_pairs,
            'ts_offset_epoch_min': int(ts.min())}
    return sort_split_reindex


# ═════════════════════════════════════════════════════════════════════════════
# 4. PastIndex — key 는 RAM, prefix 합은 memmap, order 는 만든 뒤 버림
# ═════════════════════════════════════════════════════════════════════════════
class PastIndexLM:
    def __init__(self, ent: np.ndarray, ts: np.ndarray, tag: str, scratch: Path):
        self.bits = int(ts.max()).bit_length() + 1
        self.mask = (np.int64(1) << self.bits) - 1
        assert int(ent.max()) < (1 << (62 - self.bits)), 'ent<<B 가 int64 범위를 초과'
        key = ent.astype(np.int64)                         # 제자리 연산으로 임시 배열을 안 만든다
        key <<= self.bits
        key |= ts                                          # int32 → ufunc 내부 버퍼로 캐스팅
        order = np.argsort(key, kind='stable')             # == lexsort((ts, ent)) (안정, 같은 키 순서 유지)
        self.key_s = key[order]
        del key
        self.order = order.astype(np.int32) if len(order) < 2**31 else order
        del order
        gc.collect()
        self.n, self.tag, self.scratch = len(self.key_s), tag, Path(scratch)
        self.csums: dict[str, np.ndarray] = {}

    def _key(self, ent, ts):
        return (ent.astype(np.int64, copy=False) << self.bits) | ts.astype(np.int64, copy=False)

    def locate(self, ent, ts):
        lo = np.searchsorted(self.key_s, ent.astype(np.int64, copy=False) << self.bits, side='left')
        hi = np.searchsorted(self.key_s, self._key(ent, ts), side='left')
        return lo, hi

    def window_lo(self, ent, ts, w):
        t0 = np.maximum(ts.astype(np.int64, copy=False) - w, 0)
        return np.searchsorted(self.key_s, self._key(ent, t0), side='left')

    def prev_ts(self, lo, hi):
        has = hi > lo
        prev = self.key_s[np.maximum(hi - 1, 0)] & self.mask
        return np.where(has, prev, -1), has

    def csum_fn(self, name: str, xfun, dtype=np.float64) -> np.ndarray:
        """정렬 순서 prefix 합(선두 0)을 **RAM** 에 순차 누적으로 만든다.
        xfun(idx) -> 그 행들의 float64 값. 누적은 float64 로 앞에서부터 한 번씩 더하므로 전체 cumsum 과 비트 동일.
        dtype=float32 는 값이 정수(차수 카운트, ≤ 계좌쌍 수)일 때만 쓴다 — 2^24 까지 정확.

        왜 RAM 인가: 디스크 memmap 은 27 GiB 상한의 페이지 캐시에 안 들어가 조회마다 디스크를 쳤다
        (HI-Large 실측: 188 GB 읽고 X 의 20% 만 씀). 대신 순차 접근인 `a` 배열을 memmap 으로 내린다."""
        assert self.order is not None, 'order 를 이미 버렸다'
        c = np.empty(self.n + 1, dtype=dtype)
        c[0] = 0.0
        carry = np.zeros(1, dtype=np.float64)
        for a in range(0, self.n, CHUNK):
            b = min(a + CHUNK, self.n)
            x = np.asarray(xfun(self.order[a:b]), dtype=np.float64)
            cs = np.cumsum(np.concatenate([carry, x]))[1:]
            c[a + 1:b + 1] = cs
            carry[0] = cs[-1]
        self.csums[name] = c
        return c

    def drop_order(self):
        self.order = None
        gc.collect()


# ═════════════════════════════════════════════════════════════════════════════
# 5. FeatureBuilder — 같은 피처식, 즉석 계산
# ═════════════════════════════════════════════════════════════════════════════
def make_feature_builder(ns: dict, scratch: Path):
    fit_code_cols, group_z, one_hot = ns['fit_code_cols'], ns['group_z'], ns['one_hot']
    W_1H, W_24H, Z_CLIP, EPS = ns['W_1H'], ns['W_24H'], ns['Z_CLIP'], ns['EPS']
    NIGHT_END_HOUR, HIGH_RISK_FMT = ns['NIGHT_END_HOUR'], ns['HIGH_RISK_FMT']
    USE_ABS, USD_FEATURES = ns['USE_ABSOLUTE_TIME_FEATS'], ns['USD_FEATURES']
    scratch = Path(scratch)
    scratch.mkdir(parents=True, exist_ok=True)

    def chunked_group_stats(col_fn, x_fn, tr: np.ndarray, k: int):
        """fit_group_stats 와 같은 식. train 행만, 청크 합산."""
        n = len(tr)
        cnt = np.zeros(k, dtype=np.float64); s = np.zeros(k); ss = np.zeros(k)
        g_s = 0.0; g_ss = 0.0; g_n = 0
        for a in range(0, n, CHUNK):
            b = min(a + CHUNK, n)
            m = tr[a:b]
            if not m.any():
                continue
            q = np.arange(a, b)[m]
            col, x = col_fn(q), x_fn(q)
            ok = col >= 0
            cnt += np.bincount(col[ok], minlength=k)
            s += np.bincount(col[ok], weights=x[ok], minlength=k)
            ss += np.bincount(col[ok], weights=x[ok] * x[ok], minlength=k)
            g_s += float(x.sum()); g_ss += float((x * x).sum()); g_n += len(x)
        g_mean = g_s / max(g_n, 1)
        g_std = float(np.sqrt(max(g_ss / max(g_n, 1) - g_mean ** 2, 0.0)))
        mean = np.where(cnt > 0, s / np.maximum(cnt, 1.0), g_mean)
        var = np.where(cnt > 1, ss / np.maximum(cnt, 1.0) - mean ** 2, g_std ** 2)
        return np.append(mean, g_mean), np.append(np.sqrt(np.maximum(var, 0.0)), g_std)

    class FeatureBuilderLM:
        def __init__(self, a: dict, keys: dict):
            self.a = a
            t0 = time.time()
            tr = a['split'] == 0
            self.pay_rate, self.recv_rate = a['_pay_rate'], a['_recv_rate']
            # 구축 동안만 paid/recv 를 RAM 에 복사한다(prefix 합은 정렬 순서로 무작위 접근). 끝나면 버린다.
            self._paid = np.array(a['paid'], dtype=np.float64, copy=True)
            self._recv = np.array(a['recv'], dtype=np.float64, copy=True)
            self.fmt_col, self.fmt_names = fit_code_cols(a['fmt_code'], tr, list(keys['fmt'].index))
            self.pay_col, self.pay_names = fit_code_cols(a['pay_code'], tr, list(keys['pay'].index))
            self.recv_col, self.recv_names = fit_code_cols(a['recv_code'], tr, list(keys['recv'].index))
            self.pay_mean, self.pay_std = chunked_group_stats(
                lambda q: self.pay_col[a['pay_code'][q]], lambda q: self.paid_log(q), tr, len(self.pay_names))
            self.recv_mean, self.recv_std = chunked_group_stats(
                lambda q: self.recv_col[a['recv_code'][q]], lambda q: self.recv_log(q), tr, len(self.recv_names))
            # usd_mean/std: log1p(usd_paid) 의 train 평균·표준편차 (청크 2-pass)
            n = len(tr); s = 0.0; cnt = 0
            for st_ in range(0, n, CHUNK):
                q = np.arange(st_, min(st_ + CHUNK, n))[tr[st_:st_ + CHUNK]]
                if len(q): v = np.log1p(self.usd_paid(q)); s += float(v.sum()); cnt += len(v)
            self.usd_mean = s / max(cnt, 1)
            ss = 0.0
            for st_ in range(0, n, CHUNK):
                q = np.arange(st_, min(st_ + CHUNK, n))[tr[st_:st_ + CHUNK]]
                if len(q): v = np.log1p(self.usd_paid(q)) - self.usd_mean; ss += float((v * v).sum())
            self.usd_std = float(np.sqrt(ss / max(cnt, 1)))
            self.risk_cols = [self.fmt_col[c] for c, nm in enumerate(keys['fmt'].index)
                              if nm in HIGH_RISK_FMT and self.fmt_col[c] >= 0]
            _log(f'[stats] 그룹 통계 완료 [{time.time() - t0:.0f}s]')

            _log('[index] PastIndex 3종 구축 중… (lowmem)')
            self.i_src = PastIndexLM(a['src_id'], a['ts_min'], 'src', scratch)
            self.i_dst = PastIndexLM(a['dst_id'], a['ts_min'], 'dst', scratch)
            self.i_pair = PastIndexLM(a['pair_id'], a['ts_min'], 'pair', scratch)
            ps = self.i_pair.key_s >> self.i_pair.bits
            first_sorted = np.empty(len(ps), dtype=bool)
            first_sorted[0] = True
            first_sorted[1:] = ps[1:] != ps[:-1]
            del ps
            pair_first = np.empty(len(first_sorted), dtype=bool)
            pair_first[self.i_pair.order] = first_sorted
            del first_sorted
            self.i_pair.drop_order()

            _log('[index] prefix 합 계산 중… (RAM)')
            # 차수 카운트는 정수(≤ 계좌쌍 수 < 2^24)라 float32 로도 정확 — 1.4 GB 절약
            self.c_deg_s = self.i_src.csum_fn('deg', lambda i: pair_first[i].astype(np.float64), np.float32)
            self.c_deg_d = self.i_dst.csum_fn('deg', lambda i: pair_first[i].astype(np.float64), np.float32)
            del pair_first
            self.c1s = self.i_src.csum_fn('c1', lambda i: self.paid_log(i))
            self.c2s = self.i_src.csum_fn('c2', lambda i: self.paid_log(i) ** 2)
            self.c_amt = self.i_src.csum_fn('amt', lambda i: self._paid[i])
            self.i_src.drop_order()
            self.c1d = self.i_dst.csum_fn('c1', lambda i: self.recv_log(i))
            self.c2d = self.i_dst.csum_fn('c2', lambda i: self.recv_log(i) ** 2)
            self.i_dst.drop_order()
            self._paid = None; self._recv = None                # 이후 transform 은 memmap 을 순차로 읽는다
            gc.collect()
            self.names, self.blocks = self._probe_names()
            _log(f'[feat] {len(self.names)}차원 = ' + ' + '.join(f'{k} {v}' for k, v in self.blocks.items())
                 + f'  [{time.time() - t0:.0f}s]')

        # ── 즉석 파생값 (원래 코드의 전체 배열과 원소 단위로 같은 식) ──
        def _p(self): return self._paid if self._paid is not None else self.a['paid']
        def _r(self): return self._recv if self._recv is not None else self.a['recv']
        def paid_log(self, q): return np.log1p(self._p()[q])
        def recv_log(self, q): return np.log1p(self._r()[q])
        def usd_paid(self, q): return self._p()[q] * self.pay_rate[self.a['pay_code'][q]]
        def usd_recv(self, q): return self._r()[q] * self.recv_rate[self.a['recv_code'][q]]

        def _probe_names(self):
            _, names, blocks = self._compute(np.arange(min(8, len(self.a['ts_min']))), want_names=True)
            return names, blocks

        def _hist_z(self, c1, c2, lo, hi, x_q):
            cnt = (hi - lo).astype(np.float64)
            mean = (c1[hi] - c1[lo]) / np.maximum(cnt, 1.0)
            var = np.maximum((c2[hi] - c2[lo]) / np.maximum(cnt, 1.0) - mean ** 2, 0.0)
            z = np.where(cnt >= 2, (x_q - mean) / (np.sqrt(var) + EPS), 0.0)
            return np.clip(z, -Z_CLIP, Z_CLIP)

        def _compute(self, q: np.ndarray, want_names: bool = False):
            a = self.a
            feats = []
            add = lambda nm, arr: feats.append((nm, np.asarray(arr, dtype=np.float32)))
            safe = lambda s: re.sub(r'\W+', '_', s)
            paid_log_q, recv_log_q = self.paid_log(q), self.recv_log(q)

            pay_q, recv_q = self.pay_col[a['pay_code'][q]], self.recv_col[a['recv_code'][q]]
            fmt_q = self.fmt_col[a['fmt_code'][q]]
            add('amt_paid_log', paid_log_q)
            add('amt_recv_log', recv_log_q)
            add('amt_z_pay_ccy', group_z(paid_log_q, pay_q, self.pay_mean, self.pay_std))
            add('amt_z_recv_ccy', group_z(recv_log_q, recv_q, self.recv_mean, self.recv_std))
            add('ccy_mismatch', a['ccy_mismatch'][q])
            add('amt_mismatch', a['amt_mismatch'][q])
            for prefix, col_q, nms in (('fmt', fmt_q, self.fmt_names),
                                       ('pccy', pay_q, self.pay_names),
                                       ('rccy', recv_q, self.recv_names)):
                oh = one_hot(col_q, len(nms))
                for j, v in enumerate(nms):
                    add(f'{prefix}_{safe(v)}', oh[:, j])
            if USE_ABS:
                hr, dw = a['hour'][q].astype(np.float64), a['dow'][q].astype(np.float64)
                add('hour_sin', np.sin(2 * np.pi * hr / 24.0))
                add('hour_cos', np.cos(2 * np.pi * hr / 24.0))
                add('dow_sin', np.sin(2 * np.pi * dw / 7.0))
                add('dow_cos', np.cos(2 * np.pi * dw / 7.0))
                add('is_weekend', dw >= 5)
            n_base = len(feats)

            if USD_FEATURES:
                up, ur = self.usd_paid(q), self.usd_recv(q)
                up_log = np.log1p(up)
                add('usd_paid_log', up_log)
                add('usd_recv_log', np.log1p(ur))
                add('usd_ratio_log', np.log((ur + 1.0) / (up + 1.0)))
                add('usd_gap_rel', np.clip(np.abs(up - ur) / (up + EPS), 0.0, 10.0))
                add('usd_paid_z', np.clip((up_log - self.usd_mean) / (self.usd_std + EPS), -Z_CLIP, Z_CLIP))
            n_usd = len(feats) - n_base

            sq, dq, pq, tq = a['src_id'][q], a['dst_id'][q], a['pair_id'][q], a['ts_min'][q].astype(np.int64)
            s_lo, s_hi = self.i_src.locate(sq, tq)
            d_lo, d_hi = self.i_dst.locate(dq, tq)
            p_lo, p_hi = self.i_pair.locate(pq, tq)
            s_w1, s_w24 = self.i_src.window_lo(sq, tq, W_1H), self.i_src.window_lo(sq, tq, W_24H)
            d_w1, d_w24 = self.i_dst.window_lo(dq, tq, W_1H), self.i_dst.window_lo(dq, tq, W_24H)
            p_w1, p_w24 = self.i_pair.window_lo(pq, tq, W_1H), self.i_pair.window_lo(pq, tq, W_24H)
            _, si_hi = self.i_dst.locate(sq, tq)
            si_w24 = self.i_dst.window_lo(sq, tq, W_24H)
            _, do_hi = self.i_src.locate(dq, tq)
            do_w24 = self.i_src.window_lo(dq, tq, W_24H)

            src_out_1h, src_out_24 = s_hi - s_w1, s_hi - s_w24
            dst_in_1h, dst_in_24 = d_hi - d_w1, d_hi - d_w24
            dt_src, s_has = self.i_src.prev_ts(s_lo, s_hi)
            dt_dst, d_has = self.i_dst.prev_ts(d_lo, d_hi)
            dt_pair, p_has = self.i_pair.prev_ts(p_lo, p_hi)
            dt_src = np.where(s_has, (tq - dt_src) * 60.0, 0.0)
            dt_dst = np.where(d_has, (tq - dt_dst) * 60.0, 0.0)
            dt_pair = np.where(p_has, (tq - dt_pair) * 60.0, 0.0)

            add('dt_src_log', np.log1p(dt_src));   add('dt_src_first', ~s_has)
            add('dt_dst_log', np.log1p(dt_dst));   add('dt_dst_first', ~d_has)
            add('dt_pair_log', np.log1p(dt_pair)); add('dt_pair_first', ~p_has)
            add('src_out_cnt_1h', np.log1p(src_out_1h))
            add('src_out_cnt_24h', np.log1p(src_out_24))
            add('dst_in_cnt_1h', np.log1p(dst_in_1h))
            add('dst_in_cnt_24h', np.log1p(dst_in_24))
            add('pair_cnt_1h', np.log1p(p_hi - p_w1))
            add('pair_cnt_24h', np.log1p(p_hi - p_w24))
            add('src_in_cnt_24h', np.log1p(si_hi - si_w24))
            add('dst_out_cnt_24h', np.log1p(do_hi - do_w24))
            # 차수 prefix 합은 float32(정수 정확) — 차이는 정확한 정수이므로 float64 로 올려 log1p 하면 원래와 비트 동일
            add('src_hist_out_deg', np.log1p((self.c_deg_s[s_hi] - self.c_deg_s[s_lo]).astype(np.float64)))
            add('dst_hist_in_deg', np.log1p((self.c_deg_d[d_hi] - self.c_deg_d[d_lo]).astype(np.float64)))
            n_dt = len(feats) - n_base - n_usd

            paid_q = a['paid'][q]
            add('tb_structuring_10k', np.exp(-((paid_q - 9800.0) ** 2) / (2.0 * 300.0 ** 2)))
            add('tb_structuring_50k', np.exp(-((paid_q - 49000.0) ** 2) / (2.0 * 1000.0 ** 2)))
            tb_z_src = self._hist_z(self.c1s, self.c2s, s_lo, s_hi, paid_log_q)
            tb_z_dst = self._hist_z(self.c1d, self.c2d, d_lo, d_hi, recv_log_q)
            add('tb_amt_z_src', tb_z_src)
            add('tb_amt_z_dst', tb_z_dst)
            add('tb_vel_src', np.log((src_out_1h + 1.0) * 24.0 / (src_out_24 + 24.0)))
            add('tb_vel_dst', np.log((dst_in_1h + 1.0) * 24.0 / (dst_in_24 + 24.0)))
            add('tb_dormant_burst',
                np.tanh((dt_src / 86400.0) / 3.0) * np.tanh(np.maximum(tb_z_src, 0.0) / 2.0))
            add('tb_night_cash_burst', (a['hour'][q] < NIGHT_END_HOUR) | np.isin(fmt_q, self.risk_cols))
            add('tb_self_loop', sq == dq)
            add('tb_round_amt', a['round_amt'][q] if 'round_amt' in a else (a['cents'][q] % 10000 == 0))
            out_sum_24 = self.c_amt[s_hi] - self.c_amt[s_w24]
            add('tb_amt_share_src_24h', paid_q / (paid_q + out_sum_24))
            add('tb_pair_repeat', np.log1p(p_hi - p_lo))
            n_tb = len(feats) - n_base - n_usd - n_dt

            names = [f for f, _ in feats]
            X = np.column_stack([v for _, v in feats]).astype(np.float32, copy=False)
            assert np.isfinite(X).all(), '피처에 NaN/inf 존재'
            assert int(src_out_24[~s_has].sum()) == 0, 'PastIndex 인과성 위반(첫 송금에 과거 건수)'
            blocks = {'edge_base': n_base, 'usd': n_usd, 'dt_velocity': n_dt, 'behavior': n_tb}
            return (X, names, blocks) if want_names else (X, None, None)

        def transform(self, q: np.ndarray) -> np.ndarray:
            return self._compute(q)[0]

        def close(self):
            """prefix 합(RAM)·정렬 키 해제. 예전 memmap 파일이 남아 있으면 지운다."""
            for ix in (self.i_src, self.i_dst, self.i_pair):
                ix.csums.clear()
                ix.key_s = None
            for nm in ('c_deg_s', 'c_deg_d', 'c1s', 'c2s', 'c1d', 'c2d', 'c_amt'):
                setattr(self, nm, None)
            for p in scratch.glob('csum_*.npy'):
                try:
                    p.unlink()
                except OSError:
                    pass
            gc.collect()

    return FeatureBuilderLM


def make_overrides(ns: dict, scratch: Path) -> dict:
    """nbcells.load(overrides=) 에 넣을 함수 묶음. 값이 아니라 배치만 바뀐다."""
    return {'load_compact': make_load_compact(ns), 'clean_rows': clean_rows,
            'trim_tails': make_trim_tails(ns), 'sort_split_reindex': make_sort_split_reindex(ns),
            'FeatureBuilder': make_feature_builder(ns, scratch)}


OVERRIDE_NAMES = ('load_compact', 'clean_rows', 'trim_tails', 'sort_split_reindex', 'FeatureBuilder')


## 4. 산출물 좌표계 — `prep9/coords.py`

인덱스 배열이 전역(global, 정렬 기준 0..n-1)과 그 외 좌표계 두 종류라 섞어 쓰면 엉뚱한 거래를 집는 문제를 막는 유틸.

원본 경로: `prep9/coords.py`

In [ ]:
"""산출물 좌표계 — 인덱스 배열이 두 종류라 섞어 쓰면 엉뚱한 거래를 집는다.

  전역(global)  : 그 기준(basis) 전체 행에 대한 0..n-1. 시간순 정렬돼 있다.
                  `index_full.npz`, `blocks/*.npz['rows']`, `blocks/single_alert_rows.npy`
  분할내(local) : 그 split 안에서의 0..n_split-1. `X_tr/va/te.npy` 의 행 번호와 같다.
                  `stage2_binary/idx_*.npy`, `undersample/*_tr.npy`

전역 -> 분할내 변환은 아래 함수로만 한다. split 은 시간순 연속 구간이라 경계만 알면 된다.
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np

TAGS = ('tr', 'va', 'te')


def bounds(basis_dir: str | Path) -> list[tuple[int, int]]:
    """[(lo, hi)] × 3 — 전역 행 인덱스 기준 split 경계."""
    meta = json.loads((Path(basis_dir) / 'features_meta.json').read_text(encoding='utf-8'))
    n_tr, n_va, n_te = meta['split']['rows']
    return [(0, n_tr), (n_tr, n_tr + n_va), (n_tr + n_va, n_tr + n_va + n_te)]


def to_local(g: np.ndarray, basis_dir: str | Path) -> tuple[np.ndarray, np.ndarray]:
    """전역 인덱스 -> (split 번호 0/1/2, 그 split 안의 행 번호)."""
    bs = bounds(basis_dir)
    edges = np.array([b[0] for b in bs] + [bs[-1][1]])
    s = np.clip(np.searchsorted(edges, g, side='right') - 1, 0, 2)
    return s.astype(np.int8), (g - edges[s]).astype(np.int64)


def take_rows(g: np.ndarray, basis_dir: str | Path) -> np.ndarray:
    """전역 인덱스로 피처 행을 뽑는다(피처 재계산 없음, memmap 슬라이스)."""
    d = Path(basis_dir)
    s, loc = to_local(np.asarray(g), d)
    n_feat = len(json.loads((d / 'features_meta.json').read_text(
        encoding='utf-8'))['features']['names'])
    out = np.empty((len(g), n_feat), dtype=np.float32)
    for k, tag in enumerate(TAGS):
        m = s == k
        if not m.any():
            continue
        mm = np.load(d / 'stage1_9class' / f'X_{tag}.npy', mmap_mode='r')
        out[m] = mm[loc[m]]
        del mm
    return out


## 5. 학습셋 로더 — `train9/data.py`

1차 9-Class 학습셋 로더. 어떤 피처 열을 쓸지(all81 / no_abs=76열, 절대시각 제외)와 train 표본을 어떤 변형(언더샘플 8벌 / 전량+가중)으로 쓸지 결정.

원본 경로: `train9/data.py`

In [ ]:
"""1차 9-Class 학습셋 로더.

전처리 산출물(`processed_9class/HI-Small_{basis}/`)을 그대로 읽는다. 좌표계는
`processed_9class/README.md` 규칙을 따른다 — `undersample/*_tr.npy` 는 train 내부 인덱스다.

여기서 하는 판단은 두 가지뿐이다.
  1. 어떤 피처 열을 쓸 것인가 (`all81` / `no_abs`)
  2. train 표본을 어떤 변형으로 쓸 것인가 (언더샘플 8벌 / 전량+가중)
val·test 는 **어떤 경우에도 손대지 않는다.** 평가 분포가 바뀌면 실험 간 비교가 성립하지 않는다.
"""
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np

ROOT = Path('/workspace/processed_9class')
TAGS = ('tr', 'va', 'te')
CLASS_NAMES = ['FAN-OUT', 'FAN-IN', 'CYCLE', 'SCATTER-GATHER', 'GATHER-SCATTER',
               'BIPARTITE', 'STACK', 'RANDOM', 'OUT_OF_PATTERN']
PATTERN_CLASSES = tuple(range(8))
CLASS8 = 8


@dataclass
class Basis:
    """한 기준(10day / 18day)의 산출물 묶음."""
    name: str
    dir: Path
    meta: dict
    feat_names: list[str]

    @property
    def abs_time_cols(self) -> list[int]:
        return list(self.meta['features']['absolute_time_cols'])

    def cols(self, feature_set: str) -> np.ndarray:
        """피처 열 선택. `no_abs` 는 화두 10·17(절대 시각 미사용)에 맞춘 구성."""
        n = self.meta['features']['n']
        if feature_set == 'all81':
            return np.arange(n)
        if feature_set == 'no_abs':
            return np.setdiff1d(np.arange(n), self.abs_time_cols)
        raise ValueError(feature_set)

    def X(self, tag: str, mmap: bool = True) -> np.ndarray:
        return np.load(self.dir / 'stage1_9class' / f'X_{tag}.npy',
                       mmap_mode='r' if mmap else None)

    def y(self, tag: str) -> np.ndarray:
        return np.load(self.dir / 'stage1_9class' / f'y9_{tag}.npy')

    def is_pos(self, tag: str) -> np.ndarray:
        return np.load(self.dir / 'stage1_9class' / f'is_pos_{tag}.npy').astype(bool)

    def is_oop(self, tag: str) -> np.ndarray:
        # dtype 이 bool 이 아니면 아래 마스크 연산이 조용히 정수 인덱싱으로 바뀐다
        return np.load(self.dir / 'stage1_9class' / f'is_oop_{tag}.npy').astype(bool)

    def index_full(self):
        return np.load(self.dir / 'index_full.npz')

    def sample_idx(self, variant: str) -> np.ndarray | None:
        """train 내부 인덱스. `full` 이면 None(전량 사용)."""
        if variant == 'full':
            return None
        p = self.dir / 'undersample' / f'{variant}_tr.npy'
        if not p.exists():
            raise FileNotFoundError(p)
        return np.load(p)


def load_basis(name: str = '10day', dataset: str = 'HI-Small', root: Path | str = ROOT) -> Basis:
    """`processed_9class/{dataset}_{name}` 을 연다. 기본값은 기존 호출(HI-Small_10day)과 동일."""
    d = Path(root) / f'{dataset}_{name}'
    meta = json.loads((d / 'features_meta.json').read_text(encoding='utf-8'))
    return Basis(name, d, meta, list(meta['features']['names']))


def cold_start_mask(b: Basis, tag: str) -> np.ndarray:
    """양쪽 계좌 모두 판정 시점 이전 이력이 없는 행(화두 15의 cold-start).

    `dt_src_first`·`dt_dst_first` 는 '그 계좌의 첫 등장'을 뜻하는 피처다. 둘 다 1이면
    송신·수신 어느 쪽에도 과거가 없다.
    """
    ci = {nm: i for i, nm in enumerate(b.feat_names)}
    X = b.X(tag)
    out = np.zeros(X.shape[0], dtype=bool)
    step = 500_000
    for lo in range(0, X.shape[0], step):
        blk = np.asarray(X[lo:lo + step, [ci['dt_src_first'], ci['dt_dst_first']]])
        out[lo:lo + step] = (blk[:, 0] == 1) & (blk[:, 1] == 1)
    return out


def build_train(b: Basis, variant: str, feature_set: str,
                class_weight: str = 'balanced') -> dict:
    """학습 표본 + 표본가중 + '모델이 본 사전확률'을 함께 돌려준다.

    `prior_model` 은 '가중 합의 비율'이다. **주의**: `class_weight='balanced'` 를 쓰면 이 값이
    항상 정확히 1/9 이 되어 언더샘플 비율이 식에서 소거된다 — 두 장치가 합쳐지는 게 아니라
    가중이 언더샘플을 덮어쓴다. 게다가 트리 앙상블의 `sample_weight` 는 사전확률을
    재기준화하지 않고, `min_samples_leaf=1` 이면 잎이 대부분 순수해져 가중 효과가 거의 없다.
    따라서 이 보정은 '이론적으로 올바른 사전확률 되돌리기'가 아니라 **클래스별 상수 재척도**로만
    취급해야 한다. 실제 채점에는 쓰지 않고(§final), 참고 수치로만 남긴다.
    """
    cols = b.cols(feature_set)
    y_full = b.y('tr')
    idx = b.sample_idx(variant)
    Xtr_mm = b.X('tr')
    if idx is None:
        X = np.asarray(Xtr_mm[:, cols])
        y = y_full
    else:
        # 2026-09-02: 큰 세트에서 81열 전체를 모았다가 열을 고르면 피크가 두 배다. 청크로 바로 채운다.
        X = np.empty((len(idx), len(cols)), dtype=np.float32)
        for lo in range(0, len(idx), 2_000_000):
            hi = lo + 2_000_000
            X[lo:hi] = np.asarray(Xtr_mm[idx[lo:hi]])[:, cols]
        y = y_full[idx]

    cnt = np.bincount(y, minlength=9).astype(np.float64)
    if class_weight == 'balanced':
        w_cls = np.where(cnt > 0, len(y) / (9.0 * np.maximum(cnt, 1.0)), 0.0)
    elif class_weight == 'none':
        w_cls = np.ones(9)
    else:
        raise ValueError(class_weight)
    w = w_cls[y]

    # 모델이 본 사전확률 = 클래스별 가중 합의 비율
    wsum = np.array([w[y == c].sum() for c in range(9)], dtype=np.float64)
    prior_model = wsum / wsum.sum()
    # 참 사전확률은 **train split 전량**에서만 잰다(val/test 를 보면 누수다)
    cnt_full = np.bincount(y_full, minlength=9).astype(np.float64)
    prior_true = cnt_full / cnt_full.sum()

    return {'X': X, 'y': y, 'w': w, 'cols': cols,
            'prior_model': prior_model, 'prior_true': prior_true,
            'n_rows': int(len(y)), 'class_counts': cnt.astype(int).tolist(),
            'variant': variant, 'feature_set': feature_set,
            'class_weight': class_weight}


def predict_proba_chunked(model, b: Basis, tag: str, cols: np.ndarray,
                          step: int = 400_000) -> np.ndarray:
    """전량 val/test 예측. 피처 행렬을 통째로 RAM 에 올리지 않는다."""
    X = b.X(tag)
    n = X.shape[0]
    out = np.empty((n, 9), dtype=np.float32)
    for lo in range(0, n, step):
        blk = np.asarray(X[lo:lo + step])[:, cols]
        out[lo:lo + step] = model.predict_proba(blk).astype(np.float32)
    return out


def apply_prior_correction(proba: np.ndarray, prior_model: np.ndarray,
                           prior_true: np.ndarray) -> np.ndarray:
    """p_true(c|x) ∝ p_model(c|x) · π_true(c) / π_model(c).

    언더샘플링과 클래스 가중이 바꿔 놓은 사전확률을 참 사전확률로 되돌린다.
    이걸 안 하면 모델이 패턴 클래스를 과대예측해 즉시 알림이 폭증한다.
    """
    f = np.where(prior_model > 0, prior_true / np.maximum(prior_model, 1e-12), 0.0)
    p = proba * f[None, :]
    s = p.sum(axis=1, keepdims=True)
    return np.divide(p, np.maximum(s, 1e-12))


## 6. 모델 팩토리 — `train9/models.py`

모델 패밀리(ExtraTrees/LightGBM 등)를 이름으로 생성하는 팩토리.

원본 경로: `train9/models.py`

In [ ]:
"""모델 패밀리 팩토리.

표 형태 데이터라 트리 앙상블 세 가지를 후보로 둔다. 팀 기술 스택 후보(XGBoost)와
08-25 2차 모델에서 쓴 ExtraTrees 를 모두 포함해 비교 가능하게 한다.

소수 클래스가 train 에서 60건 수준이라 잎 최소 표본(min_child_samples / min_samples_leaf)을
기본값보다 낮춘다. 기본값(20)이면 그 클래스는 잎을 하나도 못 만든다.
"""
from __future__ import annotations

SEED = 42


def make(name: str, n_jobs: int = 4, seed: int = SEED, **over):
    if name == 'lgbm':
        from lightgbm import LGBMClassifier
        kw = dict(objective='multiclass', num_class=9, n_estimators=400,
                  learning_rate=0.05, num_leaves=63, min_child_samples=5,
                  subsample=0.9, subsample_freq=1, colsample_bytree=0.8,
                  reg_lambda=1.0, n_jobs=n_jobs, random_state=seed, verbose=-1)
        kw.update(over)
        return LGBMClassifier(**kw)
    if name == 'xgb':
        from xgboost import XGBClassifier
        kw = dict(objective='multi:softprob', num_class=9, n_estimators=400,
                  learning_rate=0.05, max_depth=6, min_child_weight=1.0,
                  subsample=0.9, colsample_bytree=0.8, reg_lambda=1.0,
                  tree_method='hist', n_jobs=n_jobs, random_state=seed,
                  eval_metric='mlogloss')
        kw.update(over)
        return XGBClassifier(**kw)
    if name == 'et':
        from sklearn.ensemble import ExtraTreesClassifier
        kw = dict(n_estimators=600, max_features='sqrt', min_samples_leaf=1,
                  n_jobs=n_jobs, random_state=seed)
        kw.update(over)
        return ExtraTreesClassifier(**kw)
    if name.startswith('hier_'):
        return HierarchicalNine(family=name.split('_', 1)[1], n_jobs=n_jobs, seed=seed, **over)
    if name == 'rf':
        from sklearn.ensemble import RandomForestClassifier
        kw = dict(n_estimators=600, max_features='sqrt', min_samples_leaf=1,
                  n_jobs=n_jobs, random_state=seed)
        kw.update(over)
        return RandomForestClassifier(**kw)
    raise ValueError(name)


def fit(model, X, y, w=None):
    """XGBoost 는 라벨이 0..n-1 연속이어야 한다. 9-Class 는 이미 0..8 연속이라 그대로 쓴다."""
    return model.fit(X, y, sample_weight=w)


class HierarchicalNine:
    """9-Class 를 두 단으로 쪼개 학습하되 **출력 계약은 그대로 9-Class** 인 구현.

    왜 쪼개나: 평탄한 9-Class 는 1,176건뿐인 패턴 행을 8개 클래스로 다시 쪼개 학습한다.
    가장 작은 클래스가 train 60건이라 어떤 모델도 안정적으로 못 배운다. 반면
      (a) '패턴인가 아닌가' 이진 머리는 1,176건을 **한 덩어리로** 쓰고,
      (b) '어느 유형인가' 8-way 머리는 패턴 행만 보므로 불균형이 사라진다.

    p(c|x) = p_bin(패턴|x) · p_type(c|x, 패턴)   (c = 0..7)
    p(8|x) = 1 − p_bin(패턴|x)

    회의가 정한 '1차 = 9-Class 다중분류' 계약을 깨지 않는다 — 입력도 전 거래, 출력도 9개 확률이다.
    """

    def __init__(self, family: str = 'et', n_jobs: int = 4, seed: int = SEED, **over):
        self.family, self.n_jobs, self.seed, self.over = family, n_jobs, seed, over
        self.bin_ = None
        self.type_ = None
        self.type_classes_ = None
        self.classes_ = list(range(9))

    def fit(self, X, y, sample_weight=None):
        import numpy as np
        y = np.asarray(y)
        w = None if sample_weight is None else np.asarray(sample_weight)

        # (a) 이진 머리 — 패턴(0~7) vs 클래스 8. 가중은 두 덩어리 기준으로 다시 잡는다.
        yb = (y <= 7).astype(np.int8)
        cb = np.bincount(yb, minlength=2).astype(float)
        wb = np.where(cb > 0, len(yb) / (2.0 * np.maximum(cb, 1.0)), 0.0)[yb]
        self.bin_ = make(self.family, n_jobs=self.n_jobs, seed=self.seed, **self.over)
        self.bin_.fit(X, yb, sample_weight=wb)

        # (b) 유형 머리 — 패턴 행만. 여기서는 8개 클래스만 놓고 균형 가중을 다시 잡는다.
        m = y <= 7
        yt = y[m]
        ct = np.bincount(yt, minlength=8).astype(float)
        wt = np.where(ct > 0, m.sum() / (8.0 * np.maximum(ct, 1.0)), 0.0)[yt]
        self.type_ = make(self.family, n_jobs=self.n_jobs, seed=self.seed, **self.over)
        self.type_.fit(X[m], yt, sample_weight=wt)
        self.type_classes_ = np.asarray(self.type_.classes_)
        return self

    def predict_proba(self, X):
        import numpy as np
        pb = self.bin_.predict_proba(X)
        i1 = int(np.where(np.asarray(self.bin_.classes_) == 1)[0][0])
        p_pat = pb[:, i1]
        pt = self.type_.predict_proba(X)
        out = np.zeros((X.shape[0], 9), dtype=np.float64)
        for j, c in enumerate(self.type_classes_):
            out[:, int(c)] = p_pat * pt[:, j]
        out[:, 8] = 1.0 - p_pat
        return out

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)


## 7. 평가지표 — `train9/metrics.py`

1차 9-Class 모델 평가. Accuracy를 주지표로 쓰지 않음(99.95%가 클래스 8이라 전부 8이라 답해도 99.9%). 재현율 0.70 고정 운영점 등 파이프라인 의미에 맞춘 지표만 사용.

원본 경로: `train9/metrics.py`

In [ ]:
"""1차 9-Class 모델 평가 — 파이프라인 의미에 맞춘 지표만 만든다.

정확도를 주지표로 쓰지 않는다. 전체의 99.95%가 클래스 8이라 **전부 8이라고 답해도 99.9%** 다.
대신 파이프라인이 실제로 하는 두 가지 일을 따로 잰다.

  1. **즉시 알림** — 0~7로 예측한 거래는 그 자리에서 알림이 된다. 정밀도가 중요하다.
  2. **2차 라우팅** — 8로 예측한 거래만 2차 이진으로 넘어간다. 여기서 놓치면 영영 못 본다.

세 종류의 정밀도를 구분해서 본다. 셋을 뭉뚱그리면 "유형은 틀렸지만 세탁은 맞은" 알림이
성공인지 실패인지 말할 수 없다.
  alert_precision_type       예측 유형까지 정확히 맞은 비율
  alert_precision_any        8종 패턴 거래이긴 한 비율 (유형은 틀려도 됨)
  alert_precision_laundering 세탁 거래이긴 한 비율 (패턴 외 세탁도 성공으로 침)
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix

PATTERN = tuple(range(8))
CLASS8 = 8
CLASS_NAMES = ['FAN-OUT', 'FAN-IN', 'CYCLE', 'SCATTER-GATHER', 'GATHER-SCATTER',
               'BIPARTITE', 'STACK', 'RANDOM', 'OUT_OF_PATTERN']


def _prf(y_true: np.ndarray, y_pred: np.ndarray, c: int) -> tuple[float, float, float, int]:
    tp = int(((y_pred == c) & (y_true == c)).sum())
    fp = int(((y_pred == c) & (y_true != c)).sum())
    fn = int(((y_pred != c) & (y_true == c)).sum())
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f, tp + fn


def per_class_table(y_true: np.ndarray, y_pred: np.ndarray,
                    proba: np.ndarray | None = None) -> pd.DataFrame:
    rows = []
    for c in range(9):
        p, r, f, n = _prf(y_true, y_pred, c)
        row = {'class': c, 'name': CLASS_NAMES[c], 'support': n,
               'precision': p, 'recall': r, 'f1': f,
               'n_pred': int((y_pred == c).sum())}
        if proba is not None and 0 < n < len(y_true):
            row['pr_auc'] = float(average_precision_score((y_true == c).astype(int),
                                                          proba[:, c]))
        rows.append(row)
    return pd.DataFrame(rows)


def evaluate(y_true: np.ndarray, proba: np.ndarray, is_pos: np.ndarray,
             is_oop: np.ndarray, tau: float | None = None,
             cold: np.ndarray | None = None) -> dict:
    """argmax9(tau=None) 또는 임계 규칙(tau)으로 판정하고 지표를 낸다."""
    pat_score = proba[:, :8].max(axis=1)
    pat_arg = proba[:, :8].argmax(axis=1)
    if tau is None:
        y_pred = proba.argmax(axis=1)
    else:
        y_pred = np.where(pat_score >= tau, pat_arg, CLASS8)

    alert = y_pred <= 7
    true_pat = y_true <= 7
    n_alert = int(alert.sum())

    per = per_class_table(y_true, y_pred, proba)
    # macro 는 **평가셋에 실제로 등장한 8종 클래스**만 평균한다. support 0 인 클래스를
    # recall 0 으로 세면 분모가 표마다 달라져 서로 비교할 수 없게 된다.
    pat_rows = per[(per['class'] <= 7) & (per['support'] > 0)]

    out = {
        'tau': tau,
        'n_rows': int(len(y_true)),
        'macro_recall_8': float(pat_rows['recall'].mean()),
        'macro_precision_8': float(pat_rows['precision'].mean()),
        'macro_f1_8': float(pat_rows['f1'].mean()),
        'macro_pr_auc_8': float(pat_rows['pr_auc'].mean()) if 'pr_auc' in pat_rows else np.nan,
        'n_classes_in_macro': int(len(pat_rows)),
        # 즉시 알림
        'n_alerts': n_alert,
        'alert_rate': n_alert / max(len(y_true), 1),
        'alert_precision_type': float((y_pred[alert] == y_true[alert]).mean()) if n_alert else 0.0,
        'alert_precision_any': float(true_pat[alert].mean()) if n_alert else 0.0,
        'alert_precision_laundering': float(is_pos[alert].mean()) if n_alert else 0.0,
        'alert_recall_pattern': float(alert[true_pat].mean()) if true_pat.any() else 0.0,
        # 2차 라우팅
        'routing_recall_class8': float((y_pred[y_true == CLASS8] == CLASS8).mean()),
        'oop_routed_to_stage2': float((y_pred[is_oop] == CLASS8).mean()) if is_oop.any() else np.nan,
        'oop_alerted_as_pattern': float(alert[is_oop].mean()) if is_oop.any() else np.nan,
        'normal_false_alert_rate': float(alert[(y_true == CLASS8) & (~is_pos)].mean()),
        # 세탁 전체 기준 1차 커버리지
        'laundering_alerted': float(alert[is_pos].mean()) if is_pos.any() else np.nan,
    }
    if cold is not None and cold.any():
        cm = cold & true_pat
        out['cold_start_rows'] = int(cold.sum())
        out['cold_start_pattern_rows'] = int(cm.sum())
        def _macro8(yt_, yp_):
            t = per_class_table(yt_, yp_)
            t = t[(t['class'] <= 7) & (t['support'] > 0)]
            return float(t['recall'].mean()) if len(t) else np.nan
        out['cold_macro_recall_8'] = _macro8(y_true[cold], y_pred[cold]) if cm.any() else np.nan
        warm = ~cold
        out['warm_macro_recall_8'] = _macro8(y_true[warm], y_pred[warm])
    return out


def alert_budget_sweep(y_true: np.ndarray, proba: np.ndarray, is_pos: np.ndarray,
                       ks=None, min_k: int = 20, max_k: int = 200_000) -> pd.DataFrame:
    """알림 예산 K 별 정밀도-재현율 곡선.

    임계값을 **분위수로 자르지 않는다.** 양성이 100만 행 중 700건 수준이라 분위수 격자는
    고정밀 구간(상위 수백 건)을 통째로 건너뛴다 — 팀 목표인 정밀도 90%가 바로 그 구간에 있다.
    대신 점수 상위 K건을 알림으로 삼아 K를 로그 간격으로 훑는다. K는 그대로
    '관제팀이 하루에 볼 수 있는 알림 수'라 운영 대화에도 바로 쓰인다.

    `min_k` 아래는 만들지 않는다. 표본 1건에서 나온 '정밀도 100%'를 운영점으로 고르는 것을
    막기 위한 최소 지지 조건이다.
    """
    score = proba[:, :8].max(axis=1)
    arg = proba[:, :8].argmax(axis=1)
    order = np.argsort(-score, kind='stable')
    true_pat = y_true <= 7
    n_pat = int(true_pat.sum())
    yo, ao, po = y_true[order], arg[order], is_pos[order]

    # 누적합은 int32 로 충분하다(행 수가 int32 상한 21억보다 훨씬 작다). n=35.9M 인 HI-Large 에서
    # int64 로 잡으면 이 네 덩어리만 3.5 GiB 다.
    c_any = np.cumsum(true_pat[order], dtype=np.int32)
    c_type = np.cumsum(ao == yo, dtype=np.int32)
    c_laund = np.cumsum(po, dtype=np.int32)
    # 클래스별 누적 적중 — K 마다 macro recall 을 정확히 낸다.
    # 리스트에 모았다가 np.stack 하면 8벌이 두 번 살아 있다(n=35.9M 이면 4.3 GiB 낭비) → 미리 잡고 제자리로.
    c_cls = np.empty((8, len(yo)), dtype=np.int32)
    for c in range(8):
        np.cumsum((ao == c) & (yo == c), dtype=np.int32, out=c_cls[c])
    sup = np.array([int((y_true == c).sum()) for c in range(8)])

    if ks is None:
        hi = int(min(max_k, len(y_true)))
        ks = np.unique(np.round(np.logspace(np.log10(min_k), np.log10(hi), 45)).astype(int))
    rows = []
    for k in ks:
        k = int(min(k, len(y_true)))
        if k < min_k:
            continue
        i = k - 1
        rec = np.where(sup > 0, c_cls[:, i] / np.maximum(sup, 1), np.nan)
        rows.append({'k_alerts': k, 'tau': float(score[order[i]]),
                     'precision_type': float(c_type[i] / k),
                     'precision_any': float(c_any[i] / k),
                     'precision_laundering': float(c_laund[i] / k),
                     'recall_pattern': float(c_any[i] / max(n_pat, 1)),
                     'macro_recall_8': float(np.nanmean(rec)),
                     'n_classes_scored': int((sup > 0).sum())})
    return pd.DataFrame(rows)


def operating_point_at_precision(sw: pd.DataFrame, target: float = 0.90,
                                 field: str = 'precision_any') -> dict | None:
    """목표 정밀도를 만족하는 운영점 중 **재현율이 가장 큰** 것.

    `alert_budget_sweep` 이 이미 min_k 로 최소 지지를 걸어 뒀으므로 여기서 다시 거르지 않는다.
    """
    ok = sw[sw[field] >= target]
    if not len(ok):
        return None
    r = ok.loc[ok['recall_pattern'].idxmax()]
    return {'k_alerts': int(r['k_alerts']), 'tau': float(r['tau']),
            'precision_any': float(r['precision_any']),
            'precision_type': float(r['precision_type']),
            'precision_laundering': float(r['precision_laundering']),
            'recall_pattern': float(r['recall_pattern']),
            'macro_recall_8': float(r['macro_recall_8'])}


def operating_point_at_recall(sw: pd.DataFrame, target_recall: float = 0.70) -> dict | None:
    """**재현율을 고정해 놓고** 그 지점의 정밀도를 본다 — 모델 비교 운영점.

    "recall 0.7 이상"이 넘어야 할 기준선이 아니라, 재현율 0.70 지점에서 어느 모델의
    정밀도가 더 높은지로 비교한다는 뜻이다. 목표를 만족하는 가장 작은 알림 예산을 고른다
    (같은 재현율이면 알림이 적을수록 좋다).

    출처(2026-09-09 정리) — **이 규칙의 정본 정의 지점이다. 다른 파일은 여기를 가리킨다.**
      · 규칙 자체: 손은총 확인(2026-09-09) — "재현율을 0.7로 잡았을 때 정밀도가 제일 높은
        모델을 쓴다". 유효한 규칙이다.
      · 다만 **팀 정본 문서·회의록 11개·Jira 27건에는 이 값이 없다**(09-09 전수 grep).
        이전 코드가 "팀 비교 관례"라고 인용 없이 단언했고 그 표현이 20개 파일로 승계됐다.
        그 표기는 09-09에 걷어냈다. 팀 문서화는 미완 — 안건 A5(평가지표·기준값 공식 정의).
      · 혼동 주의: 이것은 **재현율** 0.70이다. 블록 유형분류 "목표 KPI macro-F1 ≥ 0.70"은
        전혀 다른 값이고 그쪽은 근거 없는 임의값이라 09-09에 폐기했다.
    """
    ok = sw[sw['recall_pattern'] >= target_recall]
    if not len(ok):
        return None
    r = ok.loc[ok['k_alerts'].idxmin()]
    return {'target_recall': target_recall, 'k_alerts': int(r['k_alerts']),
            'tau': float(r['tau']), 'recall_pattern': float(r['recall_pattern']),
            'precision_any': float(r['precision_any']),
            'precision_type': float(r['precision_type']),
            'precision_laundering': float(r['precision_laundering']),
            'macro_recall_8': float(r['macro_recall_8'])}


def confusion(y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    cm = confusion_matrix(y_true, y_pred, labels=list(range(9)))
    return pd.DataFrame(cm, index=[f'true {n}' for n in CLASS_NAMES],
                        columns=[f'pred {n}' for n in CLASS_NAMES])


def baselines(y_true: np.ndarray, is_pos: np.ndarray, seed: int = 42,
              prior: np.ndarray | None = None) -> pd.DataFrame:
    """비교 기준선. '전부 8' 이 정확도 99.9%라는 사실을 표로 박아둔다.

    층화 무작위의 사전확률은 **train 에서** 받아야 한다(`prior`). 평가셋 자신의 라벨 분포를
    쓰면 정답을 미리 본 기준선이 된다.
    """
    rng = np.random.default_rng(seed)
    rows = []
    for nm, pred in (
            ('전부 클래스 8', np.full(len(y_true), CLASS8)),
            ('층화 무작위(train 사전확률)', rng.choice(
                9, size=len(y_true),
                p=(prior if prior is not None
                   else np.bincount(y_true, minlength=9) / len(y_true))))):
        per = per_class_table(y_true, pred)
        alert = pred <= 7
        rows.append({
            '기준선': nm, '정확도': float((pred == y_true).mean()),
            'macro_recall_8': float(per[per['class'] <= 7]['recall'].mean()),
            'macro_precision_8': float(per[per['class'] <= 7]['precision'].mean()),
            'n_alerts': int(alert.sum()),
            'alert_precision_any': float((y_true[alert] <= 7).mean()) if alert.any() else 0.0,
        })
    return pd.DataFrame(rows)


## 8. 실험 러너 — `train9/run.py`

실험 러너 — 한 설정(모델×피처셋×표본 변형) = 한 줄로 돌리는 진입점.

원본 경로: `train9/run.py`

In [ ]:
"""실험 러너 — 한 설정 = 한 줄.

**선택은 val 로만 한다.** 이 모듈은 test 를 아예 열지 않는다. test 평가는
`final_9class.py` 가 최종 설정 하나에 대해 딱 한 번 수행한다.

주 선택지표는 `macro_pr_auc_8`(8종 one-vs-rest 평균정밀도의 macro 평균)이다.
임계값이 필요 없고 극단 불균형에서도 의미가 있기 때문이다. argmax 기준 지표는
'운영점 하나에서의 모습'으로 함께 적되 선택 근거로 쓰지 않는다.
"""
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from . import data as D, metrics as Me, models as M

# 벤치마크 결과: LightGBM 은 이 크기에서 스레드가 많을수록 느려진다(j4 14s / j32 200s).
N_JOBS = {'lgbm': 4, 'xgb': 4, 'et': 4, 'rf': 4,
          'hier_et': 4, 'hier_xgb': 4, 'hier_lgbm': 4}
# 2026-09-02: cgroup 27 GiB 상한에서 트리당 작업 배열(행수×~12B)이 스레드 수만큼 겹친다.
# 2026-09-09: 전 패밀리를 4 로 통일했다. 이 컨테이너의 cgroup CPU 상한이 4코어이고,
#   위 벤치(j4 13.7s / j32 200.3s)가 이미 초과 설정이 더 느리다는 걸 측정해 뒀는데도
#   lgbm 외에는 32 가 남아 있었다. 4 는 새로 고른 값이 아니라 이미 측정된 값이다.
#   더 줄이려면 TRAIN9_NJOBS 환경변수를 쓴다(무거운 단계는 AML_THREADS=3 선례).
import os as _os
if _os.environ.get('TRAIN9_NJOBS'):
    N_JOBS = {k: (int(_os.environ['TRAIN9_NJOBS']) if v > 4 else v) for k, v in N_JOBS.items()}


def run_one(b: D.Basis, variant: str, feature_set: str, family: str,
            class_weight: str = 'balanced', model_kw: dict | None = None,
            cold: np.ndarray | None = None, keep_proba: bool = False) -> dict:
    t0 = time.time()
    tr = D.build_train(b, variant, feature_set, class_weight)
    t_load = time.time() - t0

    t0 = time.time()
    m = M.make(family, n_jobs=N_JOBS[family], **(model_kw or {}))
    M.fit(m, tr['X'], tr['y'], tr['w'])
    t_fit = time.time() - t0

    t0 = time.time()
    pv = D.predict_proba_chunked(m, b, 'va', tr['cols'])
    t_pred = time.time() - t0

    yv, ipv, oopv = b.y('va'), b.is_pos('va'), b.is_oop('va')
    pvc = D.apply_prior_correction(pv, tr['prior_model'], tr['prior_true'])

    row = {'basis': b.name, 'variant': variant, 'feature_set': feature_set,
           'family': family, 'class_weight': class_weight,
           # 선택된 설정을 나중에 그대로 재현하려면 하이퍼파라미터가 행에 남아야 한다
           'model_kw': json.dumps(model_kw or {}, sort_keys=True),
           'n_train': tr['n_rows'], 'n_feat': len(tr['cols']),
           't_load_s': round(t_load, 1), 't_fit_s': round(t_fit, 1),
           't_pred_s': round(t_pred, 1)}
    for tag, p in (('raw', pv), ('prior', pvc)):
        r = Me.evaluate(yv, p, ipv, oopv, cold=cold)
        for k in ('macro_recall_8', 'macro_precision_8', 'macro_f1_8', 'macro_pr_auc_8',
                  'n_alerts', 'alert_precision_type', 'alert_precision_any',
                  'alert_precision_laundering', 'alert_recall_pattern',
                  'routing_recall_class8', 'oop_routed_to_stage2',
                  'normal_false_alert_rate', 'laundering_alerted'):
            row[f'{tag}.{k}'] = r[k]
        if cold is not None:
            row[f'{tag}.cold_macro_recall_8'] = r.get('cold_macro_recall_8', np.nan)
            row[f'{tag}.warm_macro_recall_8'] = r.get('warm_macro_recall_8', np.nan)
    # 팀 정량 목표(정밀도 90%)에 직접 대응하는 운영점. 분위수가 아니라 알림 예산 K 로 훑는다.
    sw = Me.alert_budget_sweep(yv, pv, ipv)
    op = Me.operating_point_at_precision(sw, 0.90)
    row['k@P90'] = op['k_alerts'] if op else np.nan
    row['recall@P90'] = op['recall_pattern'] if op else 0.0
    row['macroR8@P90'] = op['macro_recall_8'] if op else 0.0
    # 목표에 못 미쳐도 곡선의 모양은 남긴다 — 어디까지 갈 수 있는지가 보고서의 핵심이다
    row['best_precision_any'] = float(sw['precision_any'].max())
    for k in (100, 500, 2000):
        r_ = sw.iloc[(sw['k_alerts'] - k).abs().argmin()]
        row[f'P@K{k}'] = float(r_['precision_any'])
        row[f'R@K{k}'] = float(r_['recall_pattern'])

    art = {'model': m, 'tr': tr, 'proba_val': pv if keep_proba else None, 'sweep': sw}
    return row, art


def sweep(b: D.Basis, configs: list[dict], out_dir: Path, tag: str,
          cold: np.ndarray | None = None) -> pd.DataFrame:
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for i, cfg in enumerate(configs, 1):
        try:
            row, _ = run_one(b, cold=cold, **cfg)
        except Exception as e:                       # 한 설정이 죽어도 스윕은 계속한다
            row = {**cfg, 'error': f'{type(e).__name__}: {e}'}
            print(f'  [{i}/{len(configs)}] FAIL {cfg} -> {e}', flush=True)
        else:
            print(f"  [{i}/{len(configs)}] {cfg['family']:5s} {cfg['variant']:14s} "
                  f"{cfg['feature_set']:7s} | PR-AUC {row['raw.macro_pr_auc_8']:.4f} "
                  f"| macroR8 {row['raw.macro_recall_8']:.4f} "
                  f"| P@500 {row['P@K500']:.3f} R@500 {row['R@K500']:.3f} "
                  f"| R@P90 {row['recall@P90']:.3f} | {row['t_fit_s']:.0f}s", flush=True)
        rows.append(row)
        pd.DataFrame(rows).to_csv(out_dir / f'{tag}.csv', index=False)
    return pd.DataFrame(rows)


## 9. 현재 채택 학습 — `train9/lgb_ooc.py`

**현재 채택 라인.** LightGBM out-of-core 전량 학습(2026-09-03~) — 27GiB RAM 안에서 train 1억 행 전량을 쓰기 위해 memmap을 `lgb.Sequence`로 청크 단위만 읽는다. 사전확률 보정은 `train9.data.apply_prior_correction` 사용.

원본 경로: `train9/lgb_ooc.py`

In [ ]:
"""LightGBM 전량 학습(out-of-core) — 컨테이너 RAM 27 GiB 안에서 train 1억 행을 다 쓰기 위한 경로. 2026-09-03.

왜: HI-Large 는 train 전량(1억 행 × 76 float32 = 33 GB)이 RAM 에 안 들어가 random_r100 으로 학습했다.
    LightGBM 은 피처를 1바이트 bin 으로 저장하므로(1억 × 76 = 8 GB) 전량이 들어간다. 원 배열은 memmap 에서
    `lgb.Sequence` 로 청크 단위로만 읽는다(LightGBM ≥ 3.3).
구조: HierarchicalNine(train9/models.py) 과 같은 두 머리 — 이진 머리(패턴 vs 아님, 전량·Sequence) + 유형 머리(8-way, 패턴 행만·RAM).
      출력 계약도 같다(9열 확률). 사전확률 보정은 기존과 같이 train9.data.apply_prior_correction 을 쓴다.

주의: 이 모듈은 코드만 검증(HI-Small 스모크)했다. HI-Large 실행은 별도 결정 후.
"""
from __future__ import annotations

import json
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np


class MemmapSequence(lgb.Sequence):
    """X memmap 의 (행 부분집합, 열 부분집합) 을 청크로 내놓는다. 행 인덱스가 None 이면 전량.

    dtype 이 두 갈래인 이유(2026-09-09 실물 확인). 예전 주석은 "lgb.Sequence 는 double 만 받는다"
    였는데 **절반만 맞다**. LightGBM 4.7.0 은 경로마다 요구가 다르다:
      · 표본 추출 — `Dataset.__sample` 이 **int 인덱스**로 행을 모으고(`basic.py:2221`),
        `_init_from_sample` 이 `dtype != np.double` 이면 거부한다(`basic.py:1926`). → float64 필수.
      · 대량 적재 — `_push_rows` 가 **slice** 를 받아 `_c_float_array` 로 넘기는데
        이쪽은 float32·float64 를 모두 허용한다(`basic.py:684-691`). → float32 로 충분.
    표본은 20만 행 수준이고 대량은 전량이므로, 큰 쪽만 float32 로 내리면 청크 버퍼가 절반이 된다
    (50만 행 × 76열: 304 MB → 152 MB). 학습 결과는 비트 단위로 동일함을 확인했다.
    """

    def __init__(self, X_mm: np.ndarray, cols: np.ndarray, rows: np.ndarray | None = None, batch_size: int = 500_000):
        self.X, self.cols, self.rows = X_mm, np.asarray(cols), rows
        self.batch_size = batch_size
        self.n = int(X_mm.shape[0]) if rows is None else int(len(rows))

    def __getitem__(self, idx):
        if isinstance(idx, slice):
            start, stop, step = idx.indices(self.n)
            r = np.arange(start, stop, step)
            if self.rows is not None:
                r = self.rows[r]
            return np.asarray(self.X[r])[:, self.cols].astype(np.float32)   # 대량 적재 경로 — float32 허용(basic.py:684)
        i = int(idx)
        r = i if self.rows is None else int(self.rows[i])
        return np.asarray(self.X[r])[self.cols].astype(np.float64)  # 표본 추출 경로 — double 필수(basic.py:1926)

    def __len__(self):
        return self.n


class HierLGBMOOC:
    """이진 머리(전량, out-of-core) + 유형 머리(패턴 행만). predict_proba 는 HierarchicalNine 과 같은 9열."""

    def __init__(self, n_estimators: int = 400, learning_rate: float = 0.05, num_leaves: int = 63,
                 min_child_samples: int = 5, n_jobs: int = 4, seed: int = 42, batch_size: int = 500_000,
                 class_weight: str = 'balanced'):
        """class_weight — **이진 머리**의 표본 가중. 2026-09-10 추가(가중치 on/off × 비율 실험, 인계 0910 §8-2).
          'balanced' : 현행 그대로. wb = len/(2·클래스수) → 모델이 보는 사전확률 (패턴, 비패턴) = (0.5, 0.5)
          'none'     : weight=None. 모델이 보는 사전확률 = 표본의 실제 비율
        유형 머리(8-way)는 이 옵션과 무관하게 현행(balanced)을 유지한다 — 언더샘플 비율은 유형 머리 입력을
        바꾸지 않으므로(패턴 행 전량 유지), 한 번에 한 변수만 바꾸기 위해서다.
        """
        if class_weight not in ('balanced', 'none'):
            raise ValueError(f'class_weight must be balanced|none, got {class_weight!r}')
        self.kw = dict(n_estimators=n_estimators, learning_rate=learning_rate, num_leaves=num_leaves,
                       min_child_samples=min_child_samples, n_jobs=n_jobs, seed=seed)
        self.class_weight = class_weight
        self.bin_prior_model_ = None   # 이진 머리가 실제로 본 패턴 사전확률(가중 반영). fit 후 채워진다
        self.batch_size = batch_size
        self.bin_ = None; self.type_ = None; self.type_classes_ = None; self.classes_ = list(range(9))
        self.timing = {}

    def fit(self, X_mm: np.ndarray, y: np.ndarray, cols: np.ndarray, rows: np.ndarray | None = None,
            valid: tuple | None = None):
        t0 = time.time()
        y = np.asarray(y)
        yb = (y <= 7).astype(np.int8)
        cb = np.bincount(yb, minlength=2).astype(float)
        if self.class_weight == 'balanced':
            wb = np.where(cb > 0, len(yb) / (2.0 * np.maximum(cb, 1.0)), 0.0)[yb]
            self.bin_prior_model_ = 0.5          # 가중 합이 클래스마다 len/2 → 정확히 0.5
        else:
            wb = None                            # 'none' — 모델이 보는 사전확률 = 표본 비율
            self.bin_prior_model_ = float(cb[1] / len(yb))
        seq = MemmapSequence(X_mm, cols, rows, self.batch_size)
        ds = lgb.Dataset(seq, label=yb, weight=wb, free_raw_data=True,
                         params={'max_bin': 255, 'verbose': -1, 'num_threads': self.kw['n_jobs']})
        params = dict(objective='binary', learning_rate=self.kw['learning_rate'], num_leaves=self.kw['num_leaves'],
                      min_child_samples=self.kw['min_child_samples'], subsample=0.9, subsample_freq=1,
                      colsample_bytree=0.8, reg_lambda=1.0, num_threads=self.kw['n_jobs'], seed=self.kw['seed'],
                      verbose=-1)
        valid_sets, callbacks = [], []
        if valid is not None:
            Xv, yv = valid
            vds = lgb.Dataset(Xv, label=(np.asarray(yv) <= 7).astype(np.int8), reference=ds)
            valid_sets = [vds]; callbacks = [lgb.early_stopping(50, verbose=False)]
        self.bin_ = lgb.train(params, ds, num_boost_round=self.kw['n_estimators'], valid_sets=valid_sets,
                              callbacks=callbacks)
        self.timing['bin_fit_s'] = round(time.time() - t0, 1)
        # 유형 머리 — 패턴 행만 RAM 에 올린다(HI-Large 약 6만 행)
        t1 = time.time()
        mask = y <= 7
        pat_rows = np.flatnonzero(mask) if rows is None else rows[mask]
        order = np.argsort(pat_rows)
        Xp = np.asarray(X_mm[pat_rows[order]])[:, cols]
        yt = y[mask][order]
        ct = np.bincount(yt, minlength=8).astype(float)
        wt = np.where(ct > 0, len(yt) / (8.0 * np.maximum(ct, 1.0)), 0.0)[yt]
        self.type_ = lgb.LGBMClassifier(objective='multiclass', num_class=8, n_estimators=self.kw['n_estimators'],
                                        learning_rate=self.kw['learning_rate'], num_leaves=self.kw['num_leaves'],
                                        min_child_samples=self.kw['min_child_samples'], subsample=0.9, subsample_freq=1,
                                        colsample_bytree=0.8, reg_lambda=1.0, n_jobs=self.kw['n_jobs'],
                                        random_state=self.kw['seed'], verbose=-1)
        self.type_.fit(Xp, yt, sample_weight=wt)
        self.type_classes_ = np.asarray(self.type_.classes_)
        self.timing['type_fit_s'] = round(time.time() - t1, 1)
        return self

    def predict_proba(self, X):
        """**청크 한 덩어리**를 받는 계약이다. test 전량을 통째로 넘기지 말 것.

        2026-09-09 실측(`model_ooc_probe/predict_*_n35M.json`, HI-Large test 35,928,091행):
          통째로 넘기면 최대 **17.56 GB**(anon 16.84) — mem_guard WARN(anon 15.04 GiB)을 실제로
          넘겨 경고가 찍혔고 STOP(19.14 GiB)까지 2.30 GB 밖에 안 남았다. 게다가 227.0초로
          청크(2M, 214.4초)보다 **느리기까지** 하다.
        전량 채점은 `train9.data.predict_proba_chunked(model, b, tag, cols)` 를 쓴다 —
        step=400,000 의 집안 표준 경로이고(final_9class·stageC_9class·imbalance_search·
        undersample_grid·seedvar_9class·boundary_and_purging 이 이미 사용), 실측 4.50 GB 다.
        네 방식의 채점 결과가 동일함은 같은 실측에서 검산했다(p8 합·패턴 판정 행 일치).
        """
        p_pat = self.bin_.predict(X)
        pt = self.type_.predict_proba(X)
        out = np.zeros((X.shape[0], 9), dtype=np.float64)
        for j, c in enumerate(self.type_classes_):
            out[:, int(c)] = p_pat * pt[:, j]
        out[:, 8] = 1.0 - p_pat
        return out

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)


def smoke(dataset: str = 'HI-Small', basis: str = '10day', n_estimators: int = 200, n_jobs: int = 4) -> dict:
    """HI-Small 로 전량 out-of-core 학습이 도는지·val PR-AUC 가 어느 수준인지 확인한다."""
    import sys
    sys.path.insert(0, '/workspace')
    from train9 import data as D
    from sklearn.metrics import average_precision_score
    b = D.load_basis(basis, dataset=dataset)
    cols = b.cols('no_abs')
    X_tr = b.X('tr'); y_tr = b.y('tr')
    X_va = np.asarray(b.X('va'))[:, cols]; y_va = b.y('va')
    m = HierLGBMOOC(n_estimators=n_estimators, n_jobs=n_jobs)
    t0 = time.time()
    m.fit(X_tr, y_tr, cols, rows=None, valid=(X_va, y_va))
    pv = m.predict_proba(X_va)
    macro_ap = float(np.mean([average_precision_score((y_va == c).astype(int), pv[:, c]) for c in range(8)
                              if (y_va == c).any()]))
    pat_ap = float(average_precision_score((y_va <= 7).astype(int), pv[:, :8].sum(1)))
    out = dict(dataset=dataset, n_train=int(len(y_tr)), cols=int(len(cols)), elapsed_s=round(time.time() - t0, 1),
               timing=m.timing, val_macro_pr_auc_8=macro_ap, val_pattern_vs_rest_ap=pat_ap,
               best_iteration=int(m.bin_.best_iteration or n_estimators))
    print(json.dumps(out, ensure_ascii=False, indent=1))
    return out


if __name__ == '__main__':
    smoke()


## 10. 학습 실행 스크립트 — `ooc_full_train.py`

HI-Large 전량 학습 실행 러너. `--variant full`(언더샘플 없음) vs `random_r100`(현행 대조군) 등. 2026-09-09 실측: 전량 통짜(17.56GB)보다 out-of-core(4.50GB)가 안전하고 결과 동일함을 확인.

원본 경로: `ooc_full_train.py`

In [ ]:
#!/usr/bin/env python3
"""HI-Large 전량 학습 실행 — 인계 §8 문제 1(언더샘플링)의 본 실행.

무엇을 하나: `train9/lgb_ooc.py:HierLGBMOOC` 로 train 을 학습하고 val·test 확률을 저장한다.
  --variant full          → train 전량 107,783,265행 (언더샘플 없음)
  --variant random_r100   → 현행 5,988,103행(전량의 5.56%) — **같은 모델·같은 채점의 대조군**

왜 대조군이 필요한가: "전량이 더 낫다"를 말하려면 바뀐 변수가 학습 행 수 **하나**여야 한다.
현재 채택 모델은 ExtraTrees(`model_9class_large/selected_config.json`)라 패밀리가 달라 직접 비교가 안 된다.
그래서 두 팔 모두 LightGBM-OOC 로 돌린다. 이것은 인계 §6 분류의 ①(성적으로 골라도 되는 것 —
다운샘플링 기법·비율)이다. 시험지(test)는 두 팔이 완전히 동일하다.

채점 경로: `train9.data.predict_proba_chunked`(step=400,000, 집안 표준).
  2026-09-09 실측으로 전량 통짜(17.56 GB)보다 4.50 GB 로 안전하고 결과가 동일함을 확인했다
  (`model_ooc_probe/predict_*_n35M.json`).

확률은 **보정 전(raw)과 사전확률 보정 후를 모두** 저장한다. 어느 쪽으로 채점할지는 이 스크립트가
정하지 않는다 — 알림 점수식 교정(인계 안건 11)이 아직 팀 미결이기 때문이다.

사전확률 보정의 prior_model 은 지어내지 않고 두 머리의 가중에서 **유도**한다:
  이진 머리 `wb = len(y)/(2·클래스수)` → 모델이 본 (패턴, 비패턴) = (0.5, 0.5)
  유형 머리 `wt = len(y)/(8·클래스수)` → 8종 각각 1/8
  ⇒ prior_model[c<8] = 0.5 × 1/8 = 1/16 · prior_model[8] = 0.5   (lgb_ooc.py wb/wt 대조)
prior_true 는 **train split 전량**에서만 센다(val/test 를 보면 누수).

2026-09-10 추가 — 가중치 on/off × 비율 × 시드 실험(인계 0910 §8-2):
  --class-weight balanced (기본, 현행) | none (이진 머리 weight=None)
  --seed 42 (기본, 현행) — 시드 관례는 seedvar_9class.py SEEDS=(42, 7, 123, ...) 를 따른다
  class_weight=none 이면 이진 머리가 본 패턴 사전확률은 0.5 가 아니라 **표본의 실제 비율**이므로
  prior_model 을 `HierLGBMOOC.bin_prior_model_` 에서 읽는다(balanced 면 정확히 0.5 → 기존과 동일).
"""
import argparse, os, sys

ap = argparse.ArgumentParser()
ap.add_argument('--variant', default='full', help='full | random_r10 | random_r30 | random_r100 | random_r300')
ap.add_argument('--rows', type=int, default=0, help='>0 이면 variant 를 무시하고 train 앞에서부터 N행(탐침용)')
ap.add_argument('--rounds', type=int, default=400, help='models.py lgbm 과 같은 값(400)')
ap.add_argument('--jobs', type=int, default=4)
ap.add_argument('--batch', type=int, default=500_000, help='lgb.Sequence 청크 행 수')
ap.add_argument('--score-step', type=int, default=400_000, help='채점 청크 — train9/data.py:148 집안 표준')
ap.add_argument('--no-score', action='store_true', help='학습만 하고 채점은 건너뛴다')
ap.add_argument('--splits', default='va',
                help="채점할 split. 기본 'va' — 운영점은 val 로만 정하고 test 는 최종 설정 확정 후 1회만 연다"
                     "(final_9class.py:2 'test 를 여기서 딱 한 번 연다'). 'va,te' 로 둘 다 가능.")
ap.add_argument('--tag', default=None)
ap.add_argument('--class-weight', default='balanced', choices=['balanced', 'none'],
                help='이진 머리 표본 가중. balanced=현행(모델이 50:50 을 봄) / none=weight 없음')
ap.add_argument('--seed', type=int, default=42, help='모델 seed. 기본 42=현행. 관례: seedvar_9class.py SEEDS')
a = ap.parse_args()

for v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[v] = str(a.jobs)

import json, threading, time, gc
from pathlib import Path
import numpy as np
sys.path.insert(0, '/workspace')

CG = Path('/sys/fs/cgroup')
OD = Path('/workspace/model_ooc_full'); OD.mkdir(exist_ok=True)

def cg_parts():
    st = {}
    for ln in (CG / 'memory.stat').read_text().splitlines():
        k, _, v = ln.partition(' ')
        st[k] = int(v)
    return st.get('anon', 0), st.get('slab', 0)

LIMIT = int((CG / 'memory.max').read_text().strip())
GIB = 1024 ** 3
GB = lambda b: b / GIB
peak = {'v': sum(cg_parts()), 'anon': cg_parts()[0], 'stop': False}

def watch():
    while not peak['stop']:
        an, sl = cg_parts()
        peak['v'] = max(peak['v'], an + sl); peak['anon'] = max(peak['anon'], an)
        time.sleep(0.5)

threading.Thread(target=watch, daemon=True).start()

from train9 import data as D
from train9.lgb_ooc import HierLGBMOOC

tag = a.tag or (f'rows{a.rows//1_000_000}M' if a.rows else a.variant)
b = D.load_basis('main', dataset='HI-Large')
cols = b.cols('no_abs')
X_tr, y_tr = b.X('tr'), b.y('tr')
n_all = len(y_tr)

if a.rows > 0:
    rows = np.arange(min(a.rows, n_all), dtype=np.int64)
elif a.variant == 'full':
    rows = None
else:
    rows = b.sample_idx(a.variant)
    if rows is None:
        sys.exit(f'표본 인덱스 없음: {a.variant}')

y_use = y_tr if rows is None else y_tr[rows]
n_use = len(y_use)
print(f'[full] variant={a.variant} tag={tag} · 학습행 {n_use:,}/{n_all:,} ({n_use/n_all*100:.2f}%) '
      f'· 패턴행 {int((y_use<=7).sum()):,} · 열 {len(cols)} · 트리 {a.rounds} · 상한 {GB(LIMIT):.2f} GB '
      f'· class_weight={a.class_weight} · seed={a.seed}', flush=True)

t0 = time.time()
rc, err = 0, None
try:
    m = HierLGBMOOC(n_estimators=a.rounds, n_jobs=a.jobs, batch_size=a.batch,
                    seed=a.seed, class_weight=a.class_weight)
    m.fit(X_tr, y_use, cols, rows=rows, valid=None)
except Exception as e:
    rc, err = 1, f'{type(e).__name__}: {e}'
    print(f'[full] 학습 실패 — {err}', flush=True)
fit_s = round(time.time() - t0, 1)
fit_peak, fit_peak_anon = peak['v'], peak['anon']
print(f'[full] 학습 {fit_s}초 · 최대 {GB(fit_peak):.2f} GB (anon {GB(fit_peak_anon):.2f})', flush=True)

res = dict(variant=a.variant, tag=tag, rows_used=int(n_use), rows_total=int(n_all),
           frac=round(n_use / n_all, 4), cols=int(len(cols)), rounds=a.rounds, jobs=a.jobs,
           batch=a.batch, fit_s=fit_s, rc=rc, error=err,
           class_weight=a.class_weight, seed=a.seed,
           timing=getattr(m, 'timing', {}) if rc == 0 else {},
           fit_peak_gb=round(GB(fit_peak), 2), fit_peak_anon_gb=round(GB(fit_peak_anon), 2),
           guard_stop_anon_gb=19.14, mem_limit_gb=round(GB(LIMIT), 2))

if rc == 0 and not a.no_score:
    # 두 머리의 가중에서 유도한 prior_model (지어낸 값이 아니라 lgb_ooc.py wb/wt 의 결과)
    #   이진 머리가 본 패턴 사전확률 p_pat_m: balanced → 정확히 0.5 (기존과 동일) / none → 표본 비율
    #   유형 머리는 항상 balanced → 8종 각각 1/8
    p_pat_m = float(m.bin_prior_model_)
    prior_model = np.full(9, p_pat_m / 8.0); prior_model[8] = 1.0 - p_pat_m
    res['bin_prior_model'] = p_pat_m
    cnt_full = np.bincount(y_tr, minlength=9).astype(np.float64)
    prior_true = cnt_full / cnt_full.sum()
    res['prior_model'] = prior_model.tolist()
    res['prior_true'] = prior_true.tolist()
    for split in [x.strip() for x in a.splits.split(',') if x.strip()]:
        gc.collect()
        t1 = time.time()
        p = D.predict_proba_chunked(m, b, split, cols, step=a.score_step)
        np.save(OD / f'proba_raw_{split}_{tag}.npy', p)
        # 보정도 청크로 — 3,592만 × 9 를 float64 로 통째 부풀리면 그것만 9 GB 다.
        # 수식은 train9/data.py:159 apply_prior_correction 그대로, 블록에만 적용한다.
        pc = np.empty_like(p)
        for lo in range(0, p.shape[0], a.score_step):
            hi = lo + a.score_step
            pc[lo:hi] = D.apply_prior_correction(
                p[lo:hi].astype(np.float64), prior_model, prior_true).astype(np.float32)
        np.save(OD / f'proba_prior_{split}_{tag}.npy', pc)
        res[f'score_{split}_s'] = round(time.time() - t1, 1)
        res[f'score_{split}_rows'] = int(p.shape[0])
        del p, pc
        print(f'[full] {split} 채점 {res[f"score_{split}_s"]}초 · 최대 {GB(peak["v"]):.2f} GB', flush=True)

peak['stop'] = True; time.sleep(0.6)
res['peak_gb'] = round(GB(peak['v']), 2)
res['peak_anon_gb'] = round(GB(peak['anon']), 2)
res['total_s'] = round(time.time() - t0, 1)
(OD / f'run_{tag}.json').write_text(json.dumps(res, ensure_ascii=False, indent=1))
print(json.dumps(res, ensure_ascii=False, indent=1), flush=True)


## 11. 블록(사건) 묶기 — `build_blocks_9class.py`

거래(엣지) 단위 예측을 사건(블록) 단위로 묶는다. attempt(정답 시도 ID, 상한/오라클) / window(정답 양성을 계좌·시간창으로) / pred(1차 알림을 계좌·시간창으로 — 완전 운영 조건) 세 정의.

원본 경로: `build_blocks_9class.py`

In [ ]:
#!/usr/bin/env python3
"""★1 — 블록(세탁 시도) 단위 유형 분류.

분류 단위를 거래에서 블록으로 올린다. 블록 정의 세 가지를 모두 만들어 비교한다.

  attempt  정답 시도 ID 로 묶음        -> 상한(오라클). 블록이 완벽히 주어졌을 때의 성능
  window   정답 양성을 계좌·시간창으로 -> 정답 탐지 + 운영 묶기 규칙
  pred     1차 모델 알림을 계좌·시간창 -> **완전 운영 조건**(정답을 전혀 안 씀)

블록의 split 은 **마지막 간선 시각** 기준이다. 첫 간선 기준으로 하면 train 블록이 test 구간
간선을 품게 되어 미래 정보 누수다(2026-08-25 세션에서 실제로 겪은 결함).
"""
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, '/workspace')
from train9 import data as D, blocks as B                # noqa: E402
from prep9 import postprocess_blocks as pb               # noqa: E402

import argparse
_ap = argparse.ArgumentParser()
_ap.add_argument('--dataset', default='HI-Small')
_ap.add_argument('--basis', default='10day')
_ap.add_argument('--model-dir', default='/workspace/model_9class', help='1차 모델 산출물(proba_*.npy, summary.json)')
_ap.add_argument('--out', default='/workspace/model_blocks')
_ap.add_argument('--hub-exclude-degree', type=int, default=0,
                 help='이 차수 **이상**인 계좌를 묶기의 다리에서 제외한다(0 = 제외 안 함, 기존 동작). '
                      'window·pred 정의에만 적용되고 attempt(오라클)는 정답 시도 ID 라 무관하다. '
                      '2026-09-07 추가 — 거대 블록(거래 61,628건·사건 4,548개)이 계좌를 매개로 '
                      '무관한 거래를 묶는 문제를 겨냥한다.')
_ap.add_argument('--window', type=int, default=None,
                 help='묶는 공백 임계(분). 기본은 전처리 메타의 window_chosen(=18일=25920분). '
                      '2026-09-08 추가 — 09-08 실험 타당성 감사에서 8~17일 구간이 전혀 검증된 적 '
                      '없다는 게 드러났고(hub_window_grid.py 격자 공백), 그 사이 재실행한 1단계 '
                      '격자에서 17일이 순도·최대블록 양쪽에서 18일을 파레토 우위로 이겼다. 이 값이 '
                      '2단계(실제 유형 분류)에서도 이기는지는 아직 아무도 확인하지 않았다 — '
                      '허브 축에서 이미 "1단계 온전율은 오르는데 2단계 E1 은 떨어지는" 사례가 나왔으므로 '
                      '1단계 지표만으로 W 를 확정하면 안 된다. window·pred 정의에만 적용된다.')
ARGS = _ap.parse_args()
OUT = Path(ARGS.out)
BASIS = ARGS.basis
DATASET = ARGS.dataset
MODEL_DIR = Path(ARGS.model_dir)
SEED = 42


def aux_columns(b: D.Basis, rows: np.ndarray) -> dict:
    """블록 피처에 필요한 보조 값(금액·수단·통화·통화불일치)을 저장된 X 에서 꺼낸다.

    2026-09-02: 전 행(HI-Large 1.8억)을 다 읽으면 17 GB 라 **블록 멤버 행만** 읽어 전체 길이 배열의
    그 자리에 채운다. 나머지 행은 0 이지만 블록 피처는 멤버 행만 참조하므로 값은 같다.
    """
    names = b.feat_names
    ci = {nm: i for i, nm in enumerate(names)}
    fmt_cols = [i for i, nm in enumerate(names) if nm.startswith('fmt_')]
    ccy_cols = [i for i, nm in enumerate(names) if nm.startswith('pccy_')]
    need = [ci['amt_paid_log'], ci['ccy_mismatch']] + fmt_cols + ccy_cols
    n_rows = b.meta['split']['rows']
    b0 = np.cumsum([0] + n_rows)
    n = int(b0[-1])
    rows = np.unique(rows)
    amt = np.zeros(n, dtype=np.float32); ccy_mm = np.zeros(n, dtype=np.float32)
    fmt = np.zeros(n, dtype=np.int16); ccy = np.zeros(n, dtype=np.int16)
    nf = len(fmt_cols)
    for k, tag in enumerate(('tr', 'va', 'te')):
        r = rows[(rows >= b0[k]) & (rows < b0[k + 1])]
        if len(r) == 0:
            continue
        X = b.X(tag)
        A = np.asarray(X[r - b0[k]])[:, need]
        amt[r] = np.expm1(A[:, 0]); ccy_mm[r] = A[:, 1]
        fmt[r] = A[:, 2:2 + nf].argmax(1); ccy[r] = A[:, 2 + nf:2 + nf + len(ccy_cols)].argmax(1)
    return {'amt': amt, 'ccy_mm': ccy_mm, 'fmt': fmt, 'ccy': ccy}


def main() -> None:
    t0 = time.time()
    OUT.mkdir(parents=True, exist_ok=True)
    b = D.load_basis(BASIS, dataset=DATASET)
    idx = dict(np.load(b.dir / 'index_full.npz'))
    W_meta = b.meta['blocks']['window_chosen']
    W = ARGS.window if ARGS.window is not None else W_meta
    if ARGS.window is not None:
        print(f'[window] 메타 기본값 {W_meta}분({W_meta/1440:.1f}일) 대신 '
              f'{W}분({W/1440:.1f}일)을 씀 (--window override)', flush=True)
    HUB = ARGS.hub_exclude_degree
    excl = None
    if HUB > 0:
        # 차수는 **전 데이터 기준**으로 센다 — 인프라 계좌를 식별하는 것이 목적이므로.
        # concatenate 하면 큰 배열을 하나 더 잡으므로 bincount 를 두 번 더한다.
        c1 = np.bincount(idx['src_id']); c2 = np.bincount(idx['dst_id'])
        n = max(len(c1), len(c2))
        deg = np.pad(c1, (0, n - len(c1))) + np.pad(c2, (0, n - len(c2)))
        excl = deg >= HUB
        print(f'[hub] 차수 ≥ {HUB:,} 계좌 {int(excl.sum()):,}개를 다리에서 제외 '
              f'(전체 {len(deg):,}개 중 {excl.mean()*100:.1f}%)', flush=True)
        del c1, c2, deg
    print(f'[basis] {BASIS} · 행 {len(idx["y9"]):,} · 창 {W}분 · 허브제외 {HUB or "없음"}', flush=True)

    defs = {}
    # (a) attempt — 정답 시도
    r = np.flatnonzero(idx['attempt'] >= 0)
    _, comp = np.unique(idx['attempt'][r], return_inverse=True)
    defs['attempt'] = (r, comp.astype(np.int32))
    # (b) window — 정답 양성을 묶기 규칙으로
    r = np.flatnonzero(idx['is_pos'].astype(bool))
    defs['window'] = (r, pb.link_components(idx['src_id'][r], idx['dst_id'][r],
                                            idx['ts_min'][r], W, exclude=excl))
    # (c) pred — 1차 모델 알림을 묶기 규칙으로 (정답 미사용)
    smy = json.loads((MODEL_DIR / 'summary.json').read_text(encoding='utf-8'))
    tau = smy['operating_point_from_val']['tau']
    bounds = [(0, b.meta['split']['rows'][0])]
    bounds.append((bounds[-1][1], bounds[-1][1] + b.meta['split']['rows'][1]))
    bounds.append((bounds[-1][1], bounds[-1][1] + b.meta['split']['rows'][2]))
    alert = np.zeros(len(idx['y9']), dtype=bool)
    for k, tag in enumerate(('val', 'test')):
        p = np.load(MODEL_DIR / f'proba_{tag}.npy')
        lo, hi = bounds[k + 1]
        alert[lo:hi] = p[:, :8].max(1) >= tau
    r = np.flatnonzero(alert)
    defs['pred'] = (r, pb.link_components(idx['src_id'][r], idx['dst_id'][r],
                                          idx['ts_min'][r], W, exclude=excl))
    print(f'[pred] 알림 행 {len(r):,} (val+test 구간만 — train 확률은 저장돼 있지 않음)',
          flush=True)
    print('[aux] 보조 컬럼 추출 중… (블록 멤버 행만)', flush=True)
    idx.update(aux_columns(b, np.concatenate([rows for rows, _ in defs.values()])))

    for nm, (rows, comp) in defs.items():
        X, meta = B.build_block_dataset(idx, rows, comp)
        np.save(OUT / f'X_{nm}.npy', X.astype(np.float32))
        meta.to_csv(OUT / f'meta_{nm}.csv', index=False)
        # 블록 -> 구성 간선 매핑(라벨 전파용)
        np.savez_compressed(OUT / f'members_{nm}.npz', rows=rows, comp=comp)
        lab = meta[meta['y'] >= 0]
        print(f'[{nm:8s}] 블록 {len(meta):,} (라벨 있음 {len(lab):,}) · '
              f'평균 간선 {meta["n_edges"].mean():.1f} · 순도 {lab["purity"].mean():.3f} · '
              f'split 분포 {meta["split"].value_counts().sort_index().to_dict()}', flush=True)
        print(f'           클래스 분포 {lab["y"].value_counts().sort_index().to_dict()}',
              flush=True)

    (OUT / 'build_note.json').write_text(json.dumps({
        'dataset': DATASET, 'basis': BASIS, 'model_dir': str(MODEL_DIR),
        'window_min': W, 'window_min_meta_default': W_meta,
        'window_overridden': ARGS.window is not None,
        'seed': SEED, 'hub_exclude_degree': HUB,
        'feature_names': B.FEATURE_NAMES,
        'definitions': {
            'attempt': '정답 시도 ID — 오라클 상한',
            'window': '정답 양성 + 계좌·시간창 묶기',
            'pred': '1차 모델 알림 + 계좌·시간창 묶기 (완전 운영 조건, val·test 구간만)'},
        'split_rule': '블록의 split = 마지막 간선이 속한 split (미래 정보 누수 방지)',
    }, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'[done] {time.time() - t0:,.1f}s -> {OUT}')


if __name__ == '__main__':
    main()


## 12. 블록 위상 피처 — `train9/blocks.py`

블록(세탁 시도) 단위 위상 피처. ablation 결과 위상 피처만으로도(0.650) 전체 47피처(0.634)보다 나음 — 병목이 분류기가 아니라 묶기임을 보여주는 근거.

원본 경로: `train9/blocks.py`

In [ ]:
"""블록(세탁 시도) 단위 위상 피처 — ★1 처방.

왜 필요한가: 8개 라벨은 **서브그래프 위상**인데 지금 1차 모델의 피처는 엣지 국소 정보다.
Chen et al.(NeurIPS 2020)은 MPNN 조차 3노드 이상 연결 패턴의 matching-count 를 못 한다고
증명했다(Thm 2 / Cor 1). 이웃 집계가 아예 없는 엣지 단위 트리 모델이 CYCLE·STACK·BIPARTITE 를
가릴 수 없는 것은 당연하다. 그래서 분류 단위를 **거래 -> 블록**으로 올린다.

RANDOM 이 F1 0 인 이유도 같은 자리에 있다. IBM 원 논문이 RANDOM 을 "CYCLE 과 같되 자금이
원 계좌로 돌아오지 않는 것"으로 정의하므로, **간선 하나만 보면 CYCLE 과 RANDOM 은 같은 객체다.**
닫는 간선의 유무는 블록 전체를 봐야 보인다.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

FEATURE_NAMES = [
    # 규모
    'n_edges', 'n_nodes', 'edges_per_node', 'n_unique_pairs', 'unique_pair_ratio',
    # 차수 — FAN-IN/FAN-OUT 은 여기서 갈린다(Chen Thm 3: star 패턴은 MPNN 도 가능)
    'max_out_deg', 'max_in_deg', 'mean_out_deg', 'mean_in_deg',
    'std_out_deg', 'std_in_deg', 'max_in_out_min', 'n_sources', 'n_sinks', 'n_pass',
    'source_ratio', 'sink_ratio', 'pass_ratio', 'fan_out_concentration', 'fan_in_concentration',
    # 비순환 구조 — SCATTER-GATHER / STACK 의 층수
    'is_dag', 'n_topo_generations', 'dag_longest_path', 'topo_width_max', 'topo_width_mean',
    # 순환 — CYCLE vs RANDOM 을 가르는 유일한 신호
    'n_simple_cycles', 'has_cycle', 'largest_scc', 'n_nontrivial_scc', 'reciprocal_ratio',
    'self_loop_ratio',
    # 이분성 — BIPARTITE
    'is_bipartite', 'src_dst_disjoint', 'n_src_only', 'n_dst_only',
    # 금액·시간
    'amt_total_log', 'amt_mean_log', 'amt_cv', 'amt_max_min_ratio_log', 'flow_conservation',
    'span_min_log', 'mean_dt_log', 'dt_cv', 'edges_per_hour_log',
    # 범주
    'n_unique_fmt', 'n_unique_ccy', 'ccy_mismatch_ratio',
    # 시간 순서 위상 (2026-09-03) — 정적 위상만으로는 SCATTER-GATHER↔GATHER-SCATTER(시간 순서만 반대),
    # CYCLE↔RANDOM(닫는 간선 시점)이 갈리지 않는다. LAS-GNN·TeMP-TraG 가 시간 순서를 집계에 넣은 것과 같은 취지.
    't_hub_in_before_out', 't_flow_time_corr', 't_first_src_outdeg', 't_last_dst_indeg',
    't_temporal_cycle', 't_temporal_cycle_len', 't_span_frac_gap_max', 't_sources_early',
]


CYCLE_ENUM_MAX_EDGES = 200      # 이 간선 수를 넘는 블록은 단순 순환 열거를 생략(위 주석 참조)


def _block_graph(src, dst):
    import networkx as nx
    g = nx.MultiDiGraph()
    g.add_edges_from(zip(src.tolist(), dst.tolist()))
    return g


def block_features(src, dst, ts, amt, fmt, ccy, ccy_mm) -> np.ndarray:
    """블록 하나의 위상·금액·시간 피처. 라벨을 쓰지 않으므로 운영 추론에 그대로 쓸 수 있다."""
    import networkx as nx
    m = len(src)
    nodes, loc = np.unique(np.concatenate([src, dst]), return_inverse=True)
    nn = len(nodes)
    ls, ld = loc[:m], loc[m:]
    outd = np.bincount(ls, minlength=nn).astype(float)
    ind = np.bincount(ld, minlength=nn).astype(float)
    self_l = int((ls == ld).sum())
    pkey = ls.astype(np.int64) * nn + ld
    upair = np.unique(pkey)
    rkey = ld.astype(np.int64) * nn + ls
    recip = float(np.isin(upair, rkey).mean()) if len(upair) else 0.0

    g = _block_graph(ls, ld)
    simple = nx.DiGraph()
    simple.add_edges_from({(int(a), int(b)) for a, b in zip(ls, ld) if a != b})
    is_dag = nx.is_directed_acyclic_graph(simple) if simple.number_of_nodes() else True
    if is_dag and simple.number_of_nodes():
        gens = list(nx.topological_generations(simple))
        n_gen = len(gens)
        widths = [len(x) for x in gens] or [0]
        longest = nx.dag_longest_path_length(simple)
    else:
        n_gen, widths, longest = 0, [0], 0
    # 2026-09-03: 단순 순환 열거는 간선 수에 지수적이다. HI-Large 의 window 정의에서 61,628간선 블록이 생겨
    # 13시간을 먹었다. 200간선 초과 블록은 열거를 생략하고 0 으로 둔다(has_cycle 은 SCC 로 그대로 계산).
    # HI-Small 은 최대 49간선이라 산출물이 바뀌지 않는다.
    if m > CYCLE_ENUM_MAX_EDGES:
        n_cyc = 0
    else:
        try:                                    # 길이 상한을 두어 폭발을 막는다
            n_cyc = sum(1 for _ in nx.simple_cycles(simple, length_bound=min(10, max(nn, 2))))
        except Exception:
            n_cyc = 0
    scc = [len(c) for c in nx.strongly_connected_components(simple)] or [0]
    und = nx.Graph()
    und.add_edges_from({(int(a), int(b)) for a, b in zip(ls, ld) if a != b})
    try:
        is_bip = float(nx.is_bipartite(und)) if und.number_of_nodes() else 1.0
    except Exception:
        is_bip = 0.0

    ts_s = np.sort(ts.astype(np.float64))
    span = float(ts_s[-1] - ts_s[0]) if m > 1 else 0.0
    dts = np.diff(ts_s) if m > 1 else np.zeros(1)
    a = amt.astype(np.float64)
    in_sum = np.bincount(ld, weights=a, minlength=nn)
    out_sum = np.bincount(ls, weights=a, minlength=nn)
    pas = (outd > 0) & (ind > 0)
    cons = float((np.minimum(in_sum[pas], out_sum[pas]) /
                  (np.maximum(in_sum[pas], out_sum[pas]) + 1e-9)).mean()) if pas.any() else 0.0

    v = [
        np.log1p(m), np.log1p(nn), m / max(nn, 1), np.log1p(len(upair)), len(upair) / m,
        outd.max(), ind.max(), outd.mean(), ind.mean(), outd.std(), ind.std(),
        float(np.minimum(ind, outd).max()),
        float(((outd > 0) & (ind == 0)).sum()),          # n_sources: 내보내기만 하는 노드
        float(((ind > 0) & (outd == 0)).sum()),          # n_sinks:  받기만 하는 노드
        float(pas.sum()),                                # n_pass:   통과 노드
        float(((outd > 0) & (ind == 0)).sum()) / nn, float(((ind > 0) & (outd == 0)).sum()) / nn,
        float(pas.sum()) / nn, outd.max() / m, ind.max() / m,
        float(is_dag), float(n_gen), float(longest), float(max(widths)), float(np.mean(widths)),
        float(n_cyc), float(max(scc) > 1 or self_l > 0), float(max(scc)),
        float(sum(1 for s in scc if s > 1)), recip, self_l / m,
        is_bip, 1.0 if not np.intersect1d(src, dst).size else 0.0,
        float(((outd > 0) & (ind == 0)).sum()), float(((ind > 0) & (outd == 0)).sum()),
        np.log1p(a.sum()), np.log1p(a.mean()), float(a.std() / (a.mean() + 1e-9)),
        np.log1p(a.max() / (a.min() + 1e-9)), cons,
        np.log1p(span), np.log1p(dts.mean()), float(dts.std() / (dts.mean() + 1e-9)),
        np.log1p(m / (span / 60.0 + 1.0)),
        float(len(np.unique(fmt))), float(len(np.unique(ccy))), float(np.mean(ccy_mm)),
    ]
    v.extend(temporal_features(ls, ld, ts, outd, ind, nn, simple, is_dag))
    out = np.array(v, dtype=np.float64)
    assert len(out) == len(FEATURE_NAMES), f'{len(out)} != {len(FEATURE_NAMES)}'
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)


def temporal_features(ls, ld, ts, outd, ind, nn, simple, is_dag, max_len: int = 10) -> list:
    """시간 순서를 쓰는 위상 피처 8개. ls/ld 는 0..nn-1 로 재부여된 노드, ts 는 분 단위 시각."""
    import networkx as nx
    m = len(ls)
    t = ts.astype(np.float64)
    if m < 2:
        return [0.0, 0.0, float(outd[ls[0]]), float(ind[ld[0]]), 0.0, 0.0, 0.0, 0.0]
    # (1) 허브(총차수 최대 노드)의 수신이 송신보다 먼저인가: gather→scatter 면 1 에 가깝고 scatter→gather 면 0 에 가깝다
    hub = int(np.argmax(outd + ind))
    t_in, t_out = t[ld == hub], t[ls == hub]
    if len(t_in) and len(t_out):
        hub_in_before = float((t_in[:, None] < t_out[None, :]).mean())
    else:
        hub_in_before = 0.5
    # (2) 흐름 방향과 시간의 상관: DAG 세대(깊이)와 시각의 순위 상관 — 체인/층 구조가 시간순으로 흐르면 높다
    if is_dag and simple.number_of_nodes():
        gen_of = {}
        for g_i, gen in enumerate(nx.topological_generations(simple)):
            for node in gen:
                gen_of[node] = g_i
        depth = np.array([gen_of.get(int(a), 0) for a in ls], dtype=np.float64)
        if depth.std() > 0 and t.std() > 0:
            dr, tr = depth.argsort().argsort().astype(float), t.argsort().argsort().astype(float)
            flow_corr = float(np.corrcoef(dr, tr)[0, 1])
        else:
            flow_corr = 0.0
    else:
        flow_corr = 0.0
    # (3) 첫 간선 송신자의 송신 차수 / (4) 마지막 간선 수신자의 수신 차수 — fan-out 은 허브가 먼저 뿌리고, fan-in 은 허브가 마지막에 받는다
    first_src_outdeg = float(outd[ls[int(np.argmin(t))]])
    last_dst_indeg = float(ind[ld[int(np.argmax(t))]])
    # (5)(6) 시간 순서를 지키는 순환(temporal cycle)이 있는가·그 최소 길이 — 정적 순환(has_cycle)과 달리 RANDOM 을 가른다
    tcyc_len = 0
    if not is_dag:
        adj_t: dict[int, list] = {}
        for i in range(m):
            if ls[i] != ld[i]:
                adj_t.setdefault(int(ls[i]), []).append((int(ld[i]), t[i]))
        for start in list(adj_t)[:50]:                    # 노드 50개까지만 시작점으로
            frontier = [(start, -np.inf)]
            seen = set()
            for depth_i in range(1, min(max_len, m) + 1):
                nxt = []
                for node, tl in frontier:
                    for (y, ty) in adj_t.get(node, ()):
                        if ty <= tl:
                            continue
                        if y == start:
                            tcyc_len = depth_i if tcyc_len == 0 else min(tcyc_len, depth_i)
                            break
                        key = (y, ty)
                        if key not in seen:
                            seen.add(key); nxt.append(key)
                    if tcyc_len:
                        break
                if tcyc_len or not nxt:
                    break
                frontier = nxt
            if tcyc_len:
                break
    # (7) 최대 시간 공백의 비율 — 두 국면(모았다가 뿌림)이면 가운데 공백이 크다
    ts_s = np.sort(t)
    gaps = np.diff(ts_s)
    span = float(ts_s[-1] - ts_s[0])
    gap_max_frac = float(gaps.max() / span) if span > 0 else 0.0
    # (8) 소스 노드의 간선이 전반부에 몰려 있는가
    src_nodes = (outd > 0) & (ind == 0)
    e_src = src_nodes[ls]
    sources_early = float((t[e_src] <= np.median(t)).mean()) if e_src.any() else 0.5
    return [hub_in_before, flow_corr, first_src_outdeg, last_dst_indeg,
            float(tcyc_len > 0), float(tcyc_len), gap_max_frac, sources_early]


def build_block_dataset(idx: dict, rows: np.ndarray, comp: np.ndarray,
                        ccy_mm: np.ndarray | None = None) -> tuple[np.ndarray, pd.DataFrame]:
    """블록 묶음 -> (X[n_block, F], meta). 블록의 split 은 **마지막 간선 시각** 기준이다."""
    src, dst = idx['src_id'][rows], idx['dst_id'][rows]
    ts, y9 = idx['ts_min'][rows], idx['y9'][rows]
    att, split = idx['attempt'][rows], idx['split'][rows]
    amt = idx['amt'][rows] if 'amt' in idx else np.ones(len(rows))
    fmt = idx['fmt'][rows] if 'fmt' in idx else np.zeros(len(rows))
    ccy = idx['ccy'][rows] if 'ccy' in idx else np.zeros(len(rows))
    mm = ccy_mm[rows] if ccy_mm is not None else np.zeros(len(rows))

    order = np.argsort(comp, kind='stable')
    offs = np.searchsorted(comp[order], np.arange(comp.max() + 2))
    X, meta = [], []
    for k in range(int(comp.max()) + 1):
        r = order[offs[k]:offs[k + 1]]
        if len(r) == 0:
            continue
        X.append(block_features(src[r], dst[r], ts[r], amt[r], fmt[r], ccy[r], mm[r]))
        lab = np.bincount(y9[r][y9[r] <= 7], minlength=8)
        meta.append({'block': k, 'n_edges': len(r),
                     'y': int(lab.argmax()) if lab.sum() else -1,
                     'purity': float(lab.max() / lab.sum()) if lab.sum() else 0.0,
                     'split': int(split[r].max()),          # 마지막 간선이 속한 split
                     'ts_last': int(ts[r].max()),
                     'n_attempts': int(len(np.unique(att[r][att[r] >= 0]))),
                     'attempt': int(np.bincount(att[r][att[r] >= 0]).argmax())
                                if (att[r] >= 0).any() else -1})
    return np.vstack(X), pd.DataFrame(meta)


## 13. 알림 블록 후처리 — `prep9/postprocess_blocks.py`

거래별 예측을 '알림 블록'으로 묶는 후처리 프로그램(2026-08-26 회의 결정 2·3 — 패턴 적발=패턴 블록 알림, 패턴외 적발=단건 알림).

원본 경로: `prep9/postprocess_blocks.py`

In [ ]:
"""후처리 프로그램 — 거래별 예측을 '알림 블록'으로 묶는다.

2026-08-26 회의 결정 2·3:
  - 1차 9-class 에서 8종 패턴으로 적발된 거래 -> **패턴 블록 단위 알림**
  - 2차 이진에서 적발된 패턴 외 거래        -> **단건 블록 알림**
  - 트리 모델·GNN 모두 거래별 점수만 낸다. 연관 거래를 묶는 것은 별도 후처리의 몫이다.

이 모듈은 라벨을 쓰지 않는다. 입력은 '어떤 거래가 무엇으로 판정됐는가' 뿐이라
운영 추론 시점에 그대로 돌릴 수 있고, 정답 라벨을 넣으면 평가용 기준 블록이 나온다.

묶는 규칙: **계좌 공유 + window 분 이내**의 이행적 연결 컴포넌트.
  A-B 가 창 안이고 B-C 가 창 안이면 A 와 C 가 멀어도 한 블록이다.
"""
from __future__ import annotations

import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

__all__ = ['link_components', 'make_alerts', 'block_frame', 'evaluate_blocks']


def link_components(src: np.ndarray, dst: np.ndarray, ts: np.ndarray,
                    window_min: int, exclude: np.ndarray | None = None) -> np.ndarray:
    """거래들을 '계좌 공유 + window 분 이내'로 이어붙인 연결 컴포넌트 ID.

    엣지 하나를 (src, dst) 두 개의 (계좌, 시각) 사건으로 펼친 뒤 같은 계좌 안에서
    시각순 인접쌍만 잇는다. 완전 그래프를 만들지 않으므로 O(n log n) 이다.

    exclude : 계좌 id 로 색인되는 bool 배열. True 인 계좌는 **다리로 쓰지 않는다**
              (화두 13 특수 허브 — HI-Small 에서는 은행 070 의 15개 계좌가 거래 8.9% 에
              닿는다). 그 계좌에 닿은 거래는 반대쪽 계좌로만 이어지며, 양끝이 모두 제외
              계좌면 단독 블록이 된다. 2026-09-02 추가, 기본값 None 이면 원래 동작.
    """
    m = len(src)
    if m == 0:
        return np.zeros(0, dtype=np.int32)
    ent = np.concatenate([src, dst])
    eid = np.tile(np.arange(m, dtype=np.int32), 2)
    t = np.concatenate([ts, ts])
    if exclude is not None:
        keep = ~np.asarray(exclude, dtype=bool)[ent]
        ent, eid, t = ent[keep], eid[keep], t[keep]
        if len(ent) == 0:
            return np.arange(m, dtype=np.int32)
    o = np.lexsort((t, ent))
    ent_s, eid_s, t_s = ent[o], eid[o], t[o]
    link = (ent_s[1:] == ent_s[:-1]) & ((t_s[1:] - t_s[:-1]) <= window_min)
    rows, cols = eid_s[:-1][link], eid_s[1:][link]
    g = coo_matrix((np.ones(len(rows), dtype=np.int8), (rows, cols)), shape=(m, m))
    _, comp = connected_components(g, directed=False)
    return comp.astype(np.int32)


def make_alerts(pred_class: np.ndarray, pred_binary: np.ndarray | None,
                src: np.ndarray, dst: np.ndarray, ts: np.ndarray,
                window_min: int, row_id: np.ndarray | None = None) -> dict:
    """거래별 판정 -> 알림 목록.

    pred_class : 각 거래의 1차 9-class 예측(0~7 = 8종 패턴, 8 = 패턴 외)
    pred_binary: 패턴 외로 간 거래의 2차 이진 예측(1 = 세탁). None 이면 단건 알림 없음.
                 pred_class 와 같은 길이이며 8 이 아닌 행의 값은 무시한다.
    반환: {'pattern': {...}, 'single': {...}} — 두 알림 종류를 분리해 돌려준다.
    """
    n = len(pred_class)
    row_id = np.arange(n, dtype=np.int64) if row_id is None else np.asarray(row_id)
    hit = np.flatnonzero((pred_class >= 0) & (pred_class <= 7))

    comp = link_components(src[hit], dst[hit], ts[hit], window_min)
    n_blk = int(comp.max()) + 1 if len(comp) else 0
    order = np.argsort(comp, kind='stable')
    offs = np.searchsorted(comp[order], np.arange(n_blk + 1))
    pattern = {
        'member_rows': row_id[hit][order],          # 블록 순서로 정렬된 구성 거래
        'offsets': offs,                            # 블록 b = member_rows[offs[b]:offs[b+1]]
        'n_blocks': n_blk,
        'block_class': _modal_class(pred_class[hit][order], offs, n_blk),
        'window_min': window_min,
    }

    if pred_binary is None:
        single_rows = np.zeros(0, dtype=np.int64)
    else:
        sm = (pred_class == 8) & (np.asarray(pred_binary) == 1)
        single_rows = row_id[np.flatnonzero(sm)]
    single = {'member_rows': single_rows, 'n_blocks': len(single_rows),
              'unit': '단건(거래 1건 = 알림 1건)'}
    return {'pattern': pattern, 'single': single}


def _modal_class(cls_sorted: np.ndarray, offs: np.ndarray, n_blk: int) -> np.ndarray:
    """블록 라벨 = 구성 거래 예측 클래스의 최빈값."""
    if n_blk == 0:
        return np.zeros(0, dtype=np.int8)
    blk = np.repeat(np.arange(n_blk), np.diff(offs))
    cnt = np.bincount(blk.astype(np.int64) * 8 + cls_sorted.astype(np.int64),
                      minlength=n_blk * 8).reshape(n_blk, 8)
    return cnt.argmax(axis=1).astype(np.int8)


def block_frame(alerts: dict, ts: np.ndarray, src: np.ndarray,
                dst: np.ndarray) -> 'pd.DataFrame':
    """패턴 블록 요약 표(알림 화면에 그대로 올릴 최소 필드).

    `ts`/`src`/`dst` 는 `make_alerts` 에 넘긴 `row_id` 로 색인되는 배열이어야 한다.
    (`row_id` 를 생략했다면 make_alerts 에 넘긴 것과 같은 순서의 전체 배열)
    """
    import pandas as pd
    p = alerts['pattern']
    mem, offs = p['member_rows'], p['offsets']
    rows = []
    for b in range(p['n_blocks']):
        s = mem[offs[b]:offs[b + 1]]                 # 구성 거래의 row_id
        rows.append({'block_id': b, 'block_class': int(p['block_class'][b]),
                     'n_edges': len(s),
                     'n_accounts': int(len(np.unique(np.concatenate([src[s], dst[s]])))),
                     'ts_start': int(ts[s].min()), 'ts_end': int(ts[s].max()),
                     'span_min': int(ts[s].max() - ts[s].min())})
    return pd.DataFrame(rows)


def evaluate_blocks(comp: np.ndarray, attempt: np.ndarray,
                    eligible: np.ndarray | None = None,
                    per_attempt: bool = False) -> dict:
    """묶기 규칙이 정답 '시도'를 얼마나 되살리는지 잰다(정답이 있을 때만 쓴다).

    strict_recovery : 그 시도의 (여기 들어온) 거래 전부가 정확히 한 블록에 있고, 그 블록에
                      다른 시도의 거래가 섞이지 않은 시도의 비율.
    purity          : 블록 안에서 최빈 시도가 차지하는 비율의 가중 평균.
    fragmentation   : 시도 하나가 쪼개진 블록 수.

    두 가지를 반드시 알고 써야 한다.

    1. `attempt < 0` 인 행은 **계산 전에 버려진다.** 이 함수는 '라벨된 시도끼리 어떻게
       묶이는가'만 재며, 블록에 섞인 오탐(정상 거래)은 순도에 반영되지 않는다.
       따라서 여기 나오는 순도는 **운영 알림의 순도가 아니라 묶기 규칙의 상한**이다.
    2. 거래가 1건만 남은 시도는 어떤 창에서도 자동으로 strict 성공이 된다(블록도 1개다).
       기간 절단으로 잘린 시도가 섞이면 지표가 부풀려지므로, 채점 대상을 `eligible`
       (완결 + 2건 이상)로 제한해서 부른다.

    eligible : 채점에 넣을 attempt id 배열. None 이면 전부. 블록 구성 자체는
               eligible 밖 시도의 거래도 포함한 상태로 계산되므로, 오염(다른 시도와 섞임)은
               정상적으로 감지된다.
    """
    ok = attempt >= 0
    comp, attempt = comp[ok], attempt[ok]
    if len(comp) == 0:
        return {'n_attempts': 0, 'n_scored': 0}
    ua, ai = np.unique(attempt, return_inverse=True)
    ub, bi = np.unique(comp, return_inverse=True)
    na, nb = len(ua), len(ub)

    pair = np.unique(ai.astype(np.int64) * nb + bi)          # (시도, 블록) 조합
    p_att, p_blk = pair // nb, pair % nb
    blocks_per_attempt = np.bincount(p_att, minlength=na)
    attempts_per_block = np.bincount(p_blk, minlength=nb)

    # 시도가 블록 하나에만 걸쳐 있고(쪼개짐 없음), 그 블록에 다른 시도가 없을 때만 strict.
    first = np.searchsorted(p_att, np.arange(na))             # pair 는 이미 시도 순 정렬
    blk_of = p_blk[np.minimum(first, len(pair) - 1)]
    strict = (blocks_per_attempt == 1) & (attempts_per_block[blk_of] == 1)

    score = np.ones(na, dtype=bool) if eligible is None else np.isin(ua, eligible)
    cnt = np.bincount(bi.astype(np.int64) * na + ai, minlength=nb * na).reshape(nb, na)
    tot = cnt.sum(1)
    purity = float((cnt.max(1) / np.maximum(tot, 1) * tot).sum() / tot.sum())
    extra = {'attempt_ids': ua, 'strict_flags': strict, 'scored': score} if per_attempt else {}
    return {**extra, 'n_attempts': int(na), 'n_scored': int(score.sum()),
            'n_blocks': int(nb),
            'strict_recovery': float(strict[score].mean()) if score.any() else float('nan'),
            'purity': purity,
            'frag_median': float(np.median(blocks_per_attempt[score])) if score.any() else float('nan'),
            'frag_mean': float(blocks_per_attempt[score].mean()) if score.any() else float('nan'),
            'mixed_blocks': int((attempts_per_block > 1).sum())}


## 14. Cascade 재순위 — `rerank_ego_9class.py`

1차 점수 상위 후보(전체 1%, recall@1%가 97~99%라 양성이 거의 다 포함)에만 ego-서브그래프 구조 피처를 계산해 GBDT로 재순위(2026-09-03). IBM Graph Feature Preprocessor·ExSTraQt 문헌 근거 인용.

원본 경로: `rerank_ego_9class.py`

In [ ]:
#!/usr/bin/env python3
"""Cascade 재순위 — 1차 점수 상위 후보에만 ego-서브그래프 구조 피처를 계산해 GBDT 로 다시 순위를 매긴다. 2026-09-03.

목표: 팀 정량 목표 **정밀도 90%**, 비교 관례 **재현율 0.70 고정 후 정밀도**. 정밀도는 운영점 근처의 순위로 결정되므로
      상위 후보(전체의 1%: recall@1% 가 97~99% 라 양성이 거의 다 들어 있음)에만 비싼 그래프 피처를 붙이면 된다.

문헌 근거
  · IBM Graph Feature Preprocessor(2024): 거래별 부분그래프 패턴 개수(fan-in/out, scatter-gather, 순환)를 GBDT 에 넣어
    HI-Large LightGBM F1 24.5%→58.0%. 우리 1차 피처 76개는 전부 1홉이라 이 신호가 없다.
  · ExSTraQt(2026, 프리프린트): 흐름 전파·커뮤니티·2홉 ego 피처 + XGBoost 로 HI-Large F1 78.1.
  · 우리 실측: v9 에서 "전역 계좌 이력" 피처는 GNN 에 손해(−0.025) → 국소·중심 간선 기준 피처만 쓴다.

누수 원칙(화두 15): ego 창은 중심 시각 **이전**(w_back)만 쓴다(--w-fwd 0). 사후 배치라면 --w-fwd 를 열 수 있으나 기본은 0.
학습/평가: 재순위기는 **val 후보**로 학습하고(1차 모델이 안 본 구간), **test 후보**로 한 번 평가. 후보 밖 행은 알림 불가(-inf).

점수 원천
  --source 9class : model_9class/proba_{val,test}.npy (보정 확률) 의 max_{c<8}. 좌표 = processed_9class/HI-Small_10day
  --source v13    : artifacts_v13_undersample/base_{gnn,tab}_{val,test}.npy 순위 결합(w2=0.30). 좌표 = us_v14/cache (SRC/DST/TS_MIN/Y)

실행: python3 rerank_ego_9class.py --source 9class [--top-frac 0.01 --w-back 4320 --hops 2 --deg-cap 50]
"""
from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

sys.path.insert(0, '/workspace')
from train9 import blocks as B          # noqa: E402
from prep9 import ego as E              # noqa: E402

AP = argparse.ArgumentParser()
AP.add_argument('--source', default='9class', choices=['9class', 'v13'])
AP.add_argument('--top-frac', type=float, default=0.01)
AP.add_argument('--w-back', type=int, default=4320)
AP.add_argument('--w-fwd', type=int, default=0)
AP.add_argument('--hops', type=int, default=2)
AP.add_argument('--deg-cap', type=int, default=50)
AP.add_argument('--hub-k', type=int, default=15)
AP.add_argument('--out', default='')
AP.add_argument('--tag', default='')
ARGS = AP.parse_args()
OUT = Path(ARGS.out or f'/workspace/model_rerank_{ARGS.source}'); OUT.mkdir(parents=True, exist_ok=True)
T0 = time.time()
AMT_TIME = {'amt_total_log', 'amt_mean_log', 'amt_cv', 'amt_max_min_ratio_log', 'flow_conservation',
            'span_min_log', 'mean_dt_log', 'dt_cv', 'edges_per_hour_log', 'n_unique_fmt', 'n_unique_ccy',
            'ccy_mismatch_ratio'}
TOPO = [i for i, n in enumerate(B.FEATURE_NAMES) if n not in AMT_TIME]


def log(*a):
    print(f'[{time.time() - T0:7.1f}s]', *a, flush=True)


def rank01(x):
    o = np.argsort(np.argsort(x, kind='stable'), kind='stable')
    return o / max(len(o) - 1, 1)


# ── 좌표계·점수 ─────────────────────────────────────────────────────────────
if ARGS.source == '9class':
    D = Path('/workspace/processed_9class/HI-Small_10day')
    idx = dict(np.load(D / 'index_full.npz'))
    src, dst, ts, y9, split = idx['src_id'], idx['dst_id'], idx['ts_min'], idx['y9'], idx['split']
    is_pos = idx['is_pos'].astype(bool); is_pat = y9 <= 7
    meta = json.load(open(D / 'features_meta.json')); rs = meta['split']['rows']
    B0 = [0, rs[0], rs[0] + rs[1], sum(rs)]; n_nodes = meta['split']['n_accounts']
    score = np.full(len(y9), -np.inf)
    P9 = {}
    for k, tag in enumerate(('val', 'test')):
        p = np.load(f'/workspace/model_9class/proba_{tag}.npy')
        score[B0[k + 1]:B0[k + 2]] = p[:, :8].max(1); P9[tag] = p
    extra_names = [f'p9_{c}' for c in range(9)]

    def extra_feats(rows_, tag):
        lo = B0[1] if tag == 'val' else B0[2]
        return P9[tag][rows_ - lo]
else:
    C = Path('/workspace/us_v14/cache'); A = Path('/workspace/artifacts_v13_undersample')
    m = json.load(open(C / 'meta.json')); B0 = [0, m['n_tr'], m['n_va'], m['n']]
    src, dst = np.load(C / 'SRC.npy').astype(np.int32), np.load(C / 'DST.npy').astype(np.int32)
    ts = np.load(C / 'TS_MIN.npy').astype(np.int64); Y = np.load(C / 'Y.npy'); is_pos = Y > 0
    is_pat = np.load(C / 'PAT_MASK.npy').astype(bool) & is_pos
    n_nodes = int(max(src.max(), dst.max())) + 1
    gv, gt = np.load(A / 'base_gnn_val.npy'), np.load(A / 'base_gnn_test.npy')
    tv, tt = np.load(A / 'base_tab_val.npy'), np.load(A / 'base_tab_test.npy')
    w2 = 0.30
    score = np.full(len(Y), -np.inf)
    score[B0[1]:B0[2]] = (1 - w2) * rank01(gv) + w2 * rank01(tv)
    score[B0[2]:B0[3]] = (1 - w2) * rank01(gt) + w2 * rank01(tt)
    RAW = {'val': np.stack([gv, tv], 1), 'test': np.stack([gt, tt], 1)}
    extra_names = ['gnn_p', 'tab_p']

    def extra_feats(rows_, tag):
        lo = B0[1] if tag == 'val' else B0[2]
        return RAW[tag][rows_ - lo]

adj = E.TemporalAdj(src, dst, ts, n_nodes=n_nodes)
hub = np.zeros(adj.n, dtype=bool); hub[np.argsort(-adj.deg_total)[:ARGS.hub_k]] = True
log(f'source={ARGS.source} · 행 {len(score):,} · 계좌 {adj.n:,}')

# ── 후보 = split 안 점수 상위 top_frac ───────────────────────────────────────
def candidates(tag):
    lo, hi = (B0[1], B0[2]) if tag == 'val' else (B0[2], B0[3])
    rr = np.arange(lo, hi); k = int(round(ARGS.top_frac * len(rr)))
    top = rr[np.argsort(-score[rr], kind='stable')[:k]]
    return np.sort(top)

cand = {t: candidates(t) for t in ('val', 'test')}
for t, c in cand.items():
    log(f'{t} 후보 {len(c):,} (양성 {int(is_pos[c].sum())}/{int(is_pos[B0[1] if t == "val" else B0[2]:B0[2] if t == "val" else B0[3]].sum())} 포함, '
        f'패턴 {int(is_pat[c].sum())}/{int(is_pat[B0[1] if t == "val" else B0[2]:B0[2] if t == "val" else B0[3]].sum())})')

# ── ego 피처 ─────────────────────────────────────────────────────────────────
FEAT_NAMES = [B.FEATURE_NAMES[i] for i in TOPO] + E.CENTER_FEATURE_NAMES + ['base_score', 'base_rank'] + extra_names


def featurize(rows_, tag):
    X = np.empty((len(rows_), len(FEAT_NAMES)), dtype=np.float32)
    lo, hi = (B0[1], B0[2]) if tag == 'val' else (B0[2], B0[3])
    rk = rank01(score[lo:hi])
    ex = extra_feats(rows_, tag)
    for i, e in enumerate(rows_):
        eids = E.ego_edges(adj, int(e), ARGS.w_back, ARGS.w_fwd, ARGS.hops, ARGS.deg_cap, exclude=hub)
        s_, d_, t_ = src[eids], dst[eids], ts[eids]
        topo = B.block_features(s_, d_, t_, np.ones(len(eids)), np.zeros(len(eids)), np.zeros(len(eids)),
                                np.zeros(len(eids)))[TOPO]
        cf = E.center_features(s_, d_, t_, int(np.searchsorted(eids, e)))
        X[i] = np.concatenate([topo, cf, [score[e], rk[e - lo]], ex[i]])
        if i % 2000 == 0 and i:
            log(f'  {tag} {i}/{len(rows_)}')
    return X

t1 = time.time(); Xv = featurize(cand['val'], 'val'); Xt = featurize(cand['test'], 'test')
log(f'ego 피처 완료 {time.time() - t1:.0f}s · 피처 {len(FEAT_NAMES)}')
np.save(OUT / f'X_val{ARGS.tag}.npy', Xv); np.save(OUT / f'X_test{ARGS.tag}.npy', Xt)
np.save(OUT / f'cand_val{ARGS.tag}.npy', cand['val']); np.save(OUT / f'cand_test{ARGS.tag}.npy', cand['test'])

# ── 재순위기: val 후보로 학습(양성=세탁), test 후보 평가 ────────────────────
yv, yt = is_pos[cand['val']].astype(int), is_pos[cand['test']].astype(int)
# val 안에서 5-fold 로 하이퍼 대략 확인(선택은 val 내부에서만)
from sklearn.model_selection import StratifiedKFold
best = None
for params in ({'num_leaves': 15, 'learning_rate': 0.05, 'n_estimators': 400},
               {'num_leaves': 31, 'learning_rate': 0.05, 'n_estimators': 400},
               {'num_leaves': 63, 'learning_rate': 0.03, 'n_estimators': 600}):
    oof = np.zeros(len(yv))
    for tr_i, va_i in StratifiedKFold(5, shuffle=True, random_state=42).split(Xv, yv):
        mdl = lgb.LGBMClassifier(objective='binary', subsample=0.9, subsample_freq=1, colsample_bytree=0.8,
                                 reg_lambda=1.0, min_child_samples=20, n_jobs=4, random_state=42, verbose=-1, **params)
        mdl.fit(Xv[tr_i], yv[tr_i]); oof[va_i] = mdl.predict_proba(Xv[va_i])[:, 1]
    ap = average_precision_score(yv, oof)
    log(f'  val 5-fold OOF AP {ap:.4f} {params}')
    if best is None or ap > best[0]:
        best = (ap, params, oof)
ap_v, params, oof_v = best
final = lgb.LGBMClassifier(objective='binary', subsample=0.9, subsample_freq=1, colsample_bytree=0.8,
                           reg_lambda=1.0, min_child_samples=20, n_jobs=4, random_state=42, verbose=-1, **params)
final.fit(Xv, yv)
pt = final.predict_proba(Xt)[:, 1]
imp = pd.DataFrame({'피처': FEAT_NAMES, '중요도': final.feature_importances_}).sort_values('중요도', ascending=False)
imp.to_csv(OUT / f'feature_importance{ARGS.tag}.csv', index=False)


# ── 운영점 비교: 재현율 0.70 고정(패턴 기준 / 세탁 전체 기준) 정밀도, P90 지점 재현율 ─────────────
def curve(scores_full, lo, hi, denom_mask):
    rr = np.arange(lo, hi); s = scores_full[rr]; o = np.argsort(-s, kind='stable')
    pos = is_pos[rr][o]; pat = is_pat[rr][o]; den = denom_mask[rr].sum()
    c_pos = np.cumsum(pos); c_den = np.cumsum(denom_mask[rr][o])
    k = np.arange(1, len(o) + 1)
    return pd.DataFrame({'k': k, 'precision_any': c_pos / k, 'recall': c_den / max(den, 1)})


def op_points(cv, name):
    r70 = cv[cv['recall'] >= 0.70]
    p90 = cv[cv['precision_any'] >= 0.90]
    return {f'{name}_K@R70': int(r70['k'].iloc[0]) if len(r70) else None,
            f'{name}_P@R70': float(r70['precision_any'].iloc[0]) if len(r70) else None,
            f'{name}_R@P90': float(p90['recall'].max()) if len(p90) else 0.0,
            f'{name}_K@P90': int(p90.loc[p90['recall'].idxmax(), 'k']) if len(p90) else None}


rows = {}
for tag, (lo, hi), yy, pp, cc in (('val', (B0[1], B0[2]), yv, oof_v, cand['val']), ('test', (B0[2], B0[3]), yt, pt, cand['test'])):
    base = score.copy()
    rer = np.full(len(score), -np.inf); rer[cc] = pp
    for nm, sc in (('base', base), ('rerank', rer)):
        for den_nm, den in (('pat', is_pat), ('pos', is_pos)):
            cv = curve(sc, lo, hi, den)
            rows.update({f'{tag}_{nm}_{den_nm}_{k.split("_", 1)[1]}': v for k, v in op_points(cv, nm).items()})
    rows[f'{tag}_rerank_AP_cand'] = float(average_precision_score(yy, pp))
    rows[f'{tag}_base_AP_cand'] = float(average_precision_score(yy, score[cc]))

res = pd.Series(rows)
print('\n=== 운영점 비교 (재현율 고정 후 정밀도) ===')
for tag in ('val', 'test'):
    for den_nm, lab in (('pat', '패턴 재현율 0.70'), ('pos', '세탁 전체 재현율 0.70')):
        b = res.get(f'{tag}_base_{den_nm}_P@R70'); r = res.get(f'{tag}_rerank_{den_nm}_P@R70')
        bk = res.get(f'{tag}_base_{den_nm}_K@R70'); rk_ = res.get(f'{tag}_rerank_{den_nm}_K@R70')
        print(f'  {tag:4s} [{lab}] 기준 정밀도 {b if b is None else round(b, 4)} (K={bk}) → 재순위 {r if r is None else round(r, 4)} (K={rk_})'
              f' | P90 지점 재현율: 기준 {res[f"{tag}_base_{den_nm}_R@P90"]:.3f} → 재순위 {res[f"{tag}_rerank_{den_nm}_R@P90"]:.3f}')
    print(f'  {tag:4s} 후보 내 AP: 기준 {res[f"{tag}_base_AP_cand"]:.4f} → 재순위 {res[f"{tag}_rerank_AP_cand"]:.4f}')
print('\n=== 재순위기 중요도 상위 15 ==='); print(imp.head(15).round(3).to_string(index=False))
res.to_json(OUT / f'operating_points{ARGS.tag}.json', indent=2, force_ascii=False)
json.dump(dict(source=ARGS.source, top_frac=ARGS.top_frac, w_back=ARGS.w_back, w_fwd=ARGS.w_fwd, hops=ARGS.hops,
               deg_cap=ARGS.deg_cap, params=params, val_oof_ap=ap_v, n_cand={t: int(len(c)) for t, c in cand.items()},
               feature_names=FEAT_NAMES, created=time.strftime('%Y-%m-%d %H:%M:%S')),
          open(OUT / f'summary{ARGS.tag}.json', 'w'), ensure_ascii=False, indent=2)
log(f'done -> {OUT}')


## 부록 — 코드에 안 담은 파일들 (탐색/과거 단계 또는 폐기)

코드 전문 대신 파일명·역할만 남긴다(전부 넣으면 리뷰가 오히려 어려워져서 뺐다 — 필요하면 경로로 직접 열어볼 것).

| 파일 | 역할 |
|---|---|
| `train_9class.py` | 1차 9-Class 탐색 단계(val 전용) 모델. stageB/C/select/final 계열의 시작점 — 지금은 lgb_ooc.py 라인으로 대체. |
| `stageB_9class.py` | 단계 B — 모델 패밀리×피처셋×(평탄 9-Class vs 계층 분해) 비교. |
| `stageC_9class.py` | 단계 C — 지표를 고쳐 변형·운영점을 다시 고르고 seed 분산으로 검증. |
| `select_9class.py` | 최종 설정 선택 — 운영점 규칙(재현율 0.70 고정)으로 단계 C 결과 중 고름. |
| `final_9class.py` | 1차 9-Class 최종 평가 — test를 한 번만 여는 스크립트. |
| `final_blocks_9class.py` | ★1 확정 평가 — 위상 피처 35개, attempt 블록 학습. |
| `final_blocks_trainmix.py` | final_blocks_9class.py와 같은 확정 평가의 train-mix 변형. |
| `train_blocks_9class.py` | 블록 단위 8-way 유형 분류 학습·평가(거래 단위 대비 비교용). |
| `block_gnn_9class.py` | 블록(사건) 유형 분류의 GNN 계열(GINE) 변형 — 문헌 비교 실험. |
| `ego_type_9class.py` | 알림 중심 ego-서브그래프 유형 분류(전역 블록 분할 대체안). **스크립트 자체 주석에 '팀 정본·회의록·Jira 어디에도 출처가 없어 폐기' 명시.** |
| `ablate_classes_blocks.py` | 특정 패턴 클래스를 빼면 나머지 클래스 성능이 오르는지 보는 ablation. |
| `seedvar_9class.py` | 클러스터 vs 무작위 언더샘플링 — seed를 바꿔 차이가 잡음인지 검증. |
| `diagnose_blocks.py` | 묶기(블록 구성) 진단 — 오라클 묶기 0.664 → 운영 묶기 0.329로 떨어지는 원인 분석. |
| `analyze_9class.py` | 전처리 산출물 위에서 보고서 근거 수치를 만드는 분석 스크립트(모델 학습 아님). |
| `lit_compare_9class.py` | 논문 대조용 — 18day 기준 test에서 이진(세탁 vs 정상) precision/recall/F1 곡선. |
| `make_report_9class.py` | 전처리 보고서 생성 — 수치를 손으로 옮기지 않고 산출물에서 직접 읽어 채움. |
| `ooc_predict_probe.py` | HI-Large 채점(predict) 메모리·시간 실측 탐침. |
| `ooc_scale_probe.py` | HI-Large out-of-core 학습 규모 탐침 — 전량이 27.3GiB 안에 들어가는지만 확인. |